In [ ]:
%load_ext autoreload
%autoreload 2
import pandas as pd
import numpy as np
import seaborn as sns
import warnings
import openpyxl
import re
from sqlalchemy import create_engine, text
import os
from variableUtils import *
from Utils import *
from pprint import pprint
import json
from collections import defaultdict
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error 
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from matplotlib.lines import Line2D
from reportlab.lib.pagesizes import letter, landscape, A4, A3
from reportlab.platypus import SimpleDocTemplate, Table, TableStyle, PageBreak, Paragraph, Spacer, Image, HRFlowable
from reportlab.lib import colors
from matplotlib.backends.backend_pdf import PdfPages
from reportlab.lib.enums import TA_CENTER, TA_RIGHT, TA_LEFT, TA_JUSTIFY
from reportlab.platypus import Paragraph, Spacer, KeepTogether, KeepInFrame 
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from xml.sax.saxutils import escape
from reportlab.lib.units import inch
import io
from openpyxl import load_workbook
from openpyxl.styles import PatternFill
from openpyxl.formatting.rule import FormulaRule
from openpyxl import load_workbook
from openpyxl.chart import ScatterChart, BarChart, Reference, Series
from openpyxl.chart.trendline import Trendline
from openpyxl.chart.marker import Marker
from openpyxl.drawing.line import LineProperties
from openpyxl.chart.shapes import GraphicalProperties
from openpyxl.chart.label import DataLabelList
from openpyxl.drawing.fill import ColorChoice
from openpyxl.chart.axis import ChartLines

import PIL
import ast
from adjustText import adjust_text
import statsmodels.api as sm
from sklearn.linear_model import LinearRegression
import win32com.client
import win32com
from scipy.stats import f_oneway
from scipy import stats
warnings.filterwarnings('ignore')
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
print(sns.__version__)
from collections import Counter
from datetime import datetime
# get today's date
today = datetime.now().strftime('%d-%m-%Y')
import os
import requests
from psycopg.types.json import Json
from sqlalchemy import create_engine, text
from dotenv import load_dotenv
print(f"Today's date: {today}")
plt.ioff()

#### Initialize

In [ ]:
# print working directory
print(os.getcwd())

load_dotenv()

dbUser = os.environ["DB_USER"]
dbPassword = os.environ["DB_PASSWORD"]
dbHost = os.environ["DB_HOST"]
dbPort = int(os.environ["DB_PORT"])
dbName = os.environ["DB_NAME"]
bearerToken = os.environ["DASH_TOKEN"]

dbUrl = f"postgresql+psycopg://{dbUser}:{dbPassword}@{dbHost}:{dbPort}/{dbName}"
engine = create_engine(dbUrl, pool_pre_ping=True)

scoreMap = {
    "O1": 1.00,
    "O2": 0.80,
    "O3": 0.60,
    "O4": 0.40,
    "O5": 0.00,
    "Yes": 1.00,
    "No": 0.00
}
print("DASH token:", bearerToken)

### URL to postgres

In [ ]:
createTableSql = """
CREATE TABLE IF NOT EXISTS rawForms (
  id BIGSERIAL PRIMARY KEY,
  assessmentId BIGINT,
  student_number BIGINT,
  student_name text,
  student_email text,
  datetimeUtc TIMESTAMPTZ,
  cohort TEXT,
  subject TEXT,
  type TEXT,
  completed BOOLEAN,
  forms JSONB NOT NULL,
  insertedAt TIMESTAMPTZ DEFAULT now()
);

CREATE UNIQUE INDEX IF NOT EXISTS uq_rawforms_assessmentId ON rawForms (assessmentId);
CREATE INDEX IF NOT EXISTS idx_rawforms_datetimeUtc ON rawForms (datetimeUtc);
CREATE INDEX IF NOT EXISTS idx_rawforms_studentnum ON rawForms (student_number);
CREATE INDEX IF NOT EXISTS idx_rawforms_forms_gin ON rawForms USING GIN (forms);
"""

def getInsertSql(replace=False):
    if replace:
        onconflict = """
    ON CONFLICT (assessmentId) DO UPDATE
    SET
    student_number = EXCLUDED.student_number,
    student_name   = EXCLUDED.student_name,
    student_email  = EXCLUDED.student_email,
    datetimeUtc   = EXCLUDED.datetimeUtc,
    cohort        = EXCLUDED.cohort,
    subject       = EXCLUDED.subject,
    type          = EXCLUDED.type,
    completed     = EXCLUDED.completed,
    forms         = EXCLUDED.forms,
    insertedAt    = now()
    """
    else:        onconflict = "ON CONFLICT (assessmentId) DO NOTHING"
    insertSql = f"""
    INSERT INTO rawForms (
    assessmentId,
    student_number,
    student_name,
    student_email,
    datetimeUtc,
    cohort,
    subject,
    type,
    completed,
    forms
    )
    SELECT
    :assessmentId,
    :student_number,
    :student_name,
    :student_email,
    :datetimeUtc,
    :cohort,
    :subject,
    :type,
    :completed,
    :forms
    WHERE :datetimeUtc >= TIMESTAMPTZ '2026-01-01'
    {onconflict}
    """
    
    return insertSql

deletesql = """Delete from rawforms
where student_name = 'Kunal Patel'
or lower(student_name) = 'suhrid gupta'
or lower(student_name) = 'test student'
or student_number is null
;
"""

def fetchAllAssessments(apiUrl, bearerToken):
    headers = {
        "Authorization": f"Token {bearerToken}",
        "Accept": "application/json",
    }

    records = []
    nextUrl = apiUrl
    with requests.Session() as session:
        while nextUrl:
            resp = session.get(nextUrl, headers=headers, timeout=60)
            # print(f"Requesting: {nextUrl} - Status Code: {resp}")
            # print("status:", resp.status_code)
            # print("contentType:", resp.headers.get("content-type"))
            # print("textStart:", resp.text[:1500])

            resp.raise_for_status()
            payload = resp.json()

            pageResults = payload.get("results")
            # print(f"Results in this page: {len(pageResults) if pageResults else 'No results key'}")
            if not isinstance(pageResults, list):
                raise ValueError("Unexpected response: missing 'results' list")

            records.extend(pageResults)
            nextUrl = payload.get("next")  # handles pagination if present
            print(f"Fetched {len(pageResults)} records, total so far: {len(records)}")

    return records

def toRows(records, formCol = "forms"):
    rows = []
    for r in records:
        studentInfo = r.get("student")
        if not studentInfo or not isinstance(studentInfo, dict):
            continue
        else:
            studentName = studentInfo["first_name"] + " " + studentInfo["last_name"]
            studentNumber = studentInfo.get("student_number")
            studentEmail = studentInfo.get("email")
            if not studentNumber:
                continue
        rows.append({
            "assessmentId": r.get("assessment_id") or r.get("id"),
            "student_number": studentNumber,
            "student_name": studentName,
            "student_email": studentEmail,
            "datetimeUtc": r.get("datetime"),
            "cohort": r.get("cohort"),
            "subject": r.get("subject"),
            "type": r.get("type"),
            "completed": r.get("completed"),
            formCol: (r.get("forms")),
        })
    return rows

def insertRows(rows, insertSql, batchSize=1000):
    with engine.begin() as conn:
        runDdl(conn, createTableSql)
        for i in range(0, len(rows), batchSize):
            conn.execute(text(insertSql), rows[i:i + batchSize])
        runDdl(conn, deletesql)

allCohorts = ["DDS4","BOH3","DDS1","DDS2","DDS3","BOH2","BOH1"]
# ---- EDIT ONLY THIS CELL ----
useAll = False                 # True = use allCohorts
replaceExisting = True        # if True, will replace existing records in DB; if False, will skip records with existing assessmentId
includeCohorts = ["DDS3", "BOH2", "DDS2", "BOH1"]     # only used if useAll=False
# includeCohorts = ["DDS4", "BOH3"]
excludeCohorts = []           # applied in both cases
year = 2026
type_ = "caf"                 # used in API query; set to None to not filter by type
# type_ = "mini-cex"
# ----------------------------

def buildCohortList():
    if useAll:
        cohortSet = set(allCohorts)
    else:
        cohortSet = set(includeCohorts)

    cohortSet -= set(excludeCohorts)

    return [c for c in allCohorts if c in cohortSet]



cohortList = buildCohortList()
cohortText = ",".join(cohortList)

def main():
    apiUrl = f"https://api.unimelb-dash.com/assessment/{type_}/v2/get?page_size=max&page=1&cohort={cohortText}&year={year}&ordering=datetime,-cohort"
    print(apiUrl)
    if not bearerToken:
        raise RuntimeError("Set DASH_TOKEN environment variable to your Bearer token")

    records = fetchAllAssessments(apiUrl, bearerToken)
    # save records to a json file for backup
    with open(f"temp {year} {type_}.json", "w", encoding="utf-8") as f:
        json.dump(records, f, ensure_ascii=False, indent=2)
    print(f"Fetched {len(records)} assessment records from API.", records[0].keys())
    rows = toRows(records)
    print(f"Prepared {len(rows)} rows for insertion.")
    rowDf = pd.DataFrame(rows)
    # dump forms to string with double instead of single quotes for better readability in Excel
    rowDf["forms"] = rowDf["forms"].apply(lambda x: json.dumps(x, ensure_ascii=False))
    rowDf.to_excel(f"fetched_rows {year} {type_}.xlsx", index=False)
    insertSql = getInsertSql(replaceExisting)
    # Wrap for DB insertion
    for row in rows:
        row["forms"] = Json(row["forms"])
    insertRows(rows, insertSql)


if __name__ == "__main__":
    main()


#### Separating forms

In [ ]:
createTableSql = """
CREATE TABLE IF NOT EXISTS rawform_forms (
  form_code TEXT NOT NULL,
  assessmentid BIGINT NOT NULL,

  student_number BIGINT NOT NULL,
  student_name TEXT,
  student_email TEXT,
  datetimeutc TIMESTAMPTZ,
  cohort TEXT,
  subject TEXT,
  type TEXT,
  completed BOOLEAN,

  student_data JSONB,
  assessor_data JSONB,

  student_reflection TEXT,
  assessor_reflection TEXT,
  clinical_incident TEXT,

  patient_complexity TEXT,

  role TEXT,
  clinic TEXT,
  scales JSONB,
  checklists JSONB,
  version INTEGER,

  patient_age INTEGER,
  patient_drn TEXT,
  assessor_name TEXT,
  patient_details TEXT,
  patient_interpreter BOOLEAN,

  submitted_by_student BOOLEAN,
  additional_checklists JSONB,
  submitted_by_assessor BOOLEAN,

  insertedat TIMESTAMPTZ DEFAULT now(),

  PRIMARY KEY (form_code, assessmentid)
);

"""


def getInsertSql(replace=False):
    if replace:
        onconflict = """
    ON CONFLICT (form_code, assessmentid) DO UPDATE
    SET
    student_number = EXCLUDED.student_number,
    student_name   = EXCLUDED.student_name,
    student_email  = EXCLUDED.student_email,
    datetimeutc   = EXCLUDED.datetimeutc,
    cohort        = EXCLUDED.cohort,
    subject       = EXCLUDED.subject,
    type          = EXCLUDED.type,
    completed     = EXCLUDED.completed,
    student_data  = EXCLUDED.student_data,
    assessor_data = EXCLUDED.assessor_data,
    student_reflection = EXCLUDED.student_reflection,
    assessor_reflection = EXCLUDED.assessor_reflection,
    clinical_incident = EXCLUDED.clinical_incident,
    patient_complexity = EXCLUDED.patient_complexity,
    role          = EXCLUDED.role,
    clinic        = EXCLUDED.clinic,
    scales        = EXCLUDED.scales,
    checklists    = EXCLUDED.checklists,
    version       = EXCLUDED.version,
    patient_age   = EXCLUDED.patient_age,
    patient_drn   = EXCLUDED.patient_drn,
    assessor_name = EXCLUDED.assessor_name,
    patient_details = EXCLUDED.patient_details,
    patient_interpreter = EXCLUDED.patient_interpreter,
    submitted_by_student = EXCLUDED.submitted_by_student,
    additional_checklists = EXCLUDED.additional_checklists,
    submitted_by_assessor = EXCLUDED.submitted_by_assessor,
    insertedat    = now()
        """
    else:
        onconflict = "ON CONFLICT (form_code, assessmentid) DO NOTHING"
    insertSql = f"""
    INSERT INTO rawform_forms (
      form_code,
      assessmentid, student_number, student_name, student_email, datetimeutc, cohort, subject, type, completed,
      student_data, assessor_data,
      student_reflection, assessor_reflection,
      clinical_incident,
      -- time_mgmt, professionalism, communication, entrustment,
      patient_complexity,
      role, clinic, scales, checklists, version,
      patient_age, patient_drn, assessor_name, patient_details, patient_interpreter,
      submitted_by_student, additional_checklists, submitted_by_assessor, insertedat
    )
    SELECT
      f.form_code,

      r.assessmentid, r.student_number, r.student_name, r.student_email, r.datetimeutc, r.cohort, r.subject, r.type, r.completed,

      /* studentdata: remove reflection only */
      (studentJson - 'reflection') AS studentdata,

      /* assessordata: remove extracted keys */
      (assessorJson
        - 'reflection'
        -- - 'time_mgmt'
        -- - 'professionalism'
        -- - 'communication'
        -- - 'entrustment'
        - 'patient_complexity'
      - 'clinical_incident'
      ) AS assessordata,

      /* reflections */
      (studentJson->>'reflection')  AS student_reflection,
      (assessorJson->>'reflection') AS assessor_reflection,
    /* put this exactly where clinical_incident is in INSERT */
      NULLIF(COALESCE(assessorJson->>'critical_incident', assessorJson->>'clinical_incident'), '') AS clinical_incident,

      /* extracted scalars (no assessor_ prefix) */
      -- NULLIF(assessorJson->'time_mgmt'->>'scale','')::smallint       AS time_mgmt,
      -- NULLIF(assessorJson->'professionalism'->>'scale','')::smallint AS professionalism,
      -- NULLIF(assessorJson->'communication'->>'scale','')::smallint   AS communication,
      -- NULLIF(assessorJson->'entrustment'->>'scale','')::smallint     AS entrustment,
      (assessorJson->'patient_complexity'->>'scale')                 AS patient_complexity,

      f.form_value->>'role' AS role,
      COALESCE(f.form_value->>'clinic_type', f.form_value->>'clinic') AS clinic,
      f.form_value->'scales' AS scales,
      f.form_value->'checklists' AS checklists,
      NULLIF(f.form_value->>'version','')::int AS version,

      NULLIF(f.form_value->>'patient_age','')::int AS patient_age,
      NULLIF(f.form_value->>'patient_drn','') AS patient_drn,
      f.form_value->>'assessor_name' AS assessor_name,
      f.form_value->>'patient_details' AS patient_details,
      (f.form_value->>'patient_interpreter')::boolean AS patient_interpreter,

      (f.form_value->>'submitted_by_student')::boolean AS submitted_by_student,
      f.form_value->'additional_checklists' AS additional_checklists,
      (f.form_value->>'submitted_by_assessor')::boolean AS submitted_by_assessor,
      r.insertedat
    FROM rawforms r
    CROSS JOIN LATERAL (
      SELECT e.key AS form_code, e.value AS form_value
      FROM jsonb_each(r.forms) e
      WHERE jsonb_typeof(r.forms) = 'object'
      UNION ALL
      SELECT a.value->>'form_key' AS form_code, a.value AS form_value
      FROM jsonb_array_elements(r.forms) a
      WHERE jsonb_typeof(r.forms) = 'array'
    ) f
    CROSS JOIN LATERAL (
      SELECT
        COALESCE(f.form_value->'student_data',  f.form_value->'data'->'student')  AS studentJson,
        COALESCE(f.form_value->'assessor_data', f.form_value->'data'->'assessor') AS assessorJson
    ) j
    WHERE f.form_code IS NOT NULL
    AND r.datetimeUtc >= TIMESTAMPTZ '2026-01-01'
    {onconflict}
    """
    return insertSql

deletesql = """Delete from rawform_forms
where student_name = 'Kunal Patel'
or lower(student_name) = 'suhrid gupta'
or lower(student_name) = 'test student'
or student_number is null
;
"""

def processForms():
    with engine.begin() as conn:
        runDdl(conn, createTableSql)
        insertSql = getInsertSql(replace = True)
        conn.execute(text(insertSql))
        conn.execute(text(deletesql))

processForms()

In [ ]:
# get counts of cohorts from rawforms

counts ="""
SELECT cohort, count(*) FROM rawforms
WHERE datetimeUtc >= '2026-01-01'
GROUP BY cohort;
"""

df = readDf(engine, counts)
display(df)

### BOH2 DDS2 DDS3 general

In [ ]:
from boh2_dds2_dds3_utils import *

In [ ]:
sql = """-- DDS2 & DDS3 via rawform_forms (item codes are checklists keys)
SELECT
    cohort,
    cl.key AS item_code,
    COUNT(*)::int AS count
FROM rawform_forms f
CROSS JOIN LATERAL jsonb_each(COALESCE(f.checklists, '{}'::jsonb)) cl
WHERE f.cohort = ANY(ARRAY['DDS2','DDS3'])
  AND f.type = 'Clinic'
  AND f.submitted_by_assessor = true
  AND f.datetimeutc >= '2026-01-01'
  AND cl.key = ANY(ARRAY['511','512','513','514','515'])
GROUP BY 1, 2

UNION ALL

-- DDS4 via dds4_boh3_forms (item codes nested in patient_data)
SELECT
    f.cohort,
    split_part(ic->>'code', '/', 1) AS item_code,
    SUM(COALESCE(NULLIF(ic->>'quantity','')::int, 1))::int AS count
FROM dds4_boh3_forms f
CROSS JOIN LATERAL jsonb_array_elements(COALESCE(f.patient_data, '[]'::jsonb)) pd
CROSS JOIN LATERAL jsonb_array_elements(COALESCE(pd->'itemCodes', '[]'::jsonb)) ic
WHERE f.cohort = 'DDS4'
  AND f.type = 'Clinic'
  AND f.submitted_by_assessor = true
  AND f.datetimeutc >= '2026-01-01'
  AND ic ? 'code'
  AND split_part(ic->>'code', '/', 1) = ANY(ARRAY['511','512','513','514','515'])
GROUP BY 1, 2

ORDER BY cohort, item_code;"""

df = readDf(engine, sql)
display(df)

In [ ]:
# Cohort reports (previously called without engine/today)
# getCohortReports(engine, "DDS2", today)
# getCohortReports(engine, "BOH2", today)
# getCohortReports(engine, "BOH1", today)
# getCohortReports(engine, "DDS3", today, type_=["Clinic"])
getCohortReportsPerClinic(engine, "DDS3", today)

In [ ]:
for ft in ["Simulation", "Clinic"]:
    buildCohortTimeSeriesPdf(engine=engine, cohort="BOH2", outPath=f"BOH2/BOH2 {ft} Time Series ({today}).pdf",
        bannerTitle=f"BOH2 {ft} – Performance Over Time", formType=ft, subheadingStyle=subheadingStyle, uniColor=uniColor,)
    buildCohortTimeSeriesPdf(engine=engine, cohort="BOH1", outPath=f"BOH1/BOH1 {ft} Time Series ({today}).pdf",
        bannerTitle=f"BOH1 {ft} – Performance Over Time", formType=ft, subheadingStyle=subheadingStyle, uniColor=uniColor,)
    # buildCohortTimeSeriesPdf(engine=engine, cohort="DDS2", outPath=f"DDS2/DDS2 {ft} Time Series ({today}).pdf",
        # bannerTitle=f"DDS2 {ft} – Performance Over Time", formType=ft, subheadingStyle=subheadingStyle, uniColor=uniColor,)
    buildCohortTimeSeriesPdf(engine=engine, cohort="DDS3", outPath=f"DDS3/DDS3 {ft} Time Series ({today}).pdf",
        bannerTitle=f"DDS3 {ft} – Performance Over Time", formType=ft, subheadingStyle=subheadingStyle, uniColor=uniColor,)

#### Student Reports

In [ ]:
# Student reports
buildEntireCohortStudentReports(engine, "DDS3", patientInfo=True)
# buildEntireCohortStudentReports(engine, "BOH1", patientInfo=False)
# buildEntireCohortStudentReports(engine, "BOH2", patientInfo=True)
# buildEntireCohortStudentReports(engine, "DDS2", patientInfo=True)

### Assessor Analysis

In [ ]:
from assessor_analysis import run_assessor_analysis
df, assessor_stats = run_assessor_analysis(engine)

In [ ]:
from assessor_analysis import load_data, add_scores
from assessor_confound  import run_confound_analysis

df = add_scores(load_data(engine))
results = run_confound_analysis(df)

### BOH1 for the date

In [ ]:
cohort = "BOH1"
date = "29-04-2026"
sql = f"""
SELECT * 



FROM public.rawform_forms
where cohort = '{cohort}'
and datetimeutc = '{date}'
ORDER BY form_code ASC, assessmentid ASC 
"""

In [ ]:
from boh1_utils import buildTimedSessionReport, buildItemReport

# Timed session report
buildTimedSessionReport(
    engine, 
    outputPath="BOH1/BOH1_Timed_29Apr.xlsx",
    timedDate="2026-04-29",
    timedItems=["161", "114 H/S", "221", "222"],
    formsTable="rawform_forms",  # adjust to your table name
)

# Item 531 report (up to 05 May)
buildItemReport(
    engine,
    outputPath="BOH1/BOH1_Item531.xlsx",
    itemCode="531",
    dateTo="2026-05-05",
    formsTable="rawform_forms",
)

## DDS2

#### Weekly

In [ ]:
date = '2026-04-20'
def createwhereStatement(prefix):
    return f"""
      where {prefix}.datetimeutc::date = DATE '{date}'
      and {prefix}.cohort = 'DDS2'
      and {prefix}.submitted_by_assessor
      and {prefix}.type = 'Simulation'
      """

globalratingsSql = f"""
SELECT
  global_rating,
  COUNT(*) AS n
FROM (
  SELECT NULLIF(v->>'scale','')::int AS global_rating
  FROM rawform_forms f
  CROSS JOIN LATERAL jsonb_each(f.assessor_data) k(key, v)
  WHERE f.submitted_by_assessor
    AND key = 'scale-global-rating'
    and datetimeutc::date = DATE '{date}'
    and cohort ='DDS2'
    and type = 'Simulation'
) x
GROUP BY global_rating
ORDER BY global_rating;
"""

ratingDist = readDf(engine, globalratingsSql)
ntotal = ratingDist["n"].sum()
display(ratingDist)
print(f"Total assessments with global rating: {ntotal}")


In [ ]:
dataSql = f"""
WITH base AS (
  SELECT
    f.form_code,
    f.assessmentid,
    f.student_number,
    f.student_name,
    f.student_email,
    f.assessor_name,
    f.cohort,
    f.subject,
    f.type,
    f.datetimeutc,
    f.clinic,
    f.assessor_reflection,
    f.student_reflection,
    f.clinical_incident,
    
    -- Item Codes
    (
      SELECT string_agg(DISTINCT ic.key, '/')
      FROM jsonb_each(COALESCE(f.assessor_data, '{{}}'::jsonb)) ic
      WHERE ic.key NOT LIKE 'scale-%'
    ) AS item_code,

    -- Pivot scales into columns from assessordata
    MAX(CASE WHEN s.key = 'scale-global-rating'      THEN NULLIF(s.value->>'scale','')::int END) AS "GR",
    MAX(CASE WHEN s.key = 'scale-time-mgmt'          THEN NULLIF(s.value->>'scale','')::int END) AS "TS",
    MAX(CASE WHEN s.key = 'scale-communication'      THEN NULLIF(s.value->>'scale','')::int END) AS "CS",
    MAX(CASE WHEN s.key = 'scale-professionalism'    THEN NULLIF(s.value->>'scale','')::int END) AS "PS",
    MAX(CASE WHEN s.key = 'scale-practice-readiness' THEN NULLIF(s.value->>'scale','')::int END) AS "PR",
    MAX(CASE WHEN s.key = 'scale-position-ergonomics'THEN NULLIF(s.value->>'scale','')::int END) AS "PEC"

  FROM rawform_forms f
  LEFT JOIN LATERAL jsonb_each(COALESCE(f.assessor_data,'{{}}'::jsonb)) s(key, value)
    ON jsonb_typeof(s.value) = 'object' AND (s.value ? 'scale')
  {createwhereStatement('f')}
  GROUP BY
    f.form_code, f.assessmentid
),
assessorChecklistPivot AS (
  SELECT
    f.form_code,
    f.assessmentid,
        (
        SELECT value
        FROM jsonb_each(f.assessor_data)
        WHERE jsonb_typeof(value) = 'object'
          AND NOT (key LIKE 'scale-%')
        LIMIT 1
    ) AS mc_json

  FROM rawform_forms f
  CROSS JOIN LATERAL jsonb_each(COALESCE(f.assessor_data,'{{}}'::jsonb)) ck(ckey, cval)
  CROSS JOIN LATERAL jsonb_each_text(ck.cval) it(key, value)
  CROSS JOIN LATERAL (
    SELECT CASE it.value
      WHEN 'O1' THEN 1.00
      WHEN 'O2' THEN 0.80
      WHEN 'O3' THEN 0.60
      WHEN 'O4' THEN 0.40
      WHEN 'O5' THEN 0.00
      WHEN 'Yes' THEN 1.00
      WHEN 'No'  THEN 0.00
      ELSE NULL::numeric
    END AS score01
  ) m
  {createwhereStatement('f')}
    AND jsonb_typeof(ck.cval) = 'object'
    AND NOT (ck.cval ? 'scale')
  GROUP BY f.form_code, f.assessmentid
),

assessorRubricAvg AS (
  SELECT
    f.form_code,
    f.assessmentid,
    ROUND(AVG(
      CASE it.value
        WHEN 'O1' THEN 1.00
        WHEN 'O2' THEN 0.80
        WHEN 'O3' THEN 0.60
        WHEN 'O4' THEN 0.40
        WHEN 'O5' THEN 0.00
        WHEN 'Yes' THEN 1.00
        WHEN 'No'  THEN 0.00
        ELSE NULL::numeric
      END
    ), 2) AS assessor_score01
  FROM rawform_forms f
  CROSS JOIN LATERAL jsonb_each(COALESCE(f.assessor_data,'{{}}'::jsonb)) ck(ckey, cval)
  CROSS JOIN LATERAL jsonb_each_text(ck.cval) it(key, value)
  {createwhereStatement('f')}
    AND jsonb_typeof(ck.cval) = 'object'
    AND NOT (ck.cval ? 'scale')  -- excludes scale-* objects
  GROUP BY f.form_code, f.assessmentid
),

studentRubricAvg AS (
  SELECT
    f.form_code,
    f.assessmentid,
    ROUND(AVG(
      CASE it.value
        WHEN 'O1' THEN 1.00
        WHEN 'O2' THEN 0.80
        WHEN 'O3' THEN 0.60
        WHEN 'O4' THEN 0.40
        WHEN 'O5' THEN 0.00
        WHEN 'Yes' THEN 1.00
        WHEN 'No'  THEN 0.00
        ELSE NULL::numeric
      END
    ), 2) AS student_score01
  FROM rawform_forms f
  CROSS JOIN LATERAL jsonb_each(COALESCE(f.student_data,'{{}}'::jsonb)) ck(ckey, cval)
  CROSS JOIN LATERAL jsonb_each_text(ck.cval) it(key, value)
  {createwhereStatement('f')}
    AND jsonb_typeof(ck.cval) = 'object'
    AND NOT (ck.cval ? 'scale')
  GROUP BY f.form_code, f.assessmentid
)

SELECT
  b.student_number,
  b.student_name,
  b.student_email,
  b.assessor_name,
  b.item_code,
--  b.cohort,
--  b.subject,
--  b.type,
  b.datetimeutc::date AS date,
--  b.clinic,
  a.mc_json,

  b."GR",
  b."TS",
  b."CS",
  b."PS",
  b."PR",
  b."PEC",

  
  r.assessor_score01 AS "Assessor Score",
  s.student_score01  AS "Student Score",

  b.clinical_incident,
  b.assessor_reflection,
  b.student_reflection,
  b.assessmentid,
  b.form_code
FROM base b
LEFT JOIN assessorChecklistPivot a
  ON a.assessmentid = b.assessmentid AND a.form_code = b.form_code
LEFT JOIN assessorRubricAvg r
  ON r.assessmentid = b.assessmentid AND r.form_code = b.form_code
LEFT JOIN studentRubricAvg s
  ON s.assessmentid = b.assessmentid AND s.form_code = b.form_code
ORDER BY b.cohort, b.student_number, b.datetimeutc, b.form_code;
"""


dataDf = readDf(engine, dataSql)
# Expand JSON into columns
mcDf = pd.json_normalize(dataDf["mc_json"])

# Convert O-values to numeric
mcDf = mcDf.replace(scoreMap)

# Ensure numeric dtype
mcDf = mcDf.apply(pd.to_numeric, errors="coerce")

# Merge back
dataDf = pd.concat([dataDf.drop(columns=["mc_json"]), mcDf], axis=1)
# dataDf.drop(columns = "mc_json", inplace=True)
print(f"Data rows: {len(dataDf)}")
display(dataDf.head(30))

dataDf.to_excel(f"DDS2/dds2 {date} assessment_data.xlsx", index=False)
# autofit columns in the saved excel file
with pd.ExcelWriter(f"DDS2/dds2 {date} assessment_data.xlsx", engine='openpyxl', mode='a', if_sheet_exists='overlay') as writer:
    ws = writer.sheets['Sheet1']
    autoFitColumns(ws)

In [ ]:
def addAssessmentCharts(filepath, dataDf, sheet1Name='Sheet1', chartSheetName='Charts', blrFilter=None):
    """
    Adds charts on a second sheet:
      1. Student Score vs Assessor Score (scatter)
      2. GR counts (bar)
      3. Assessor Score vs GR (scatter with trendline, optionally filtered)
    """
    wb = load_workbook(filepath)
    ws1 = wb[sheet1Name]

    if chartSheetName in wb.sheetnames:
        del wb[chartSheetName]
    ws2 = wb.create_sheet(chartSheetName)

    # Column letters from dataDf order (openpyxl is 1-indexed)
    headers = {cell.value: cell.column for cell in ws1[1]}
    grCol = headers['GR']
    assessorCol = headers['Assessor Score']
    studentCol = headers['Student Score']
    nRows = ws1.max_row
    lightGrid = ChartLines(
        spPr=GraphicalProperties(ln=LineProperties(solidFill="D9D9D9"))
    )
    

    # ── Chart 1: Student Score vs Assessor Score ─────────────────
    chart1 = ScatterChart()
    chart1.title = "Student Score vs Assessor Score"
    chart1.x_axis.title = "Assessor Score"
    chart1.y_axis.title = "Student Score"
    chart1.x_axis.scaling.min = 0
    chart1.x_axis.scaling.max = 1
    chart1.y_axis.scaling.min = 0
    chart1.y_axis.scaling.max = 1
    chart1.x_axis.majorUnit = 0.1
    chart1.x_axis.majorTickMark = "out"
    chart1.x_axis.delete = False
    chart1.y_axis.majorUnit = 0.1
    chart1.y_axis.majorTickMark = "out"
    chart1.y_axis.delete = False
    chart1.legend = None
    chart1.height = 10
    chart1.width = 15
    chart1.y_axis.majorGridlines = lightGrid
    xRef = Reference(ws1, min_col=assessorCol, min_row=2, max_row=nRows)
    yRef = Reference(ws1, min_col=studentCol, min_row=2, max_row=nRows)
    s1 = Series(yRef, xRef, title="Scores")
    s1.marker = Marker(
        symbol='circle',
        size=6,
        spPr=GraphicalProperties(solidFill="1F77B4", ln=LineProperties(solidFill="1F77B4"))
    )
    s1.graphicalProperties = GraphicalProperties(ln=LineProperties(noFill=True))
    chart1.series.append(s1)
        # ── 45-degree reference line ─────────────────
    # Write helper values somewhere off-screen on ws2
    refRow = 80
    ws2.cell(row=refRow,     column=5, value=0)
    ws2.cell(row=refRow,     column=6, value=0)
    ws2.cell(row=refRow + 1, column=5, value=1)
    ws2.cell(row=refRow + 1, column=6, value=1)

    xRefLine = Reference(ws2, min_col=5, min_row=refRow, max_row=refRow + 1)
    yRefLine = Reference(ws2, min_col=6, min_row=refRow, max_row=refRow + 1)

    refSeries = Series(yRefLine, xRefLine, title="y = x")
    refSeries.marker = Marker(symbol='none')
    refSeries.graphicalProperties = GraphicalProperties(
        ln=LineProperties(solidFill="888888", prstDash="dash", w=9525)
    )
    chart1.series.append(refSeries)
    ws2.add_chart(chart1, "A1")

    
    # ── Chart 2: GR counts bar ─────────────────
    grValues = [row[0] for row in ws1.iter_rows(
        min_row=2, min_col=grCol, max_col=grCol, values_only=True) if row[0] is not None]
    grCounts = pd.Series(grValues).value_counts().sort_index()

    startRow = 45
    ws2.cell(row=startRow, column=1, value='GR')
    ws2.cell(row=startRow, column=2, value='Count')
    for i, (gr, cnt) in enumerate(grCounts.items(), start=1):
        ws2.cell(row=startRow + i, column=1, value=int(gr))
        ws2.cell(row=startRow + i, column=2, value=int(cnt))

    chart2 = BarChart()
    chart2.title = "Global Rating Distribution"
    chart2.x_axis.title = "GR"
    chart2.y_axis.title = "Count"
    chart2.legend = None
    chart2.height = 10
    chart2.width = 15
    chart2.y_axis.majorGridlines = lightGrid

    dataRef = Reference(ws2, min_col=2, min_row=startRow, max_row=startRow + len(grCounts))
    catRef = Reference(ws2, min_col=1, min_row=startRow + 1, max_row=startRow + len(grCounts))
    chart2.add_data(dataRef, titles_from_data=True)
    chart2.set_categories(catRef)

    # Uniform blue color for all bars
    chart2.series[0].graphicalProperties = GraphicalProperties(solidFill="1F77B4")

    # Show value labels on top of bars
    chart2.dataLabels = DataLabelList(
        showVal=True,
        showCatName=False,
        showSerName=False,
        showLegendKey=False,
    )

    # Ensure x-axis tick labels are shown
    chart2.x_axis.delete = False
    chart2.y_axis.delete = False

    ws2.add_chart(chart2, "J1")

# ── Chart 3: Assessor Score vs GR ─────────────────
    chart3 = ScatterChart()
    chart3.title = "Assessor Score vs GR"
    chart3.x_axis.title = "GR"
    chart3.y_axis.title = "Assessor Score"
    chart3.legend = None
    chart3.height = 10
    chart3.width = 15

    chart3.x_axis.delete = False
    chart3.y_axis.delete = False
    chart3.x_axis.scaling.min = 0
    chart3.x_axis.scaling.max = 5
    chart3.x_axis.majorUnit = 1
    chart3.x_axis.majorTickMark = "out"
    chart3.y_axis.scaling.min = 0
    chart3.y_axis.scaling.max = 1
    chart3.y_axis.majorUnit = 0.1
    chart3.y_axis.majorTickMark = "out"
    chart3.y_axis.number_format = '0.0'

    xRef3 = Reference(ws1, min_col=grCol, min_row=2, max_row=nRows)
    yRef3 = Reference(ws1, min_col=assessorCol, min_row=2, max_row=nRows)

    s3 = Series(yRef3, xRef3, title="Assessor Score vs GR")
    s3.marker = Marker(
        symbol='circle',
        size=6,
        spPr=GraphicalProperties(solidFill="1F77B4", ln=LineProperties(solidFill="1F77B4"))
    )
    s3.graphicalProperties = GraphicalProperties(ln=LineProperties(noFill=True))
    s3.trendline = Trendline(trendlineType='linear', dispEq=True, dispRSqr=True)
    chart3.series.append(s3)
    ws2.add_chart(chart3, "A22")

    wb.save(filepath)

addAssessmentCharts(f"DDS2/dds2 {date} assessment_data.xlsx", dataDf, blrFilter="BLR")

In [ ]:
# combined report
from openpyxl.styles import Font, PatternFill
DEFAULT_DATE_REGEX = r"\d{4}-\d{2}-\d{2}"
DEFAULT_FILE_PATTERN = r"assessment_data\.xlsx$"
DEFAULT_ID_COLS = ["student_number"]
FAIL_FILL = PatternFill(start_color="FFC7CE", end_color="FFC7CE", fill_type="solid")  # light red
FAIL_TEXT = Font(color="9C0006")  # dark red text
folderPath = f"DDS2"
from enum import Enum

class FlagMethod(str, Enum):
    """Method used to identify low-performing students."""
    PERCENTILE = "percentile"
    BLR = "blr"

def loadWeeklyFiles(folderPath, dateRegex =DEFAULT_DATE_REGEX, filePattern=DEFAULT_FILE_PATTERN):
    """Load all weekly assessment files, return list of (dateStr, df) tuples."""
    compiledFile = re.compile(filePattern, re.IGNORECASE)
    compiledDate = re.compile(dateRegex)
    files = []
    for filename in os.listdir(folderPath):
        if filename.startswith("~$"):
            continue
        if not compiledFile.search(filename):
            continue
        dateMatch = compiledDate.search(filename)
        if not dateMatch:
            continue
        dateStr = dateMatch.group(0)
        filePath = os.path.join(folderPath, filename)
        df = pd.read_excel(filePath, engine="openpyxl")
        files.append((dateStr, df))
    return sorted(files, key=lambda x: x[0])

def buildWideDf(folderPath, valueCol, dateRegex=DEFAULT_DATE_REGEX,filePattern=DEFAULT_FILE_PATTERN, idCols=DEFAULT_ID_COLS,):
    weeklyFiles = loadWeeklyFiles(folderPath, dateRegex, filePattern)
    rows = []
    for dateStr, df in weeklyFiles:
        if not all(c in df.columns for c in idCols + [valueCol]):
            continue
        tmp = df[idCols + [valueCol]].copy()
        tmp["date"] = dateStr
        rows.append(tmp)

    if not rows:
        return pd.DataFrame(columns=idCols)

    longDf = pd.concat(rows, ignore_index=True)
    wideDf = (
        longDf.pivot_table(index=idCols, columns="date", values=valueCol, aggfunc="mean")
        .reset_index()
    )

    dateCols = sorted([c for c in wideDf.columns if c not in idCols])
    wideDf = wideDf[idCols + dateCols]
    wideDf[dateCols] = wideDf[dateCols].apply(pd.to_numeric, errors="coerce").round(2)
    wideDf['Avg Score'] = wideDf[dateCols].mean(axis=1).round(2)

    return wideDf

def runBlrAnalysis(df, dateStr, borderlineGr, mcCols=None, grCol = "GR"):
    """Run full BLR analysis for a single day's data. Returns dict of results."""
    if mcCols is None:
        mcCols = [c for c in df.columns if c.startswith("MC")]

    # ── 1. Borderline group method: mean score of borderline group ──
    borderlineScores = df.loc[df[grCol] == borderlineGr, "assessor_score"]
    borderlineMean = borderlineScores.mean()
    borderlineSd = borderlineScores.std()

    # ── 2. Linear regression: assessor_score ~ global_rating ──
    slope, intercept, rVal, pVal, stdErr = stats.linregress(
        df[grCol], df["assessor_score"]
    )
    regressionCutoff = slope * borderlineGr + intercept
    rSquared = rVal ** 2

    # ── 3. Per-rating group stats ──
    ratingGroups = df.groupby(grCol)["assessor_score"]
    ratingStats = ratingGroups.agg(["mean", "std", "count"]).rename(
        columns={"mean": "meanScore", "std": "sdScore", "count": "n"}
    )

    # ── 4. Item analysis ──
    itemStats = []
    for mc in mcCols:
        if mc not in df.columns:
            continue
        itemMean = df[mc].mean()
        borderlineItemMean = df.loc[df[grCol] == borderlineGr, mc].mean()
        itemTotalCorr = df[mc].corr(df["assessor_score"])
        itemStats.append({
            "item": mc,
            "mean": round(itemMean, 3),
            "borderlineMean": round(borderlineItemMean, 3),
            "itemTotalCorr": round(itemTotalCorr, 3),
        })
    itemStatsDf = pd.DataFrame(itemStats)

    # ── 5. Cronbach's alpha ──
    validMcCols = [c for c in mcCols if c in df.columns]
    mcData = df[validMcCols].dropna()
    k = len(validMcCols)
    itemVars = mcData.var(axis=0, ddof=1)
    totalVar = mcData.sum(axis=1).var(ddof=1)
    cronbachAlpha = (k / (k - 1)) * (1 - itemVars.sum() / totalVar) if k > 1 else np.nan

    # ── 6. Flag students below cutoff ──
    belowCutoffDf = df.loc[
        df["assessor_score"] <= regressionCutoff,
        ["student_number", "student_name", "assessor_score", grCol, "assessor_name"],
    ].sort_values("assessor_score")

    return {
        "date": dateStr,
        "n": len(df),
        "borderlineMean": round(borderlineMean*100, 2),
        "borderlineSd": round(borderlineSd*100, 2),
        "regressionCutoff": round(regressionCutoff*100, 2),
        "slope": round(slope, 2),
        "intercept": round(intercept, 2),
        "rSquared": round(rSquared, 2),
        "pVal": round(pVal, 4),
        "overallMean": round(df["assessor_score"].mean()*100, 2),
        "overallSd": round(df["assessor_score"].std()*100, 2),
        "cronbachAlpha": round(cronbachAlpha, 2),
        "ratingStats": ratingStats,
        "itemStatsDf": itemStatsDf,
        "belowCutoffDf": belowCutoffDf,
        "df": df,
    }

def runBlrAnalysisForAllFiles(folderPath, borderlineGr=2, dateRegex=r"\d{4}-\d{2}-\d{2}", filePattern=r"assessment_data\.xlsx$"):
    """Run BLR analysis for all weekly files in folder. Returns list of results dicts."""
    files = loadWeeklyFiles(folderPath, dateRegex, filePattern)
    allResults = []
    for dateStr, df in files:
        result = runBlrAnalysis(df, dateStr, borderlineGr)
        allResults.append(result)
    allResultsDf = pd.DataFrame(allResults)
    return allResultsDf

# ─── Student flagging ─────────────────────────────────────────────────────────

def highlightFlaggedCells(ws, flagged_df, thresholds, id_cols):
    """
    Highlight cells in the scores_flagged sheet where the student's score
    is at or below the threshold for that date column.
    Also highlights the Pass/Fail cell for Fail rows.
    """
    date_cols = [c for c in flagged_df.columns if c not in id_cols + ["Avg Score", "low_count", "Pass/Fail"]]

    # Build column-name -> Excel-column-index mapping (1-based)
    col_index = {col: i + 1 for i, col in enumerate(flagged_df.columns)}

    for row_idx, (_, row) in enumerate(flagged_df.iterrows(), start=2):  # row 1 = header
        # Highlight individual date cells that are below threshold
        for dc in date_cols:
            val = row[dc]
            thresh = thresholds.get(dc)
            if pd.notna(val) and pd.notna(thresh) and val <= thresh:
                cell = ws.cell(row=row_idx, column=col_index[dc])
                cell.fill = FAIL_FILL
                cell.font = FAIL_TEXT

        # Highlight the Pass/Fail cell if Fail
        if row.get("Pass/Fail") == "Fail":
            pf_cell = ws.cell(row=row_idx, column=col_index["Pass/Fail"])
            pf_cell.fill = FAIL_FILL
            pf_cell.font = FAIL_TEXT

def _flag_by_percentile(
    score_df: pd.DataFrame,
    id_cols: list[str],
    percentile: float = 0.15,
    min_low_count: int = 3,
) -> tuple[pd.DataFrame, pd.Series]:
    """
    Flag students who score at or below the *percentile* threshold in at
    least *min_low_count* assessment dates.
    """
    date_cols = [
        c for c in score_df.columns if c not in id_cols and c != "Avg Score"
    ]
    thresholds = score_df[date_cols].quantile(percentile)

    def _count_low(row):
        return sum(
            1 for c in date_cols
            if pd.notna(row[c]) and row[c] <= thresholds[c]
        )

    result = score_df.copy()
    result["low_count"] = result.apply(_count_low, axis=1).astype("Int64")
    result["Pass/Fail"] = result["low_count"].apply(
        lambda n: "Fail" if n >= min_low_count else "Pass"
    )
    return result, thresholds


def _flag_by_blr(
    folder_path: str,
    score_df: pd.DataFrame,
    id_cols: list[str],
    borderline_gr: int = 2,
    min_low_count: int = 3,
    date_regex: str = DEFAULT_DATE_REGEX,
    file_pattern: str = DEFAULT_FILE_PATTERN,
) -> tuple[pd.DataFrame, pd.Series]:
    """
    Flag students who fall at or below the BLR regression cutoff in at
    least *min_low_count* assessment dates.
    """
    files = loadWeeklyFiles(folder_path, date_regex, file_pattern)
    date_cols = [
        c for c in score_df.columns if c not in id_cols and c != "Avg Score"
    ]

    # Build a cutoff per date from BLR
    cutoffs = {}
    for date_str, df in files:
        if date_str not in date_cols:
            continue
        res = runBlrAnalysis(df, date_str, borderline_gr)
        cutoffs[date_str] = res["regressionCutoff"] / 100  # stored as pct

    thresholds = pd.Series(cutoffs).reindex(date_cols)

    def _count_low(row):
        return sum(
            1 for c in date_cols
            if pd.notna(row[c]) and pd.notna(thresholds.get(c))
            and row[c] <= thresholds[c]
        )

    result = score_df.copy()
    result["low_count"] = result.apply(_count_low, axis=1).astype("Int64")
    result["Pass/Fail"] = result["low_count"].apply(
        lambda n: "Fail" if n >= min_low_count else "Pass"
    )
    return result, thresholds


def flag_low_students(
    score_df: pd.DataFrame,
    method: FlagMethod | str = FlagMethod.PERCENTILE,
    id_cols: list[str] | None = None,
    min_low_count: int = 3,
    # percentile-specific
    percentile: float = 0.15,
    # BLR-specific
    folder_path: str | None = None,
    borderline_gr: int = 2,
    date_regex: str = DEFAULT_DATE_REGEX,
    file_pattern: str = DEFAULT_FILE_PATTERN,
) -> tuple[pd.DataFrame, pd.Series]:
    """
    Unified interface for flagging low-performing students.

    Parameters
    ----------
    method : "percentile" or "blr"
        * **percentile** – flag if score <= per-date percentile in ≥ min_low_count dates.
        * **blr** – flag if score <= BLR regression cutoff in ≥ min_low_count dates.
    percentile : float
        Only used when method="percentile".  Default 0.15 (bottom 15%).
    folder_path : str
        Required when method="blr" (needs raw files to compute regression).
    borderline_gr : int
        Global-rating value defining the borderline group (default 2).

    Returns
    -------
    (flagged_df, thresholds) – the score table with Pass/Fail + low_count
    columns, and a Series of per-date thresholds that were applied.
    """
    if id_cols is None:
        id_cols = list(DEFAULT_ID_COLS)
    method = FlagMethod(method)

    if method == FlagMethod.PERCENTILE:
        return _flag_by_percentile(score_df, id_cols, percentile, min_low_count)

    if method == FlagMethod.BLR:
        if folder_path is None:
            raise ValueError("folder_path is required for BLR flagging")
        return _flag_by_blr(
            folder_path, score_df, id_cols, borderline_gr,
            min_low_count, date_regex, file_pattern,
        )

    raise ValueError(f"Unknown method: {method}")

def exportCombinedNotebook(
    folder_path: str,
    out_path: str,
    method: FlagMethod | str = FlagMethod.PERCENTILE,
    id_cols: list[str] | None = None,
    min_low_count: int = 3,
    percentile: float = 0.15,
    borderline_gr: int = 2,
    date_regex: str = DEFAULT_DATE_REGEX,
    file_pattern: str = DEFAULT_FILE_PATTERN,
):
    """
    Build wide tables for scores / global ratings / practice readiness,
    flag low students with the chosen *method*, and write everything to a
    multi-sheet Excel workbook.
    """
    if id_cols is None:
        id_cols = list(DEFAULT_ID_COLS)

    common = dict(
        folderPath=folder_path, idCols=id_cols,
        dateRegex=date_regex, filePattern=file_pattern,
    )

    score_df = buildWideDf(valueCol="assessor_score", **common)
    global_rating_df = buildWideDf(valueCol="GR", **common)
    practice_readiness_df = buildWideDf(valueCol="PR", **common)

    flagged_df, thresholds = flag_low_students(
        score_df,
        method=method,
        id_cols=id_cols,
        min_low_count=min_low_count,
        percentile=percentile,
        folder_path=folder_path,
        borderline_gr=borderline_gr,
        date_regex=date_regex,
        file_pattern=file_pattern,
    )

    print(f"Thresholds ({method}) per date:")
    print(thresholds.sort_index())
    print()

    fail_ids = flagged_df.loc[flagged_df["Pass/Fail"] == "Fail", id_cols[0]].tolist()
    print(f"Students flagged as Fail ({len(fail_ids)}): {fail_ids}")

    with pd.ExcelWriter(out_path, engine="openpyxl") as writer:
        score_df.to_excel(writer, sheet_name="scores", index=False)
        global_rating_df.to_excel(writer, sheet_name="global_ratings", index=False)
        practice_readiness_df.to_excel(writer, sheet_name="practice_readiness", index=False)
        flagged_df.to_excel(writer, sheet_name="scores_flagged", index=False)
        ws_flagged = writer.sheets["scores_flagged"]
        highlightFlaggedCells(ws_flagged, flagged_df, thresholds, id_cols)
        
        for ws in writer.sheets.values():
            autoFitColumns(ws)

    print(f"Saved: {out_path}")


blrResultsDf = runBlrAnalysisForAllFiles(folderPath=folderPath, borderlineGr=2)
blrResultsDf.drop(columns=["df", "belowCutoffDf"], inplace=True)
blrResultsDf.to_excel(os.path.join(folderPath, "BLR_analysis_results.xlsx"), index=False)
print("Saved BLR analysis results to Excel.")

exportCombinedNotebook(
    folder_path=folderPath,
    out_path=os.path.join(folderPath, "dds2 combined scores and ratings.xlsx"),
    method=FlagMethod.BLR,
    min_low_count=3,
    borderline_gr=2,
)


#### mini-cex

In [ ]:
import os
import requests
from psycopg.types.json import Json

from dotenv import load_dotenv
from pprint import pprint
# print working directory
print(os.getcwd())

load_dotenv()

folder = r"C:\Users\Kunal Patel\D folder\MDS Work\2026"
dbUser = os.environ["DB_USER"]
dbPassword = os.environ["DB_PASSWORD"]
dbHost = os.environ["DB_HOST"]
dbPort = int(os.environ["DB_PORT"])
dbName = os.environ["DB_NAME"]

bearerToken = os.environ["DASH_TOKEN"]


dbUrl = f"postgresql+psycopg://{dbUser}:{dbPassword}@{dbHost}:{dbPort}/{dbName}"
engine = create_engine(dbUrl, pool_pre_ping=True)


createTableSql = """
CREATE TABLE IF NOT EXISTS rawForms_mcex (
  assessmentId BIGINT PRIMARY KEY,
  studentNumber BIGINT,
  studentName text,
  assessorName text,
  datetimeUtc TIMESTAMPTZ,
  cohort TEXT,
  subject TEXT,
  completed BOOLEAN,
  data JSONB,
  forms JSONB NOT NULL,
  insertedAt TIMESTAMPTZ DEFAULT now()
);

CREATE UNIQUE INDEX IF NOT EXISTS uq_rawforms_assessmentId ON rawForms_mcex (assessmentId);
CREATE INDEX IF NOT EXISTS idx_rawforms_datetimeUtc ON rawForms_mcex (datetimeUtc);
CREATE INDEX IF NOT EXISTS idx_rawforms_forms_gin ON rawForms_mcex USING GIN (forms);
"""

insertSql = """
INSERT INTO rawForms_mcex (
  assessmentId, studentNumber, studentName, assessorName, datetimeUtc,
  cohort, subject, completed, data, forms
)
VALUES (
  :assessmentId, :studentNumber, :studentName, :assessorName, :datetimeUtc,
  :cohort, :subject, :completed, :data, :forms
)
ON CONFLICT (assessmentId) DO UPDATE SET
  studentNumber = EXCLUDED.studentNumber,
  studentName   = EXCLUDED.studentName,
  datetimeUtc   = EXCLUDED.datetimeUtc,
  cohort        = EXCLUDED.cohort,
  subject       = EXCLUDED.subject,
  completed     = EXCLUDED.completed,
  data          = EXCLUDED.data,
  forms         = EXCLUDED.forms;
"""


def runDdl(conn, ddl):
    for stmt in ddl.strip().split(";"):
        if stmt.strip():
            conn.execute(text(stmt))

def fetchAllAssessments(apiUrl, bearerToken):
    headers = {
        "Authorization": f"Token {bearerToken}",
        "Accept": "application/json",
    }

    records = []
    nextUrl = apiUrl
    with requests.Session() as session:
        while nextUrl:
            resp = session.get(nextUrl, headers=headers, timeout=60)
            resp.raise_for_status()
            payload = resp.json()

            pageResults = payload.get("results")
            if not isinstance(pageResults, list):
                raise ValueError("Unexpected response: missing 'results' list")

            records.extend(pageResults)
            nextUrl = payload.get("next")  # handles pagination if present

    return records

def toRows(records):
    rows = []
    for r in records:
        if r.get("student") is None or r.get("student") == "Test Student":
            continue
        rows.append({
            "assessmentId": r.get("assessment_id") or r.get("id"),
            "studentNumber": r.get("student_number"),
            "studentName": r.get("student"),
            "datetimeUtc": r.get("datetime"),
            "assessorName": r.get("assessor"),
            "cohort": r.get("cohort"),
            "subject": r.get("subject"),
            "completed": r.get("completed") or r.get("submitted"),
            "forms": Json(r.get("forms") or r.get("form")),
            "data": Json(r["form"]["data"]["assessor"])
        })
    return rows

def insertRows(rows, batchSize=1000):
    with engine.begin() as conn:
        runDdl(conn, createTableSql)
        for i in range(0, len(rows), batchSize):
            conn.execute(text(insertSql), rows[i:i + batchSize])

def main():
    apiUrl = "https://api.unimelb-dash.com/assessment/mini-cex/v2/get?cohort=BOH1&page_size=max&page=1&year=2026"
    if not bearerToken:
        raise RuntimeError("Set DASH_TOKEN environment variable to your Bearer token")

    records = fetchAllAssessments(apiUrl, bearerToken)
    print(f"Fetched {len(records)} assessment records from API.")
    pprint(records)
    rows = toRows(records)
    # save rows to csv for inspection
    df = pd.DataFrame(rows)
    df.to_excel(f"{folder}/mcex_rows_inspection.xlsx", index=False)
    from collections import Counter

    assessmentIds = [r["assessmentId"] for r in rows]
    counts = Counter(assessmentIds)
    dupes = [aid for aid, c in counts.items() if c > 1]

    print("rows:", len(rows))
    print("distinct assessmentId:", len(set(assessmentIds)))
    print("dupe assessmentIds:", dupes[:20])

    print(f"Prepared {len(rows)} rows for insertion.")
    insertRows(rows)


if __name__ == "__main__":
    main()


Process the mcex data

In [ ]:
rubricKeys = [
    "scale-professionalism",
    "scale-time-mgmt",
    "scale-entrustment",
    "scale-communication",
    "scale-practice-readiness",
    "scale-global-rating",
    "scale-position-ergonomics"
]
cohort = 'BOH1'
commentKey = "comments" 

createTableSql = """
CREATE TABLE IF NOT EXISTS rawForms_mcex_processed(
  assessmentId BIGINT NOT NULL,
  itemCode TEXT,

  studentNumber BIGINT,
  studentName TEXT,
  assessorName TEXT,
  datetimeUtc TIMESTAMPTZ,
  cohort TEXT,
  subject TEXT,
  completed BOOLEAN,

  time_mgmt SMALLINT,
  entrustment SMALLINT,
  professionalism SMALLINT,
  communication SMALLINT,
  practice_readiness SMALLINT,
  global_rating SMALLINT,
  comments TEXT,

  dataClean JSONB NOT NULL,
  insertedAt TIMESTAMPTZ DEFAULT now(),

  PRIMARY KEY (assessmentId)
);

CREATE INDEX IF NOT EXISTS idx_mcex_proc_datetimeUtc ON rawForms_mcex_processed (datetimeUtc);
CREATE INDEX IF NOT EXISTS idx_mcex_proc_itemCode ON rawForms_mcex_processed (itemCode);
CREATE INDEX IF NOT EXISTS idx_mcex_proc_dataClean_gin ON rawForms_mcex_processed USING GIN (dataClean);
"""

upsertSql = """
INSERT INTO rawForms_mcex_processed (
  assessmentId, itemCode,
  studentNumber, studentName, assessorName, datetimeUtc,
  cohort, subject, completed,
  time_mgmt, entrustment, professionalism, communication, practice_readiness, global_rating,
  comments,
  dataClean
)
SELECT
  r.assessmentId,
  e.key AS itemCode,

  r.studentNumber,
  r.studentName,
  r.assessorName,
  r.datetimeUtc,
  r.cohort,
  r.subject,
  r.completed,

  NULLIF(r.data->'scale-time-mgmt'->>'scale','')::smallint     AS time_mgmt,
  NULLIF(r.data->'scale-entrustment'->>'scale','')::smallint   AS entrustment,
  NULLIF(r.data->'scale-professionalism'->>'scale','')::smallint     AS professionalism,
  NULLIF(r.data->'scale-communication'->>'scale','')::smallint AS communication,
  NULLIF(r.data->'scale-practice-readiness'->>'scale','')::smallint AS practice_readiness,
  NULLIF(r.data->'scale-global-rating'->>'scale','')::smallint AS global_rating,
  COALESCE(
    r.data->>:commentKey,
    r.forms->'data'->'assessor'->>:commentKey
  ) AS comments,

  e.value AS dataClean
FROM rawForms_mcex r
CROSS JOIN LATERAL jsonb_each(r.data) e(key, value)
WHERE jsonb_typeof(r.data) = 'object'
  AND e.key <> ALL(:rubricKeys)
  AND e.key <> :commentKey
  AND r.datetimeUtc >= '2026-03-03'
  AND r.cohort = :cohort
  AND r.completed
ON CONFLICT (assessmentId) DO UPDATE SET
  studentNumber = EXCLUDED.studentNumber,
  studentName   = EXCLUDED.studentName,
  assessorName  = EXCLUDED.assessorName,
  datetimeUtc   = EXCLUDED.datetimeUtc,
  cohort        = EXCLUDED.cohort,
  subject       = EXCLUDED.subject,
  completed     = EXCLUDED.completed,

  time_mgmt = EXCLUDED.time_mgmt,
  entrustment = EXCLUDED.entrustment,
  professionalism = EXCLUDED.professionalism,
  communication = EXCLUDED.communication,
  practice_readiness = EXCLUDED.practice_readiness,
  global_rating = EXCLUDED.global_rating,
  comments = EXCLUDED.comments,
  dataClean = EXCLUDED.dataClean;
"""

testsql = """
SELECT assessmentId, k
FROM rawForms_mcex r
CROSS JOIN LATERAL jsonb_object_keys(r.data) t(k)
WHERE r.completed
  AND r.cohort = :cohort
  AND datetimeUtc >= '2026-03-03'
  AND k NOT LIKE 'scale-%'
  AND k <> ALL(:rubricKeys)
  AND k <> :commentKey;
"""

def runDdl(conn, ddl):
    for stmt in ddl.strip().split(";"):
        if stmt.strip():
            conn.execute(text(stmt))

def process():
    with engine.begin() as conn:
        runDdl(conn, createTableSql)
        testdf = readDf(engine, testsql, {"rubricKeys": rubricKeys, "commentKey": commentKey, "cohort": cohort})
        display(testdf)
        print("Test query executed successfully.")
        conn.execute(
            text(upsertSql),
            {"rubricKeys": rubricKeys, "commentKey": commentKey, "cohort": cohort},
        )
        rowCount = conn.execute(text("SELECT COUNT(*) FROM rawForms_mcex_processed")).scalar()
        print(f"rawForms_mcex_processed_items rows: {rowCount}")

def main():
    process()

if __name__ == "__main__":
    main()

In [ ]:
date = "2026-05-11"
createViewSql = f"""
CREATE OR REPLACE VIEW v_mcex_{cohort.lower()}_{date.replace('-', '')}_exploded AS
SELECT
  t.assessmentid,
  t.itemcode,
  -- t.studentnumber,
  t.studentname,
  t.assessorname,
  t.datetimeutc,
  -- t.cohort,
  -- t.subject,
  -- t.completed,
  -- t.time_mgmt,
  -- t.entrustment,
  -- t.professionalism,
  -- t.communication,
  t.practice_readiness,
  t.global_rating,
  t.comments,
  t.dataclean

FROM rawforms_mcex_processed t

WHERE t.cohort = '{cohort}'
AND t.datetimeutc = '{date}'
  AND t.completed
GROUP BY
  t.assessmentid, t.itemcode
"""


def createView():
    with engine.begin() as conn:
        conn.execute(text(createViewSql))
    print(f"View v_mcex_{cohort.lower()}_{date.replace('-', '')}_exploded created/updated.")

# mcRequiredDict = {
#     "MC1":"yes","MC2":"no","MC3":"yes","MC4":"yes","MC5":"yes","MC6":"yes","MC7":"yes","MC8":"yes","MC9":"yes","MC10":"yes",
#     "MC11":"yes","MC12":"yes","MC13":"yes","MC14":"yes","MC15":"yes","MC16":"no","MC17":"yes","MC18":"yes","MC19":"no","MC20":"yes",
#     "MC21":"yes","MC22":"yes","MC23":"yes","MC24":"yes","MC25":"yes","MC26":"yes","MC27":"yes","MC28":"yes","MC29":"yes","MC30":"yes",
#     "MC31":"yes","MC32":"yes","MC33":"yes","MC34":"yes","MC35":"yes","MC36":"yes","MC37":"no","MC38":"yes","MC39":"yes","MC40":"yes",
#     "MC41":"yes","MC42":"yes","MC43":"yes","MC44":"yes","MC45":"yes","MC46":"yes","MC47":"yes","MC48":"yes","MC49":"yes","MC50":"yes",
#     "MC51":"yes","MC52":"yes","MC53":"yes","MC54":"yes","MC55":"yes","MC56":"no","MC57":"yes","MC58":"yes","MC59":"no","MC60":"yes",
#     "MC61":"yes","MC62":"yes","MC63":"yes","MC64":"yes","MC65":"yes","MC66":"yes","MC67":"yes","MC68":"yes","MC69":"yes","MC70":"yes",
#     "MC71":"yes","MC72":"yes","MC73":"yes","MC74":"yes","MC75":"yes","MC76":"yes","MC77":"yes","MC78":"yes","MC79":"yes","MC80":"yes",
#     "MC81":"yes"
# }

mcRequiredDict = {
    "MC1": "yes", "MC2": "no", "MC3": "yes", "MC4": "yes", "MC5": "yes", "MC6": "no",
    "MC7": "yes", "MC8": "no", "MC9": "yes", "MC10": "yes",
    "MC11": "no", "MC12": "no", "MC13": "yes", "MC14": "yes", "MC15": "yes", "MC16": "no", "MC17": "yes", "MC18": "no",
    "MC19": "no", "MC20": "yes", "MC21": "yes",
    "MC22": "yes", "MC23": "yes",
    "MC24": "no", "MC25": "yes", "MC26": "yes",
    "MC27": "no", "MC28": "yes", "MC29": "yes", "MC30": "yes",
    "MC31": "no", "MC32": "no", "MC33": "yes", "MC34": "yes", "MC35": "yes", "MC36": "yes", "MC37": "no", "MC38": "yes", "MC39": "no",
    "MC40": "yes", "MC41": "yes",
    "MC42": "no", "MC43": "no", "MC44": "no",
} # BOH2 specific required MCs based on rubric
mapping = {
    "Outstanding": 1.0,
    "Done well": 0.8,
    "Mostly done": 0.6,
    "Borderline": 0.4,
    "Unsatisfactory": 0.0,
    "Satisfactory": 1,
    "Yes": 1,
    "No": 0,
}

def main():
    createView()
    print(f"View created. Now reading data and calculating scores...")
    df = readDf(engine, f"SELECT * FROM v_mcex_{cohort.lower()}_{date.replace('-', '')}_exploded")
    mcDf = pd.json_normalize(df["dataclean"])
    # replace Satisfactory with 1, others with 0
    mcDf = mcDf.applymap(lambda x: 1 if x == "Yes" or x == "Satisfactory" else 0)
    

    df["Score"] = (
        df["dataclean"]
        .apply(lambda x: json.loads(x) if isinstance(x, str) else x)
        .apply(lambda d: sum(mapping.get(v, 0) for v in d.values()))
    )
    df['datetimeutc'] = pd.to_datetime(df['datetimeutc']).dt.date
    df["%Score"] = (
    df["Score"] /
    df["dataclean"].apply(lambda x: len(x) if isinstance(x, dict) and len(x) > 0 else None)
    * 100
).round(2)
    df = pd.concat([df.drop(columns=["dataclean"]), mcDf], axis=1)
    
    # determine pass/fail baseed on mcRequiredDict
    def determinePassFail(row):
        for mc, required in mcRequiredDict.items():
            if row.get(mc) is None:
                continue
            if required == "yes" and row.get(mc) != 1:
                return "Fail"
        return "Pass"
    
    # df["Pass/Fail"] = df.apply(determinePassFail, axis=1)
    display(df)

    df.to_excel(f"mcex/{cohort}/mcex {date} scores.xlsx", index=False)
    # fit the columns to headers
    with pd.ExcelWriter(f"mcex/{cohort}/mcex {date} scores.xlsx", engine="openpyxl", mode="a") as writer:
        ws = writer.sheets["Sheet1"]
        autoFitColumns(ws)

if __name__ == "__main__":
    main()

In [ ]:
filePath = "mcex/BOH1/test 1.xlsx"
folder = "mcex/BOH1"
df = pd.read_excel(filePath)

df2 = df.copy()
df = df.replace(mapping)
display(df.head())
scoreColumns = [
    "% Score",
]

# round the scores to 2 decimal places
# df[scoreColumns] = df[scoreColumns].round(2)

outputPath = f"{folder}/mcexPivotTables.xlsx"

with pd.ExcelWriter(outputPath, engine="openpyxl") as writer:
    for scoreCol in scoreColumns:
        pivotDf = pd.pivot_table(
            df,
            index="studentname",
            columns="assessorname",
            values=scoreCol,
            # aggfunc="mean"
        )
        pivotDf.to_excel(writer, sheet_name=scoreCol)

In [ ]:
def makeAssessorKdeFig(df: pd.DataFrame, scoreCol: str = "score_overall"):
    fig = plt.figure(figsize=(18, 6)) 
    for assessorName, g in df.groupby("assessorname"):
        s = pd.to_numeric(g[scoreCol], errors="coerce").dropna()
        if len(s) < 2:
            continue
        s.plot(kind="kde", label=str(assessorName))
        # dotted lines instead of solid
        # sns.kdeplot(s, label=str(assessorName), linestyle="dotted")

    ax = plt.gca()
    ax.set_xlabel("Score")
    ax.set_ylabel("Density")
    ax.set_title(f"KDE of {scoreCol} by Assessor")
    ax.legend(loc="upper left", bbox_to_anchor=(1.02, 1.0), borderaxespad=0.0)
    fig.tight_layout()
    return fig

def makeStudentAssessorBarFig(df, studentNumber, scoreCol, figsize=(10, 4)):
    if studentNumber is None:
        studentDf = df.copy()
    else:
        studentDf = df[df["studentname"] == studentNumber].copy()
    studentDf[scoreCol] = pd.to_numeric(studentDf[scoreCol], errors="coerce")

    aggDf = (
        studentDf.groupby("assessorname", dropna=False)[scoreCol]
        .mean()
        .reset_index()
        .dropna(subset=[scoreCol])
        .sort_values(scoreCol, ascending=False)
    )

    fig = plt.figure(figsize=figsize)
    ax = plt.gca()

    ax.bar(aggDf["assessorname"].astype(str), aggDf[scoreCol])
    ax.set_ylim(0, 1)  # since your mapping is 0..1
    ax.set_ylabel("Mean score")
    ax.set_title(f"{scoreCol} by Assessor")
    # put values on top of bars
    for i, v in enumerate(aggDf[scoreCol]):
        ax.text(i, v + 0.02, f"{v:.2f}", ha="center", va="bottom")
    ax.grid(axis="y")

    ax.tick_params(axis="x", labelrotation=45)
    fig.tight_layout()
    return fig

filename = f"{folder}/mcex_report_assessor differences.pdf"
# df = pd.read_excel(f"{folder}/scores.xlsx")
doc = SimpleDocTemplate(filename, pagesize=pageSize, rightMargin=rightMargin, 
                        leftMargin=leftMargin, topMargin=topMargin, bottomMargin=bottomMargin)
elements = []
elements.append(Paragraph("MCEX Report", headingStyle))
elements.append(Spacer(1, 0.2*inch))
elements.append(Paragraph("Overall Assessor analysis", subheadingStyle))
elements.append(Spacer(1, 0.1*inch))
# get kde plot
fig = makeAssessorKdeFig(df, scoreCol="% Score")
img = addPlotImage(fig)
elements.append(img)

# fig2 = makeAssessorKdeFig(df, scoreCol="score_533_577")
# img2 = addPlotImage(fig2)
# elements.append(img2)

# fig3 = makeAssessorKdeFig(df, scoreCol="score_524_578")
# img3 = addPlotImage(fig3)
# elements.append(img3)

elements.append(PageBreak())


# bar chart of overall scores of assessors
fig1 = makeStudentAssessorBarFig(df, studentNumber=None, scoreCol="% Score", figsize=(12, 5))
img1 = addPlotImage(fig1)
# elements.append(Paragraph("Overall Assessor Mean Scores", subheadingStyle))
# elements.append(Spacer(1, 0.1*inch))
# elements.append(img1)
# elements.append(Spacer(1, 0.2*inch))
# fig2 = makeStudentAssessorBarFig(df, studentNumber=None, scoreCol="score_533_577", figsize=(12, 5))
# img2 = addPlotImage(fig2)
# # elements.append(Paragraph("Assessor Mean Scores for MC 1-7", subheadingStyle))
# elements.append(Spacer(1, 0.1*inch))
# elements.append(img2)
# elements.append(Spacer(1, 0.2*inch))
# fig3 = makeStudentAssessorBarFig(df, studentNumber=None, scoreCol="score_524_578", figsize=(12, 5))
# img3 = addPlotImage(fig3)
# # elements.append(Paragraph("Assessor Mean Scores for MC 8-18", subheadingStyle))
# elements.append(Spacer(1, 0.1*inch))    
# elements.append(img3)
elements.append(PageBreak())


# for each page, I want info for each student

# sort students by name Model 1, Model 2, ... based on number in name
def studentSortKey(name):
    match = re.search(r'Model (\d+)', name)
    if match:
        return int(match.group(1))
    return float('inf')  # put non-matching names at the end
df["studentname"] = df["studentname"].astype(str)
df = df.sort_values("studentname", key=lambda col: col.map(studentSortKey))

# for studentName in df["studentname"].unique():
#     elements.append(Paragraph(f"Student: {studentName}", subheadingStyle))
#     elements.append(Spacer(1, 0.1*inch))

#     fig = makeStudentAssessorBarFig(df, studentName, scoreCol="score_overall", figsize=(10, 4))
#     img = addPlotImage(fig)
#     elements.append(img)

#     fig2 = makeStudentAssessorBarFig(df, studentName, scoreCol="score_533_577", figsize=(10, 4))
#     img2 = addPlotImage(fig2)
#     elements.append(img2)

#     fig3 = makeStudentAssessorBarFig(df, studentName, scoreCol="score_524_578", figsize=(10, 4))
#     img3 = addPlotImage(fig3)
#     elements.append(img3)

#     elements.append(PageBreak())


doc.build(elements)

#### pair wise differences

In [ ]:

levelToIndex = {
    "Unsatisfactory": 0,
    "Borderline": 1,
    "Mostly done": 2,
    "Done well": 3,
    "Outstanding": 4
}

def getAssessorPairsAndItemFlagCounts(df: pd.DataFrame, diffThreshold: int = 2):
    checklistCols = [c for c in df.columns if c.startswith("MC")]

    df2 = df.copy()
    for col in checklistCols:
        df2[col] = df2[col].map(levelToIndex)

    longDf = df2.melt(
        id_vars=["studentname", "assessorname"],
        value_vars=checklistCols,
        var_name="checklistItem",
        value_name="levelIndex"
    ).dropna(subset=["levelIndex"])

    rows = []
    for (stationName, checklistItem), g in longDf.groupby(["studentname", "checklistItem"]):
        assessorToLevel = dict(zip(g["assessorname"].astype(str), g["levelIndex"].astype(int)))
        for a1, a2 in combinations(sorted(assessorToLevel.keys()), 2):
            l1, l2 = assessorToLevel[a1], assessorToLevel[a2]
            levelDiff = abs(l1 - l2)
            if levelDiff >= diffThreshold:
                rows.append({
                    "station": stationName,
                    "checklistItem": checklistItem,
                    "assessor1": a1,
                    "assessor2": a2,
                    "score1": l1,
                    "score2": l2,
                    "levelDiff": levelDiff
                })

    pairDiffDf = pd.DataFrame(rows)

    # Pair summary per station (optional)
    stationPairSummaryDf = (
        pairDiffDf.groupby(["station", "assessor1", "assessor2"], as_index=False)
        .agg(
            nItemsFlagged=("checklistItem", "count"),
            maxDiff=("levelDiff", "max"),
            avgDiff=("levelDiff", "mean")
        )
        .sort_values(["station", "nItemsFlagged", "maxDiff"], ascending=[True, False, False])
        if len(pairDiffDf) else pairDiffDf.copy()
    )

    # NEW: checklist item flagged counts per station
    # "flagged count" here = number of assessor-pairs (within that station) that differ by >= threshold for that item
    itemFlagCountsDf = (
        pairDiffDf.groupby(["station", "checklistItem"], as_index=False)
        .agg(
            nFlaggedPairs=("levelDiff", "count"),
            maxDiff=("levelDiff", "max"),
            avgDiff=("levelDiff", "mean")
        )
        .sort_values(["station", "nFlaggedPairs", "maxDiff"], ascending=[True, False, False])
        if len(pairDiffDf) else pairDiffDf.copy()
    )

    # Optional: station totals row (sum across items)
    if len(itemFlagCountsDf):
        stationTotalsDf = (
            itemFlagCountsDf.groupby("station", as_index=False)["nFlaggedPairs"].sum()
            .rename(columns={"nFlaggedPairs": "totalFlaggedPairsAllItems"})
        )
    else:
        stationTotalsDf = itemFlagCountsDf.copy()

    return pairDiffDf, stationPairSummaryDf, itemFlagCountsDf, stationTotalsDf


def writeDisagreementExcel(
    df: pd.DataFrame,
    outPath: str,
    diffThreshold: int = 2
):
    pairDiffDf, stationPairSummaryDf, itemFlagCountsDf, stationTotalsDf = \
        getAssessorPairsAndItemFlagCounts(df, diffThreshold=diffThreshold)

    with pd.ExcelWriter(outPath, engine="openpyxl") as writer:
        pairDiffDf.to_excel(writer, sheet_name="pairDiffDetails", index=False)
        stationPairSummaryDf.to_excel(writer, sheet_name="stationPairSummary", index=False)
        itemFlagCountsDf.to_excel(writer, sheet_name="itemFlagCountsByStation", index=False)
        stationTotalsDf.to_excel(writer, sheet_name="stationTotals", index=False)

diffThreshold = 2

writeDisagreementExcel(
    df2,
    outPath=f"{folder}/mcex_assessor_disagreement_report_{diffThreshold}.xlsx",
    diffThreshold=diffThreshold
)

In [ ]:
def computePairDiffs(df: pd.DataFrame, diffThreshold: int = 2):
    checklistCols = [c for c in df.columns if c.startswith("MC")]

    df2 = df.copy()
    for col in checklistCols:
        df2[col] = df2[col].map(levelToIndex)

    longDf = df2.melt(
        id_vars=["studentname", "assessorname"],
        value_vars=checklistCols,
        var_name="checklistItem",
        value_name="levelIndex"
    ).dropna(subset=["levelIndex"])

    rows = []
    from itertools import combinations
    for (station, checklistItem), g in longDf.groupby(["studentname", "checklistItem"]):
        assessorToLevel = dict(zip(g["assessorname"].astype(str), g["levelIndex"].astype(int)))
        for a1, a2 in combinations(sorted(assessorToLevel.keys()), 2):
            l1, l2 = assessorToLevel[a1], assessorToLevel[a2]
            levelDiff = abs(l1 - l2)
            if levelDiff >= diffThreshold:
                rows.append({
                    "station": station,
                    "checklistItem": checklistItem,
                    "assessor1": a1,
                    "assessor2": a2,
                    "levelDiff": levelDiff
                })

    pairDiffDf = pd.DataFrame(rows)

    overallItemCountsDf = (
        pairDiffDf.groupby("checklistItem", as_index=False)
        .agg(
            nFlaggedPairs=("levelDiff", "count"),
            nStationsFlagged=("station", "nunique"),
            maxDiff=("levelDiff", "max"),
            avgDiff=("levelDiff", "mean")
        )
        .sort_values(["nFlaggedPairs", "nStationsFlagged", "maxDiff"], ascending=[False, False, False])
        if len(pairDiffDf) else pd.DataFrame(columns=["checklistItem","nFlaggedPairs","nStationsFlagged","maxDiff","avgDiff"])
    )

    itemFlagCountsByStationDf = (
        pairDiffDf.groupby(["station", "checklistItem"], as_index=False)
        .agg(
            nFlaggedPairs=("levelDiff", "count"),
            maxDiff=("levelDiff", "max"),
            avgDiff=("levelDiff", "mean")
        )
        .sort_values(["station", "nFlaggedPairs", "maxDiff"], ascending=[True, False, False])
        if len(pairDiffDf) else pd.DataFrame(columns=["station","checklistItem","nFlaggedPairs","maxDiff","avgDiff"])
    )

    return overallItemCountsDf, itemFlagCountsByStationDf

# -------------------------
# Report helpers
# -------------------------
def roundAvgCols(df: pd.DataFrame, colsToRound=None, decimals: int = 2) -> pd.DataFrame:
    if colsToRound is None:
        colsToRound = [c for c in df.columns if c.lower().startswith("avg")]
    outDf = df.copy()
    for c in colsToRound:
        if c in outDf.columns:
            outDf[c] = pd.to_numeric(outDf[c], errors="coerce").round(decimals)
    return outDf

def figToReportlabImage(fig, widthInch=3.3, dpi=200):
    buf = BytesIO()
    fig.savefig(buf, format="png", dpi=dpi, bbox_inches="tight")
    plt.close(fig)
    buf.seek(0)

    img = Image(buf)
    img.drawWidth = widthInch * inch
    img.drawHeight = img.drawWidth * (img.imageHeight / img.imageWidth)
    return img

def makeStationBarFig(stationDf: pd.DataFrame, figsize=(5, 3.2)):
    # bar chart of nFlaggedPairs by checklistItem (no manual colors)
    d = stationDf.sort_values(["nFlaggedPairs", "maxDiff"], ascending=[False, False]).copy()

    fig = plt.figure(figsize=figsize)
    ax = plt.gca()
    ax.bar(d["checklistItem"].astype(str), d["nFlaggedPairs"].astype(float))
    ax.set_ylabel("Flagged assessor-pairs")
    ax.set_title("Disagreement count by item")
    ax.tick_params(axis="x", labelrotation=60)
    fig.tight_layout()
    return fig

def dfToReportlabTable(df: pd.DataFrame, colOrder: list[str], maxRows: int = 40):
    showDf = df[colOrder].copy()
    if len(showDf) > maxRows:
        showDf = showDf.head(maxRows)

    data = [colOrder] + showDf.values.tolist()
    t = Table(data, repeatRows=1)
    t.setStyle(TableStyle([
        ("FONTNAME", (0, 0), (-1, 0), "Helvetica-Bold"),
        ("FONTSIZE", (0, 0), (-1, -1), 8),
        ("GRID", (0, 0), (-1, -1), 0.25, colors.black),
        ("BACKGROUND", (0, 0), (-1, 0), colors.lightgrey),
        ("VALIGN", (0, 0), (-1, -1), "MIDDLE"),
        ("ALIGN", (0, 0), (-1, 0), "CENTER"),
    ]))
    return t

# -------------------------
# Build PDF: per station table + bar chart beside it; avg rounded to 2 dp
# -------------------------
def buildStationDisagreementPdf(
    df: pd.DataFrame,
    folder: str,
    fileName: str = "mcex_item_disagreement_report.pdf",
    diffThreshold: int = 2,
    pageSize=None,
    rightMargin=36, leftMargin=36, topMargin=36, bottomMargin=36,
    headingStyle=None, subheadingStyle=None
):
    styles = getSampleStyleSheet()
    if headingStyle is None:
        headingStyle = styles["Heading1"]
    if subheadingStyle is None:
        subheadingStyle = styles["Heading2"]

    overallItemCountsDf, itemFlagCountsByStationDf = computePairDiffs(df, diffThreshold=diffThreshold)
    overallItemCountsDf = roundAvgCols(overallItemCountsDf, colsToRound=["avgDiff"], decimals=2)
    itemFlagCountsByStationDf = roundAvgCols(itemFlagCountsByStationDf, colsToRound=["avgDiff"], decimals=2)

    outPath = f"{folder}/{fileName}"
    doc = SimpleDocTemplate(
        outPath,
        pagesize=A4 if pageSize is None else pageSize,
        rightMargin=rightMargin,
        leftMargin=leftMargin,
        topMargin=topMargin,
        bottomMargin=bottomMargin
    )

    elements = []
    elements.append(Paragraph("MCEX Checklist Disagreement Report", subheadingStyle))
    elements.append(Spacer(1, 0.15 * inch))
    elements.append(Paragraph(f"Rule: assessor pair differs by ≥ {diffThreshold} levels (within-station only)", subsubheadingStyle))
    elements.append(Spacer(1, 0.2 * inch))

    # Overall table
    overallCols = ["checklistItem", "nFlaggedPairs", "nStationsFlagged", "maxDiff", "avgDiff"]
    elements.append(Paragraph("Overall: checklist items with highest disagreements", subsubheadingStyle))
    elements.append(Spacer(1, 0.1 * inch))
    elements.append(dfToReportlabTable(overallItemCountsDf, overallCols, maxRows=60))

    elements.append(PageBreak())

    # Per-station: table + bar chart BESIDE it
    stationCols = ["checklistItem", "nFlaggedPairs", "maxDiff", "avgDiff"]

    if len(itemFlagCountsByStationDf):
        for station, stationDf in itemFlagCountsByStationDf.groupby("station"):
            stationDf = stationDf.sort_values(["nFlaggedPairs", "maxDiff"], ascending=[False, False])

            elements.append(Paragraph(f"Station: {station}", headingStyle))
            elements.append(Spacer(1, 0.12 * inch))
            tableObj = createTable(stationDf, colRatio= [1, 1, 1, 1, 1], title='', tableWidth = 0.7,
                                   bottomPadding=6, topPadding=6, customTextCols=[0, 1, 2, 3, 4])
            # tableObj = dfToReportlabTable(stationDf, stationCols, maxRows=80)
            fig = makeStationBarFig(stationDf, figsize=(8, 4))
            elements.append(tableObj)
            elements.append(Spacer(1, 0.2 * inch))
            chartImg = addPlotImage(fig, 0.7)
            elements.append(chartImg)
    
            elements.append(PageBreak())
    else:
        elements.append(Paragraph("No flagged disagreements found for this threshold.", subheadingStyle))

    doc.build(elements)
    return outPath

In [ ]:

pdfPath = buildStationDisagreementPdf(
    df=df2,
    folder=folder,
    diffThreshold=diffThreshold,
    pageSize=pageSize,
    rightMargin=rightMargin,
    leftMargin=leftMargin,
    topMargin=topMargin,
    bottomMargin=bottomMargin,
    headingStyle=headingStyle,
    subheadingStyle=subheadingStyle,
    fileName=f"mcex_item_disagreement_report_{diffThreshold}.pdf"
)

#### PDF reports


In [ ]:
file = 'DDS2 mcex test\\mcex 2\\LA Mini-CEX feedback.xlsx'
file = 'mcex/BOH2/mcex mar 06 scores.xlsx'
file = 'mcex\BOH1\mcex apr 27 scores.xlsx'
df = pd.read_excel(file, sheet_name="Sheet1")
display(df.head())
folder = getFolderandFileName(file)[0]
savefolder = f"{folder}/feedback_reports"
os.makedirs(savefolder, exist_ok=True)

def sanitizeExcelText(value):
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return value
    text = str(value)
    # If it could be parsed as a formula, prefix with apostrophe
    if len(text) > 0 and text[0] in ("=", "+", "-", "@"):
        return "'" + text
    return text
# cols to add in report studentname, assessorname, comments, Pass/Fail
for i, row in df.iterrows():
    studentName = row["studentname"]
    # assessorName = row["assessorname"]
    comments = row["comments"]
    comments = comments.replace("\n", "<br/>") if isinstance(comments, str) else ""
    comments = sanitizeExcelText(comments)
    # result = row["Pass/Fail"]
    # resit = row["Proposed resit"] if row["Proposed resit"] is not None and not pd.isna(row["Proposed resit"]) else "N/A"
    pageSize = landscape(A4)
    pdfdoc = SimpleDocTemplate(f"{savefolder}/{studentName}.pdf", pagesize=pageSize, rightMargin=rightMargin, 
                        leftMargin=leftMargin, topMargin=topMargin, bottomMargin=bottomMargin)
    elements = []
    tableContent = [
        ["Student Name", studentName],
        # ["Assessor Name", assessorName],
        # ["Result", result],
        ["Comments", comments],
        # ["Proposed Resit", resit]
    ]
    tableDf = pd.DataFrame(tableContent, columns=["", " "])
    table = createTable(tableDf, colRatio = [1, 3], customTextCols=[0, 1], headerColor = uniColor, 
                        bottomPadding=6, topPadding=6, title = f"", titleStyle = subheadingStyle,
                        tableTextStyle=tableTextStyleSmall)
    elements.append(Spacer(1, 72))  # add some space at the top
    elements.append(table)
    pdfdoc.build(elements, onFirstPage = getBannerDrawer("BOH2 Mini-CEX LA (06 Mar)", studentName))

# get name and email mapping for students from dataDf.to_excel(f"DDS2/dds2 {date} assessment_data.xlsx", index=False)
studentDf = pd.read_excel(f"DDS2/dds2 2026-02-02 assessment_data.xlsx", usecols=["student_name", "student_number", "student_email"])
display(studentDf.head())
# create a mapping of student name to email
outlook = win32com.client.Dispatch("Outlook.Application")
subject = "BOH2 Mini-CEX LA Feedback Report"
body = "Please find attached your BOH2 Mini-CEX LA feedback report.\n Regards, \nKunal Patel"
def send_email(recipient_email: str, filename: str):
        """
        Send an email with the report attachment.
        :param recipient_email: The email of the recipient.
        :param filename: The report file to attach.
        """
        mail = outlook.CreateItem(0)
        mail.To = recipient_email
        mail.Subject = subject
        mail.Body = body
        mail.Attachments.Add(os.path.abspath(os.path.join(savefolder, filename)))
        mail.Send()
        print(f"Email sent to {recipient_email} with attachment {os.path.abspath(os.path.join(savefolder, filename))}\n")
nameToEmail = dict(zip(studentDf["student_name"], studentDf["student_email"]))


In [ ]:
for file in os.listdir(f"{savefolder}"):
    if file.endswith(".pdf"):
        studentName = file.replace(".pdf", "")
        email = nameToEmail.get(studentName)
        print(f"{studentName}: {email}")
        if email:
            send_email(email, file)
        else:
            print(f"No email found for {studentName}, skipping email sending.")

## DDS4 BOH3

### Data processing

In [ ]:
from boh3_dds4_utils import *
targetCohorts = ["DDS4", "BOH3"]
replaceExisting = True

In [ ]:

def processDds4BohForms():
    with engine.begin() as conn:
        createTableSql, upsertSql = getBoh3Dds4FormsProcessSql(replaceExisting)
        runDdl(conn, createTableSql)
        conn.execute(text(upsertSql), {"targetCohorts": targetCohorts})
        rowCount = conn.execute(text("SELECT COUNT(*) FROM dds4_boh3_forms")).scalar()

        # Standardize clinic names for better reporting (e.g. combine RDHM PC and Primary Care RDHM into one name)
        rdhmSql = getClinicStandardizationSql(fromNames=['RDHM PC', 'RDHM PC Emergency', 'PC', 'Primary Care RDHM'], toName='RDHM PC')
        cohealthSql = getClinicStandardizationSql(fromNames=['Cohealth footscray', 'Cohealth Footscray'], toName='Cohealth (Footscray)')
        beaufortSql = getClinicStandardizationSql(fromNames=['Beaufort and Skipton Clinic', 'Beaufort and Skipton'], toName='Beaufort and Skipton')
        panchSql = getClinicStandardizationSql(fromNames=['PANCH', 'Your Community (Preston)', 'Preston'], toName='Your Community (Preston)')
        eemSql = getClinicStandardizationSql(fromNames=['EEM', 'Eyes ears and mouth'], toName='Eyes, Ears and Mouth')
        conn.execute(text(rdhmSql))
        conn.execute(text(cohealthSql))
        conn.execute(text(beaufortSql))
        conn.execute(text(panchSql))
        conn.execute(text(eemSql))
    print(f"dds4_boh3_forms rows: {rowCount}")

processDds4BohForms()


In [ ]:
# Check on clinic names
clinicCounts = readDf(engine, "SELECT external_clinic, COUNT(*) FROM dds4_boh3_forms GROUP BY external_clinic ORDER BY external_clinic;")
display(clinicCounts)

### Cohort reports

#### Patient Stats per cohort

In [ ]:
# Avg patients per form row
for cohort in targetCohorts:
    countsDf = getPatientCountPerRow(engine, cohort)
    print(f"{cohort} avg patients per row: {countsDf['patients'].mean():.2f}")

# Patient per student (attended + FTA) — used later in summary PDF
patientperstudentBOH3 = getPatientPerStudent(engine, "BOH3").merge(
    getPatientPerStudent(engine, "BOH3", attended=False),
    how='left', on='student_name', suffixes=('_attended', '_fta'))

patientperstudentDDS4 = getPatientPerStudent(engine, "DDS4").merge(
    getPatientPerStudent(engine, "DDS4", attended=False),
    how='left', on='student_name', suffixes=('_attended', '_fta'))

#### Cohort summary PDFs

In [ ]:
dds4concerns = getAdditionalConcerns(engine, "DDS4")
boh3concerns = getAdditionalConcerns(engine, "BOH3")
dds4incidents = getClinicalIncidentSummary(engine, "DDS4")
boh3incidents = getClinicalIncidentSummary(engine, "BOH3")

# A super excel file for each cohort with multiple sheets (concerns, incidents, patient per student) can also be generated if needed using pd.ExcelWriter and writing each df to a different sheet
superExcelPathBOH3 = "BOH3_DDS4/BOH3_Summary_Details.xlsx"
superExcelPathDDS4 = "BOH3_DDS4/DDS4_Summary_Details.xlsx"

# Shared PDF style args
pdfArgs = dict(
    pageSize=pageSize, rightMargin=rightMargin, leftMargin=leftMargin,
    topMargin=topMargin, bottomMargin=bottomMargin,
    subheadingStyle=subheadingStyle, subheadingColor=uniColor,
    tableTextStyleSmall=tableTextStyleSmall, uniColor=uniColor, figSize=figSize,
)

buildCohortSummaryPdf(
    engine=engine, cohort="DDS4", outPath="BOH3_DDS4/Summary_DDS4.pdf",
    bannerTitle="Summary till date - DDS4",
    concernsDf=dds4concerns, incidentsDf=dds4incidents,
    patientPerStudentDf=patientperstudentDDS4, **pdfArgs,
    superExcelPath=superExcelPathDDS4)

buildCohortSummaryPdf(
    engine=engine, cohort="BOH3", outPath="BOH3_DDS4/Summary_BOH3.pdf",
    bannerTitle="Summary till date - BOH3",
    concernsDf=boh3concerns, incidentsDf=boh3incidents,
    patientPerStudentDf=patientperstudentBOH3, **pdfArgs,
    superExcelPath=superExcelPathBOH3)

#### Concerns & incidents Excel export

In [ ]:
combinedConcerns = pd.concat([dds4concerns, boh3concerns], ignore_index=True)
combinedIncidents = pd.concat([dds4incidents, boh3incidents], ignore_index=True)

path = "BOH3_DDS4/Concerns_CI_Report.xlsx"
with pd.ExcelWriter(path, **getmodeArgs(path)) as writer:
    combinedConcerns.to_excel(writer, sheet_name="Additional Concerns", index=False)
    combinedIncidents.to_excel(writer, sheet_name="Clinical Incidents", index=False)
    ws1 = writer.sheets["Additional Concerns"]
    autoFitColumns(ws1)
    ws2 = writer.sheets["Clinical Incidents"]
    autoFitColumns(ws2)

# Not-submitted forms
filepath = "BOH3_DDS4/Not_Submitted.xlsx"
with pd.ExcelWriter(filepath, **getmodeArgs(filepath)) as writer:
    getNotSubmitted(engine, "BOH3").to_excel(writer, sheet_name="BOH3", index=False)
    getNotSubmitted(engine, "DDS4").to_excel(writer, sheet_name="DDS4", index=False)
    ws = writer.sheets["BOH3"]
    autoFitColumns(ws)
    ws2 = writer.sheets["DDS4"]
    autoFitColumns(ws2)

#### Clinic stats per rotation

In [ ]:
createPatientStatsPerClinicPerRotation(engine, "DDS4")
createPatientStatsPerClinicPerRotation(engine, "BOH3")

### Student PDF reports

In [ ]:
studentPdfArgs = dict(
    formsTable="dds4_boh3_forms", pageSize=pageSize,
    leftMargin=leftMargin, rightMargin=rightMargin,
    topMargin=topMargin, bottomMargin=bottomMargin,
    styles=styles, subheadingStyle=subheadingStyle,
    subsubheadingStyleL=subsubheadingStyleL,
    uniColor=uniColor, tableTextStyleSmall=tableTextStyleSmall,
)

buildCohortStudentReports(engine, cohort="DDS4",
    outputDir="BOH3_DDS4/DDS4_StudentReports", **studentPdfArgs)

buildCohortStudentReports(engine, cohort="BOH3",
    outputDir="BOH3_DDS4/BOH3_StudentReports", **studentPdfArgs)

In [ ]:
buildEntrustmentTimeSeriesPdf(
    engine=engine, cohort="DDS4",
    outPath="BOH3_DDS4/DDS4_Entrustment_TimeSeries.pdf",
    bannerTitle="Entrustment & Practice Readiness - DDS4",
    subheadingStyle=subheadingStyle, uniColor=uniColor,
)

buildEntrustmentTimeSeriesPdf(
    engine=engine, cohort="BOH3",
    outPath="BOH3_DDS4/BOH3_Entrustment_TimeSeries.pdf",
    bannerTitle="Entrustment & Practice Readiness - BOH3",
    subheadingStyle=subheadingStyle, uniColor=uniColor,
)

#### Textual Excel reports

In [ ]:
exportStudentTextWorkbook(engine, cohort="DDS4", outPath="BOH3_DDS4/DDS4_Textual_Report.xlsx")
exportStudentTextWorkbook(engine, cohort="BOH3", outPath="BOH3_DDS4/BOH3_Textual_Report.xlsx")

#### Individual detailed entry (ad-hoc)

In [ ]:
assessmentId = 14632  # ← change this for the entry you want

row = readDf(engine, INDIVIDUAL_ENTRY_SQL, {"assessmentId": assessmentId}).iloc[0]
mcDf = readDf(engine, INDIVIDUAL_MC_SQL, {"assessmentId": assessmentId})

doc = SimpleDocTemplate("BOH3_DDS4/Student_Detailed_Entry_Test.pdf",
    pagesize=pageSize, rightMargin=rightMargin, leftMargin=leftMargin,
    topMargin=topMargin, bottomMargin=bottomMargin)

elements = [Spacer(1, 72)]
buildIndividualEntryPage(elements, row, mcDf,
    uniColor=uniColor, subheadingStyle=subheadingStyle,
    tableTextStyleSmall=tableTextStyleSmall)

doc.build(elements, onFirstPage=getBannerDrawer(
    "Student Detailed Entry", f"{row['student_name']} ({row['student_number']})"))

## Send Emails

In [ ]:

class StudentReportMailer:
    def __init__(self, report_folder, cohort: str, email_file: str, col_email: str, col_id: str, col_name: str, col_cohort: str, name_type: int = 1,
                  subject="Student Submission Report", body="Dear Student,\n\nPlease find attached your assessment submissions.\n\nBest regards,\nKunal Patel"):
        """
        Initialize the mailer with the report folder and student email data.
        :param report_folder: Path to the folder containing student reports.
        :param email_file: Path to the file containing student email addresses.
        :param col_email: Column name containing student emails.
        :param col_id: Column name containing student IDs.
        :param col_first_name: Column name containing student first names.
        :param col_last_name: Column name containing student last names.
        :param col_cohort: Column name containing student cohort.
        """
        self.report_folder = report_folder
        self.col_email = col_email
        self.col_id = col_id
        self.col_name = col_name
        self.col_cohort = col_cohort
        self.cohort = cohort
        self.name_type = name_type
        self.subject = subject
        self.body = body

        self.email_df = pd.read_excel(email_file)  # Read the Excel file
        self.email_df = self.email_df[self.email_df[self.col_cohort].str.lower() == self.cohort.lower()]  # Filter by cohort
        self.email_df.columns = self.email_df.columns.str.lower()
        # Convert necessary columns to string for matching
        self.email_df[self.col_id] = self.email_df[self.col_id].astype('Int64')
        self.email_df[self.col_id] = self.email_df[self.col_id].astype(str)
        self.email_df.set_index(self.col_id, inplace=True)  # Set student ID as index for easy lookup
        self.no_email_list = []
        self.outlook = win32com.client.Dispatch("Outlook.Application")
    
    def send_emails(self):
        """
        Iterate through the report folder and send emails.
        """
        files = os.listdir(self.report_folder)
        for file in files: # files are id numbers
            if file.endswith(".pdf"):
                student_id = os.path.splitext(file)[0]
                student_report_path = os.path.join(self.report_folder, file)
                student_email = self.email_df.loc[student_id, self.col_email] if student_id in self.email_df.index else None
                print(f"Processing report for student ID: {student_id}, Email: {student_email}")
                if student_email:
                    self.send_email(student_email, file)
                else:
                    self.no_email_list.append(student_id)
                    print(f"No email found for student ID: {student_id}. Skipping email.")
    
   
    def send_email(self, recipient_email: str, filename: str):
        """
        Send an email with the report attachment.
        :param recipient_email: The email of the recipient.
        :param filename: The report file to attach.
        """
        mail = self.outlook.CreateItem(0)
        mail.To = recipient_email
        mail.Subject = self.subject
        mail.Body = self.body
        mail.Attachments.Add(os.path.abspath(os.path.join(self.report_folder, filename)))
        mail.Send()
        print(f"Email sent to {recipient_email} with attachment {os.path.abspath(os.path.join(self.report_folder, filename))}\n")


mailerBOH3 = StudentReportMailer(report_folder="BOH3_DDS4\\BOH3_StudentReports", email_file=variableUtils.studentEmailFile,
                                col_email="student_email", col_id="student_number", col_name="student_name", cohort="BOH3", col_cohort="cohort", subject="BOH3 Report", 
                                body="Dear Student,\n\nPlease find attached your BOH3 assessment report.\n\nBest regards,\nKunal Patel")

mailerDDS4 = StudentReportMailer(report_folder="BOH3_DDS4\\DDS4_StudentReports", email_file=variableUtils.studentEmailFile,
                                col_email="student_email", col_id="student_number", col_name="student_name", cohort="DDS4", col_cohort="cohort", subject="DDS4 Report", 
                                body="Dear Student,\n\nPlease find attached your DDS4 assessment report.\n\nBest regards,\nKunal Patel")

mailerDDS3 = StudentReportMailer(report_folder="DDS3\\Individual Student Reports", email_file=variableUtils.studentEmailFile,
                                col_email="student_email", col_id="student_number", col_name="student_name", cohort="DDS3", col_cohort="cohort", subject="DDS3 Report", 
                                body="Dear Student,\n\nPlease find attached your DDS3 assessment report.\n\nBest regards,\nKunal Patel")
mailerBOH2 = StudentReportMailer(report_folder="BOH2\\Individual Student Reports", email_file=variableUtils.studentEmailFile,
                                col_email="student_email", col_id="student_number", col_name="student_name", cohort="BOH2", col_cohort="cohort", subject="BOH2 Report", 
                                body="Dear Student,\n\nPlease find attached your BOH2 assessment report.\n\nBest regards,\nKunal Patel")
mailerDDS2 = StudentReportMailer(report_folder="DDS2\\Individual Student Reports", email_file=variableUtils.studentEmailFile,
                                col_email="student_email", col_id="student_number", col_name="student_name", cohort="DDS2", col_cohort="cohort", subject="DDS2 Report", 
                                body="Dear Student,\n\nPlease find attached your DDS2 assessment report.\n\nBest regards,\nKunal Patel")
mailerBOH1 = StudentReportMailer(report_folder="BOH1\\Individual Student Reports", email_file=variableUtils.studentEmailFile,
                                col_email="student_email", col_id="student_number", col_name="student_name", cohort="BOH1", col_cohort="cohort", subject="BOH1 Report", 
                                body="Dear Student,\n\nPlease find attached your BOH1 assessment report.\n\nBest regards,\nKunal Patel")

In [ ]:
mailerDDS2.send_emails()

In [ ]:
mailerBOH1.send_emails()

In [ ]:
mailerBOH2.send_emails()

In [ ]:
mailerDDS3.send_emails()

In [ ]:
mailerBOH3.send_emails()

In [ ]:
mailerDDS4.send_emails()

## Archive code

### BOH2 and others

In [ ]:
def getWhereStatement(formType=None, subject=None, clinic=None):
    whereType = "AND datetimeutc>='01-01-2026'"
    params = {}

    if formType is not None:
        whereType += " AND type = :formType"
        params["formType"] = formType
    
    if subject is not None:
        whereType += " AND subject = :subject"
        params["subject"] = subject
    
    if clinic is not None:
        whereType += " AND clinic = :clinic"
        params["clinic"] = clinic


    return whereType, params

def getStudentScaleSummary(engine, cohort, formsTable="rawform_forms", formType=None, subject = None, clinic=None):
    whereType, params = getWhereStatement(formType, subject, clinic)
    params["cohort"] = cohort


    sql = f"""
    WITH base AS (
        SELECT
            student_number AS "Student ID",
            student_name AS "Student Name",
            type,

            COALESCE(
                NULLIF(assessor_data->'scale-practice-readiness'->>'scale', '')::int,
                NULLIF(assessor_data->'entrustment'->>'scale', '')::int,
                NULLIF(assessor_data->'practice-readiness'->>'scale', '')::int
            ) AS entrustment,

            COALESCE(
                NULLIF(assessor_data->'scale-professionalism'->>'scale', '')::int,
                NULLIF(assessor_data->'professionalism'->>'scale', '')::int
            ) AS professionalism,

            COALESCE(
                NULLIF(assessor_data->'scale-communication'->>'scale', '')::int,
                NULLIF(assessor_data->'communication'->>'scale', '')::int
            ) AS communication,

            COALESCE(
                NULLIF(assessor_data->'scale-time-mgmt'->>'scale', '')::int,
                NULLIF(assessor_data->'time_mgmt'->>'scale', '')::int
            ) AS timeManagement

        FROM {formsTable}
        WHERE cohort = :cohort
          {whereType}
    )
    SELECT
        "Student ID",
        "Student Name",
        type AS "Type",

        COUNT(*) FILTER (WHERE entrustment = 1) AS "Entrustment Lvl 1",
        COUNT(*) FILTER (WHERE entrustment = 2) AS "Entrustment Lvl 2",
        COUNT(*) FILTER (WHERE entrustment = 3) AS "Entrustment Lvl 3",
        COUNT(*) FILTER (WHERE entrustment = 4) AS "Entrustment Lvl 4",
        ROUND(AVG(entrustment)::numeric, 2) AS "Entrustment Average",

        COUNT(*) FILTER (WHERE professionalism = 1) AS "Professionalism Lvl 1",
        COUNT(*) FILTER (WHERE professionalism = 2) AS "Professionalism Lvl 2",
        ROUND(AVG(professionalism)::numeric, 2) AS "Professionalism Average",

        COUNT(*) FILTER (WHERE communication = 1) AS "Communication Lvl 1",
        COUNT(*) FILTER (WHERE communication = 2) AS "Communication Lvl 2",
        ROUND(AVG(communication)::numeric, 2) AS "Communication Average",

        COUNT(*) FILTER (WHERE timeManagement = 1) AS "Time Management Lvl 1",
        COUNT(*) FILTER (WHERE timeManagement = 2) AS "Time Management Lvl 2",
        COUNT(*) FILTER (WHERE timeManagement = 3) AS "Time Management Lvl 3",
        COUNT(*) FILTER (WHERE timeManagement = 4) AS "Time Management Lvl 4",
        ROUND(AVG(timeManagement)::numeric, 2) AS "Time Management Average"

    FROM base
    GROUP BY "Student ID", "Student Name", type
    ORDER BY "Student Name", type;
    """
    return readDf(engine, sql, params)

def convertToMultiLevel(df):

    columnMap = {
        "Student ID": ("", "Student ID"),
        "Student Name": ("", "Student Name"),
        "Type": ("", "Type"),

        "Entrustment Lvl 1": ("Entrustment", "Lvl 1"),
        "Entrustment Lvl 2": ("Entrustment", "Lvl 2"),
        "Entrustment Lvl 3": ("Entrustment", "Lvl 3"),
        "Entrustment Lvl 4": ("Entrustment", "Lvl 4"),
        "Entrustment Average": ("Entrustment", "Average"),

        "Professionalism Lvl 1": ("Professionalism", "Lvl 1"),
        "Professionalism Lvl 2": ("Professionalism", "Lvl 2"),
        "Professionalism Average": ("Professionalism", "Average"),

        "Communication Lvl 1": ("Communication", "Lvl 1"),
        "Communication Lvl 2": ("Communication", "Lvl 2"),
        "Communication Average": ("Communication", "Average"),

        "Time Management Lvl 1": ("Time Management", "Lvl 1"),
        "Time Management Lvl 2": ("Time Management", "Lvl 2"),
        "Time Management Lvl 3": ("Time Management", "Lvl 3"),
        "Time Management Lvl 4": ("Time Management", "Lvl 4"),
        "Time Management Average": ("Time Management", "Average"),
    }

    df.columns = pd.MultiIndex.from_tuples([columnMap[c] for c in df.columns])
    return df

def getCriticalIncidentDf(engine, cohort, formsTable="rawform_forms", formType=None, subject=None, clinic=None):
    whereType, params = getWhereStatement(formType, subject, clinic)
    params["cohort"] = cohort

    sql = f"""
    SELECT
        student_number AS "Student ID",
        student_name AS "Student Name",
        datetimeutc::date AS "Date",
        clinical_incident AS "Critical Incident"
    FROM {formsTable}
    WHERE cohort = :cohort
      {whereType}
      AND NULLIF(TRIM(clinical_incident), '') IS NOT NULL
    ORDER BY student_name, datetimeutc;
    """
    return readDf(engine, sql, params)

def getStudentFormCountDf(engine, cohort, formsTable="rawform_forms", formType=None, subject=None, clinic=None):
    whereType, params = getWhereStatement(formType, subject, clinic)
    params["cohort"] = cohort

    sql = f"""
    SELECT
        student_number AS "Student ID",
        student_name AS "Student Name",
        COUNT(*)::int AS "# Forms"
    FROM {formsTable}
    WHERE cohort = :cohort
      {whereType}
    GROUP BY student_number, student_name
    ORDER BY "# Forms" DESC, "Student Name";
    """
    return readDf(engine, sql, params)

def getStudentItemCodePivot(engine, cohort, formsTable="rawform_forms", formType=None, clinic=None):

    whereType, params = getWhereStatement(formType, clinic=clinic)
    params["cohort"] = cohort

    sql = f"""
    SELECT
        f.student_number AS "Student ID",
        f.student_name AS "Student Name",
        ic.key AS "Item Code"
    FROM {formsTable} f
    CROSS JOIN LATERAL jsonb_each(COALESCE(f.student_data,'{{}}'::jsonb)) ic
    WHERE f.cohort = :cohort
      {whereType}
    """

    df = readDf(engine, sql, params)

    pivotDf = (
        df.assign(count=1)
        .pivot_table(
            index=["Student ID", "Student Name"],
            columns="Item Code",
            values="count",
            aggfunc="sum",
            fill_value=np.nan
        )
        .reset_index()
    )

    pivotDf.columns.name = None

    return pivotDf

def getStudentItemCodeGlobalRatingPivot(engine, cohort, formsTable="rawform_forms", formType=None, clinic=None):

    whereType, params = getWhereStatement(formType, clinic=clinic)
    params["cohort"] = cohort

    sql = f"""
    SELECT
        f.student_number AS "Student ID",
        f.student_name AS "Student Name",
        ic.key AS "Item Code",
        NULLIF(f.assessor_data->'scale-global-rating'->>'scale','')::int AS "Global Rating"
    FROM {formsTable} f
    CROSS JOIN LATERAL jsonb_each(COALESCE(f.student_data,'{{}}'::jsonb)) ic
    WHERE f.cohort = :cohort
      {whereType}
      AND f.assessor_data ? 'scale-global-rating'
    """

    df = readDf(engine, sql, params)

    pivotDf = (
        df.pivot_table(
            index=["Student ID", "Student Name"],
            columns="Item Code",
            values="Global Rating",
            aggfunc="mean"
        )
        .round(2)
        .reset_index()
    )

    pivotDf.columns.name = None

    return pivotDf

def getFlaggedFormDetails(engine, cohort, formsTable="rawform_forms", formType=None, subject=None, clinic=None, 
                            globalRatingThresholds=(1,), entrustmentThresholds=(1,), includeEmptyComments=False):
    whereType, params = getWhereStatement(formType, subject, clinic)
    params['cohort'] = cohort

    globalRatingList = ",".join(str(int(x)) for x in globalRatingThresholds) if globalRatingThresholds else ""
    entrustmentList = ",".join(str(int(x)) for x in entrustmentThresholds) if entrustmentThresholds else ""

    triggerConditions = []
    triggerLabels = []

    if globalRatingList:
        triggerConditions.append(
            f"""NULLIF(assessor_data->'scale-global-rating'->>'scale', '')::int IN ({globalRatingList})"""
        )
        triggerLabels.append(
            f"""CASE
                    WHEN NULLIF(assessor_data->'scale-global-rating'->>'scale', '')::int IN ({globalRatingList})
                    THEN 'Global Rating'
                END"""
        )

    if entrustmentList:
        triggerConditions.append(
            f"""NULLIF(assessor_data->'scale-practice-readiness'->>'scale', '')::int IN ({entrustmentList})"""
        )
        triggerLabels.append(
            f"""CASE
                    WHEN NULLIF(assessor_data->'scale-practice-readiness'->>'scale', '')::int IN ({entrustmentList})
                    THEN 'Entrustment'
                END"""
        )

    if not triggerConditions:
        raise ValueError("At least one threshold list must be provided.")

    commentFilter = ""
    if not includeEmptyComments:
        commentFilter = """
        AND (
            NULLIF(TRIM(COALESCE(student_reflection, '')), '') IS NOT NULL
            OR NULLIF(TRIM(COALESCE(assessor_reflection, '')), '') IS NOT NULL
            OR NULLIF(TRIM(COALESCE(clinical_incident, '')), '') IS NOT NULL
        )
        """

    triggerExpr = " || '; ' || ".join(
        [f"COALESCE({label}, '')" for label in triggerLabels]
    )

    sql = f"""
    SELECT
        form_code AS "Form Code",
        assessmentid AS "Assessment ID",
        student_number AS "Student ID",
        student_name AS "Student Name",
        student_email AS "Student Email",
        datetimeutc::date AS "Date",
        type AS "Type",
        clinic AS "Clinic",
        assessor_name AS "Assessor Name",

        NULLIF(assessor_data->'scale-global-rating'->>'scale', '')::int AS "Global Rating",
        NULLIF(assessor_data->'scale-practice-readiness'->>'scale', '')::int AS "Entrustment",

        student_reflection AS "Student Reflection",
        assessor_reflection AS "Assessor Reflection",
        clinical_incident AS "Critical Incident",

        student_data AS "Student Checklist Responses",
        assessor_data AS "Assessor Checklist Responses",

        TRIM(BOTH '; ' FROM {triggerExpr}) AS "Triggered By"

    FROM {formsTable}
    WHERE cohort = :cohort
      {whereType}
      AND (
          {" OR ".join(triggerConditions)}
      )
      {commentFilter}
    ORDER BY student_name, datetimeutc;
    """

    df = readDf(engine, sql, params)

    jsonCols = [
        "Student Checklist Responses",
        "Assessor Checklist Responses"
    ]

    # for col in jsonCols:
    #     if col in df.columns:
    #         df[col] = df[col].apply(
    #             lambda x: json.dumps(x, indent=2, ensure_ascii=False)
    #             if isinstance(x, (dict, list))
    #             else x
    #         )

    return df

def getClinicList(engine, cohort, formsTable="rawform_forms", subject=None):
    whereType, params = getWhereStatement(subject=subject)
    params["cohort"] = cohort

    sql = f"""
    SELECT DISTINCT clinic
    FROM {formsTable}
    WHERE cohort = :cohort
      AND clinic IS NOT NULL
      {whereType}
    ORDER BY clinic;
    """
    df = readDf(engine, sql, params)
    return df["clinic"].dropna().tolist()

def getSubmissionInfo(engine, cohort, formsTable="rawform_forms", formType=None, subject=None, clinic=None):
    whereType, params = getWhereStatement(formType, subject, clinic)
    params["cohort"] = cohort

    sql = f"""
    SELECT
        student_number AS "Student ID",
        student_name AS "Student Name",
        assessor_name AS "Assessor Name",
        datetimeutc::date AS "Date",
        type AS "Type",
        clinic AS "Clinic",
        submitted_by_student AS "Submitted by Student",
        submitted_by_assessor AS "Submitted by Assessor"
    FROM {formsTable}
    WHERE cohort = :cohort
      {whereType}
      and (submitted_by_student = false OR submitted_by_assessor = false)
    ORDER BY student_name, datetimeutc;
    """
    return readDf(engine, sql, params)

def getCohortReports(cohort, subject=None, type_ = ["Simulation", "Clinic"]):
    filepathScales = f"{cohort}/Scale Information {cohort} {subject} ({today}).xlsx" if subject else f"{cohort}/Scale Information {cohort} ({today}).xlsx"
    filepathPivot = f"{cohort}/Item Code Pivot {cohort} {subject} ({today}).xlsx" if subject else f"{cohort}/Item Code Pivot {cohort} ({today}).xlsx"
    filepathSubmission = f"{cohort}/Submission Info {cohort} {subject} ({today}).xlsx" if subject else f"{cohort}/Submission Info {cohort} ({today}).xlsx"
    simScaleSummary = getStudentScaleSummary(engine, cohort, formsTable="rawform_forms", formType="Simulation", subject=subject)
    simScaleSummary = convertToMultiLevel(simScaleSummary)
    clinicScaleSummary = getStudentScaleSummary(engine, cohort, formsTable="rawform_forms", formType="Clinic", subject=subject)
    clinicScaleSummary = convertToMultiLevel(clinicScaleSummary)
    simCIDf = getCriticalIncidentDf(engine, cohort, formsTable="rawform_forms", formType='Simulation', subject=subject)
    clinicCIDf = getCriticalIncidentDf(engine, cohort, formsTable="rawform_forms", formType='Clinic', subject=subject)
    simFormCountDf = getStudentFormCountDf(engine, cohort, formsTable="rawform_forms", formType='Simulation', subject=subject)
    clinicFormCountDf = getStudentFormCountDf(engine, cohort, formsTable="rawform_forms", formType='Clinic', subject=subject)

    simPivot = getStudentItemCodePivot(engine, cohort, formsTable="rawform_forms", formType='Simulation')
    clinicPivot = getStudentItemCodePivot(engine, cohort, formsTable="rawform_forms", formType='Clinic')
    simGlobalRatingPivot = getStudentItemCodeGlobalRatingPivot(engine, cohort, formsTable="rawform_forms", formType='Simulation')
    clinicGlobalRatingPivot = getStudentItemCodeGlobalRatingPivot(engine, cohort, formsTable="rawform_forms", formType='Clinic')

    simSubmissionInfoDf = getSubmissionInfo(engine, cohort, formsTable="rawform_forms", formType='Simulation', subject=subject)
    clinicSubmissionInfoDf = getSubmissionInfo(engine, cohort, formsTable="rawform_forms", formType='Clinic', subject=subject)

    with pd.ExcelWriter(filepathScales, engine="openpyxl") as writer:
        if "Simulation" in type_:
            simScaleSummary.to_excel(writer, sheet_name="Simulation Scale Summary")
            simCIDf.to_excel(writer, sheet_name="Critical Incidents Sim", index=False)
            simFormCountDf.to_excel(writer, sheet_name="Simulation Form Count", index=False)
        if "Clinic" in type_:
            clinicScaleSummary.to_excel(writer, sheet_name="Clinic Scale Summary")
            clinicCIDf.to_excel(writer, sheet_name="Critical Incidents Clinic", index=False)
            clinicFormCountDf.to_excel(writer, sheet_name="Clinic Form Count", index=False)
    
    with pd.ExcelWriter(filepathPivot, engine="openpyxl") as writer:
        if "Simulation" in type_:
            simPivot.to_excel(writer, sheet_name="Simulation Item Code Pivot", index=False)
            simGlobalRatingPivot.to_excel(writer, sheet_name="Simulation Global Rating Pivot", index=False)
        if "Clinic" in type_:
            clinicPivot.to_excel(writer, sheet_name="Clinic Item Code Pivot", index=False)
            clinicGlobalRatingPivot.to_excel(writer, sheet_name="Clinic Global Rating Pivot", index=False)
    with pd.ExcelWriter(filepathSubmission, engine="openpyxl") as writer:
        if "Simulation" in type_:
            simSubmissionInfoDf.to_excel(writer, sheet_name="Simulation Submission Info", index=False)
        if "Clinic" in type_:
            clinicSubmissionInfoDf.to_excel(writer, sheet_name="Clinic Submission Info", index=False)

    flaggedSimDf = getFlaggedFormDetails(engine, cohort, formsTable="rawform_forms", formType='Simulation', subject=subject)
    flaggedClinicDf = getFlaggedFormDetails(engine, cohort, formsTable="rawform_forms", formType='Clinic', subject=subject)
    display(flaggedSimDf)
    display(flaggedClinicDf)

def getCohortReportsPerClinic(cohort, subject=None, type_ = ["Clinic"]):
    clinics = getClinicList(engine, cohort, formsTable="rawform_forms", subject=subject)
    filepathScales = f"{cohort}/Scale Information by Clinic {cohort} {subject} ({today}).xlsx" if subject else f"{cohort}/Scale Information by Clinic {cohort} ({today}).xlsx"
    filepathPivot = f"{cohort}/Item Code Pivot by Clinic {cohort} {subject} ({today}).xlsx" if subject else f"{cohort}/Item Code Pivot by Clinic {cohort} ({today}).xlsx"
    filepathSubmission = f"{cohort}/Submission Info by Clinic {cohort} {subject} ({today}).xlsx" if subject else f"{cohort}/Submission Info by Clinic {cohort} ({today}).xlsx"

    for clinic in clinics:
        wargsScales = getmodeArgs(filepathScales)
        wargsPivot = getmodeArgs(filepathPivot)
        wargsSubmission = getmodeArgs(filepathSubmission)
        print(f"Generating report for clinic: {clinic}")
 
        clinicScaleSummary = getStudentScaleSummary(engine, cohort, formsTable="rawform_forms", formType="Clinic", subject=subject, clinic=clinic)
        clinicScaleSummary = convertToMultiLevel(clinicScaleSummary)
        clinicCIDf = getCriticalIncidentDf(engine, cohort, formsTable="rawform_forms", formType='Clinic', subject=subject, clinic=clinic)
        clinicFormCountDf = getStudentFormCountDf(engine, cohort, formsTable="rawform_forms", formType='Clinic', subject=subject, clinic=clinic)    

        clinicPivot = getStudentItemCodePivot(engine, cohort, formsTable="rawform_forms", formType='Clinic', clinic=clinic)
        clinicGlobalRatingPivot = getStudentItemCodeGlobalRatingPivot(engine, cohort, formsTable="rawform_forms", formType='Clinic', clinic=clinic)

        with pd.ExcelWriter(filepathScales, **wargsScales) as writer:
            clinicScaleSummary.to_excel(writer, sheet_name=f"{clinic} Clinic Scale Summary")
            clinicCIDf.to_excel(writer, sheet_name=f"{clinic} Critical Incidents Clinic", index=False)
            clinicFormCountDf.to_excel(writer, sheet_name=f"{clinic} Clinic Form Count", index=False)

        with pd.ExcelWriter(filepathPivot, **wargsPivot) as writer:
            clinicPivot.to_excel(writer, sheet_name=f"{clinic} Clinic Item Code Pivot", index=False)
            clinicGlobalRatingPivot.to_excel(writer, sheet_name=f"{clinic} Clinic Global Rating Pivot", index=False)
        
        clinicSubmissionInfoDf = getSubmissionInfo(engine, cohort, formsTable="rawform_forms", formType='Clinic', subject=subject, clinic=clinic)
        with pd.ExcelWriter(filepathSubmission, **wargsSubmission) as writer:
            clinicSubmissionInfoDf.to_excel(writer, sheet_name=f"{clinic} Clinic Submission Info", index=False)


getCohortReports("DDS2")
getCohortReports("BOH2")
getCohortReports("DDS3", type_= ["Clinic"]) # DDS3 only has clinic forms
getCohortReportsPerClinic("DDS3")


In [ ]:
def getStudentData(engine, cohort, studentNumber, formsTable="rawform_forms"):
    # Self: practice readiness + reflection + CAF avg per form + overall avg
    sql = f"""
      SELECT f.*,
      f.assessor_data->'scale-practice-readiness'->>'scale' AS entrustment,
      f.assessor_data->'scale-global-rating'->>'scale' AS global_rating,
      ic.item_codes as item_codes
        FROM {formsTable} f
        LEFT JOIN LATERAL (
            SELECT array_agg(DISTINCT ic.key) AS item_codes
            FROM jsonb_each(COALESCE(f.assessor_data,'{{}}'::jsonb)) ic
            WHERE ic.key NOT LIKE 'scale-%' -- exclude scales
        ) ic ON true
      WHERE cohort = :cohort
        AND student_number = :studentNumber
        AND datetimeUtc >= TIMESTAMPTZ '2026-01-01'
       -- AND submitted_by_assessor
    """
    return readDf(engine, sql, {"cohort": cohort, "studentNumber": studentNumber})

def calcScore(row, scoreMap):
    assessorData = row['assessor_data']
    if assessorData is None or (row['role'] != 'Operator' and row['role'] is not None):
        return {}
    scores = {} # key as item code, value as score
    for itemCode, itemData in assessorData.items():
        if 'scale' in itemCode:
            continue
        if not isinstance(itemData, dict):
            continue
        itemScore = 0
        validLength = 0
        for k, v in itemData.items():
            if v in scoreMap:
                itemScore += scoreMap[v]
                validLength += 1
        itemScore = itemScore/validLength if validLength > 0 else np.nan # normalize to 0-1 range
        itemScore = round(itemScore, 2)
        scores[itemCode] = {"score": itemScore}
        # row['assessor_data'][itemCode]['score'] = itemScore
    return scores

def truncateText(textValue, maxLength=2000):
    if textValue is None:
        return ""
    textValue = str(textValue)
    if len(textValue) <= maxLength:
        return textValue
    return textValue[:maxLength] + "..."

def getColor(row):
    if row['NA_Flag']:
        return 'gray'
    elif row['Patient Complexity'] == 'complex':
        return 'red'
    else:
        return 'blue'

def plotStudentScoresTimeSeries(df, 
                      dateCol='Date', 
                      scoreDictCol='scores', 
                      scoreKey='score', 
                      fallbackKey=None,
                      title='Student Performance Over Time',
                      pageSize=variableUtils.pageSize):
    """
    Creates and returns a scatter plot of scores over time with fallback keys.

    Args:
        df (pd.DataFrame): DataFrame with date and score dict columns.
        dateCol (str): Column name for date.
        scoreDictCol (str): Column with assessor score dictionaries.
        scoreKey (str): Primary key to extract.
        fallbackKeys (list of str): List of fallback keys to use if primary key missing.
        title (str): Title of the plot.

    Returns:
        matplotlib.figure.Figure: The generated figure object.
    """

    # df[dateCol] = pd.to_datetime(df[dateCol])
    pecCodes = ['Consent', 'Record_keeping', 'infection_control', 'positioning'] + ['Record keeping', 'Positioning', 'Infection control']

    expandedRows = []
    for _, row in df.iterrows():
        if not isinstance(row[scoreDictCol], dict):
            continue
        complexity = row['assessor_data'].get('scale-patient-complexity', {}).get('scale', None) if isinstance(row['assessor_data'], dict) else None
        for itemCode, scoreDict in row[scoreDictCol].items():
            naFlag = False
            if itemCode in pecCodes:
                print(f"Skipping PEC item code: {itemCode}")
                continue
            score = scoreDict.get(scoreKey)
            print(f"Item: {itemCode}, ScoreDict: {scoreDict}, Primary Score ({scoreKey}): {score}")
            if score is None or pd.isna(score):
                if fallbackKey is not None:
                    score = scoreDict.get(fallbackKey)
                    print(f"Item: {itemCode}, ScoreDict: {scoreDict}, Fallback Score ({fallbackKey}): {score}")
                if pd.isna(score):
                    score = -5
                    naFlag = True
            if score is not None:
                expandedRows.append({
                    'Date': row[dateCol],
                    'Item': itemCode,
                    'Score': score*100, # convert to percentage
                    'Assessor Name': row['assessor_name'],
                    'NA_Flag': naFlag,
                    'Patient Complexity': complexity
                })

    expandedDf = pd.DataFrame(expandedRows)
    # sort by date and then turn date to string for better plotting
    expandedDf.sort_values('Date', inplace=True)
    expandedDf['Date'] = expandedDf['Date'].dt.strftime('%Y-%m-%d')
    expandedDf['Color'] = expandedDf.apply(getColor, axis=1)
    # naDf = expandedDf[expandedDf['NA_Flag']]
    # validDf = expandedDf[~expandedDf['NA_Flag']]

    if expandedDf.empty:
        print(f"No valid data for '{scoreKey}' or fallbacks {fallbackKey}.")
        return None

    # Plot
    fig, ax = plt.subplots(figsize=(14, 8))
    ax.scatter(expandedDf['Date'], expandedDf['Score'], color=expandedDf['Color'], label='Scores')
    # if not naDf.empty:
    #     ax.scatter(naDf['Date'], naDf['Score'], color='gray', label='All NA scores')
    #     ax.legend()

    # display(expandedDf)
    offsetCounter = defaultdict(int)
    for _, row in expandedDf.iterrows():
        key = (row['Date'], row['Score'])
        offset = offsetCounter[key] * 5
        offsetCounter[key] += 1
        ax.annotate(f"{row['Item']}",# ({row[colAssessorName]})",
                    (row['Date'], row['Score']),
                    textcoords="offset points", xytext=(10, 3+2*offset),
                    ha='center', fontsize=8)
        # ax.annotate(f"{row['assessor_name']}",
        #             (row['Date'], row['Score']),
        #             textcoords="offset points", xytext=(0, -10 - 2*offset),
        #             ha='center', fontsize=6)

    # ax.set_title(title)
    ax.set_xlabel("Date")
    ax.set_ylabel("Score (% Yes or Weighted)")
    ax.grid(True, linestyle='--', alpha=0.5)
    ax.tick_params(axis='x', rotation=45)
    ax.set_ylim(-10, 1.2*expandedDf['Score'].max())
    ax.set_yticks(range(0, int(expandedDf['Score'].max() + 1), int(max(expandedDf['Score'].max()//10, 1))))
    legendElems = [
    Line2D([0], [0], marker='o', color='w', label='Complex', markerfacecolor='red', markersize=8),
    Line2D([0], [0], marker='o', color='w', label='All NA', markerfacecolor='gray', markersize=8),
    # Line2D([0], [0], marker='o', color='w', label='Valid', markerfacecolor='blue', markersize=8),
    ]
    ax.legend(handles=legendElems, bbox_to_anchor=(0.9, 0.97), loc='upper left')
    fig.tight_layout()

    return fig

def rubricPlot(ax, studentDf, label, color, xLabelRotation=45, maxY=None):
    studentDf.dropna(subset=[label], inplace=True)
    if studentDf.empty:
        ax.text(0.5, 0.5, 'No data available', horizontalalignment='center', verticalalignment='center', transform=ax.transAxes)
        return
    ax.plot(studentDf[colDate], studentDf[label], label=label, color=color, marker='o')
    ax.set_title(label)
    # ax.set_xlabel('Date')
    if maxY is None:
        ax.set_ylim(0, studentDf[label].max() + 0.5)
        ax.set_yticks(range(0, int(studentDf[label].max()+1), 1))
        ax.set_yticklabels(range(0, int(studentDf[label].max()+1), 1))
    else:
        ax.set_ylim(0, maxY)
        ax.set_yticks(range(0, int(maxY)+1, 1))
        ax.set_yticklabels(range(0, int(maxY)+1, 1))
    # reduce size of x labels
    ax.set_xticklabels(ax.get_xticklabels(), rotation=xLabelRotation, fontsize=8)
    ax.tick_params(axis='x', rotation=xLabelRotation)
    ax.grid(True, linestyle='--', alpha=0.5)
    
def makeSafeParagraph(value):
    if pd.isna(value):
        textValue = ""
    else:
        textValue = str(value)

    textValue = escape(textValue).replace("\n", "<br/>")
    return textValue

def buildStudentReport(studentDataDf, patientInfo = False):
    elements = []
    elements.append(Spacer(1, 72))
    

    # display(studentDataDf)

    # things to put on front page is the summary table
    nForms = len(studentDataDf)
    nAssessorSubmittedForms = studentDataDf['submitted_by_assessor'].sum()
    nStudentSubmittedForms = studentDataDf['submitted_by_student'].sum()
    studentDataDf["datetimeutc"] = (pd.to_datetime(studentDataDf["datetimeutc"], utc=True).dt.tz_convert("Australia/Melbourne"))
    studentDataDf = studentDataDf[studentDataDf['submitted_by_assessor']]
    studentDataDf['scores'] = studentDataDf.apply(lambda row: calcScore(row, scoreMap), axis=1)
    studentDataDf.sort_values('datetimeutc', inplace=True)  
    display(studentDataDf)
    roleCounts = studentDataDf['role'].value_counts().to_dict()
    roleCountsText = "<br/> ".join(f"{k}: {v}" for k, v in roleCounts.items())
    entrustmentCounts = studentDataDf['entrustment'].value_counts().to_dict()
    esCountsText = "<br/> ".join(f"Lvl {k}: {v}" for k, v in sorted(entrustmentCounts.items()))
    avgGlobalRating = studentDataDf['global_rating'].dropna().astype(float).mean()
    ciDf = studentDataDf[studentDataDf['clinical_incident'].notna()][['datetimeutc', 'clinical_incident', 'assessor_name', 'clinic']]
    ciCounts = ciDf.shape[0]
    allItemCodes = studentDataDf['item_codes'].dropna().explode().value_counts().to_dict()
    topItemCodes = dict(sorted(allItemCodes.items(), key=lambda x: x[1], reverse=True)[:5])
    topItemCodesText = ", ".join(f"{k}: {v}" for k, v in topItemCodes.items())
    # display(allItemCodes)
    # row wise summary dataframe
    summaryDf = pd.DataFrame({
        "Metric": ["# Forms", "# Forms Submitted by Assessor", "# Forms Submitted by Student", "Entrustment Counts", "Average Global Rating", "Critical Incidents", "Top Item Codes"],
        "": [nForms, nAssessorSubmittedForms, nStudentSubmittedForms, esCountsText, f"{avgGlobalRating:.2f}/5" if not np.isnan(avgGlobalRating) else "N/A", f"{ciCounts}", topItemCodesText]
    })
    if patientInfo:
        meanAge = studentDataDf['patient_age'].dropna().mean()
        patientDetails = studentDataDf['patient_details'].value_counts().to_dict()
        patientDetailsText = "<br/> ".join(f"{k}: {v}" for k, v in patientDetails.items())
        # get age counts in 0-6 7-17 and 18+ buckets
        ageBuckets = {"0-6": studentDataDf[(studentDataDf['patient_age'] >= 0) & (studentDataDf['patient_age'] <= 6)].shape[0],
                      "7-17": studentDataDf[(studentDataDf['patient_age'] >= 7) & (studentDataDf['patient_age'] <= 17)].shape[0],
                      "18+": studentDataDf[studentDataDf['patient_age'] >= 18].shape[0]}
        ageCountsText = "<br/> ".join(f"{k}: {v}" for k, v in ageBuckets.items())
        summaryDf = pd.concat([summaryDf, pd.DataFrame({
            "Metric": ["Mean Patient Age", " Patient age distribution", 'Role Counts', 'Patient Details'],
            "": [f"{meanAge:.2f}" if not np.isnan(meanAge) else "N/A", ageCountsText, roleCountsText, patientDetailsText]
        })], ignore_index=True)

    summaryTable = createTable(summaryDf, title="Summary", colRatio = [2, 1], customTextCols=[0, 1], titleStyle=subheadingStyle, tableTextStyle=tableTextStyle,
                               headerColor = uniColor, bottomPadding=6, topPadding=6)
    
    elements.append(Paragraph("This is a summary report of your activity so far in 2026.      \
      For detailed information please review your completed forms in the DASH program.<br/> We are working on an interactive live dashboard for future reports.", subsubheadingStyleL))
    elements.append(summaryTable)
    elements.append(PageBreak())

    # Now a time series plot of score
    timeSeriesDf = studentDataDf[['datetimeutc', 'entrustment', 'global_rating', 'item_codes', 'scores', 'assessor_data', 'assessor_name']].copy()
    timeSeriesDf['Date'] = timeSeriesDf['datetimeutc']
    fig = plotStudentScoresTimeSeries(timeSeriesDf, dateCol='Date', scoreDictCol='scores', scoreKey='score', fallbackKey=None, title='Performance on Assessed Items Over Time')
    timeSeriesImg = addPlotImage(fig)
    elements.append(Spacer(1, 24))
    elements.append(Paragraph("Performance Over Time", subheadingStyle))
    elements.append(timeSeriesImg)
    # elements.append(PageBreak())

    # rubric plots for entrustment and global rating
    fig, axes = plt.subplots(2, 1, figsize=(14, 6))
    rubricPlotDf = studentDataDf[['datetimeutc', 'entrustment', 'global_rating']].copy()
    rubricPlotDf.rename(columns={'datetimeutc': 'Date', 'entrustment': 'Entrustment', 'global_rating': 'Global Rating'}, inplace=True)
    rubricPlotDf.sort_values('Date', inplace=True)
    rubricPlotDf['Date'] = rubricPlotDf['Date'].dt.strftime('%Y-%m-%d')
    rubricPlotDf['Entrustment'] = rubricPlotDf['Entrustment'].astype('Int64')
    rubricPlotDf['Global Rating'] = rubricPlotDf['Global Rating'].astype('Int64')
    rubricPlot(axes[0], rubricPlotDf, 'Entrustment', 'blue', maxY=4.5)
    rubricPlot(axes[1], rubricPlotDf, 'Global Rating', 'green', maxY=5.5)
    # incerease gap between two plots
    plt.subplots_adjust(hspace=0.5)
    rubricImg = addPlotImage(fig)
    elements.append(Spacer(1, 24))
    elements.append(Paragraph("Entrustment and Global Rating Over Time", subheadingStyle))
    elements.append(rubricImg)

    # reflections of both student and assessor
    reflectionsDf = studentDataDf[['datetimeutc', 'role', 'student_reflection', 'assessor_reflection']].copy()
    # replace newlines in reflections with <br/> for better display in table
    reflectionsDf['student_reflection'] = reflectionsDf['student_reflection'].apply(truncateText).str.replace('\n', '<br/>')
    reflectionsDf['assessor_reflection'] = reflectionsDf['assessor_reflection'].apply(truncateText).str.replace('\n', '<br/>')
    reflectionsDf.columns = ['Date', 'Role', 'Student Reflection', 'Assessor Reflection']
    reflectionsDf = reflectionsDf.sort_values('Date')
    reflectionsDf['Date'] = reflectionsDf['Date'].dt.strftime('%Y-%m-%d')
    reflectionsTable = createTable(reflectionsDf, title="Reflections", colRatio=[1.2, 1, 5, 5], customTextCols=[0, 1, 2, 3], titleStyle=subheadingStyle, 
                                   tableTextStyle=tableTextStyleSmall, headerColor=uniColor, bottomPadding=6, topPadding=6)
    elements.append(reflectionsTable)
    elements.append(PageBreak())


    return elements

def send_email( recipient_email: str, filename: str, subject: str = "Your Performance Report", body: str = "Please find attached your performance report."):
    """
    Send an email with the report attachment.
    :param recipient_email: The email of the recipient.
    :param filename: The report file to attach.
    """
    outlook = win32com.client.Dispatch("Outlook.Application")
    mail = outlook.CreateItem(0)
    mail.To = recipient_email
    mail.Subject = subject
    mail.Body = body
    mail.Attachments.Add(os.path.abspath(os.path.join(filename)))
    mail.Send()
    print(f"Email sent to {recipient_email} with attachment {os.path.abspath(filename)}\n")    

def buildEntireCohortStudentReports(cohort, patientInfo=False, formsTable="rawform_forms"):
    sql = f"""
    SELECT DISTINCT student_number, student_name, student_email
    FROM {formsTable}
    WHERE cohort = :cohort
      AND datetimeUtc >= TIMESTAMPTZ '2026-01-01'
    """
    studentInfoDf = readDf(engine, sql, {"cohort": cohort}).set_index("student_number")
    studentIds = studentInfoDf.index.tolist()
    savefolder = f"{cohort}/Individual Student Reports"
    os.makedirs(savefolder, exist_ok=True)
    for studentNumber in studentIds:
        # if studentNumber != 1615593:
        #     continue
        studentName = studentInfoDf.loc[studentNumber, "student_name"]
        print(f"Building report for student {studentNumber} - {studentName}")
        studentDataDf = getStudentData(engine, cohort, studentNumber, formsTable)
        filename = f"{savefolder}/{studentNumber}.pdf"
        doc = SimpleDocTemplate(filename, pagesize=pageSize,
                                rightMargin=rightMargin, leftMargin=leftMargin, topMargin=topMargin, bottomMargin=bottomMargin)
        elements = buildStudentReport(studentDataDf, patientInfo=patientInfo)
        doc.build(elements, onFirstPage = getBannerDrawer('Till Date performance report', f'{studentName} ({studentNumber})'))
        print(f"Report saved to {filename}\n")
        # send email to student with report attached
        recipient_email = studentInfoDf.loc[studentNumber, "student_email"]
        # send_email(recipient_email, filename, subject="Your Performance Report", 
                #    body=f"Dear {studentName},\n\nPlease find attached your performance report up to date. For detailed information please review your completed forms in the DASH program.\n\nBest regards,\nKunal Patel")
        # break # remove this break to generate for all students

buildEntireCohortStudentReports("DDS3", patientInfo=True)
buildEntireCohortStudentReports("BOH2", patientInfo=True)
buildEntireCohortStudentReports("DDS2", patientInfo=True)

### BOH3 and DDS4

In [ ]:
def getFullDf(engine, cohort, formsTable="dds4_boh3_forms", filters: dict = None):
    whereClause = getWhereStatement(cohort, filters) if filters else "cohort = :cohort"
    sql = f"""
    SELECT *
    FROM {formsTable}
    WHERE {whereClause};
    """
    params = {"cohort": cohort}
    if filters:
        params.update(filters)
    return readDf(engine, sql, params)

def getTotalForms(engine, cohort, formsTable="dds4_boh3_forms", filters: dict = None):
    whereClause = getWhereStatement(cohort, filters) if filters else "cohort = :cohort"
    sql = f"""
    SELECT COUNT(*)::bigint AS totalForms
    FROM {formsTable}
    WHERE {whereClause};
    """
    params = {"cohort": cohort}
    if filters:
        params.update(filters)
    return readDf(engine, sql, params)

def getAgeCounts(engine, cohort, formsTable="dds4_boh3_forms", filters: dict = None):
    whereClause = getWhereStatement(cohort, filters) if filters else "cohort = :cohort"
    sql = f"""
    SELECT
      COUNT(*) FILTER (WHERE age BETWEEN 0 AND 6)::bigint  AS age0to6,
      COUNT(*) FILTER (WHERE age BETWEEN 7 AND 17)::bigint AS age7to17,
      COUNT(*) FILTER (WHERE age >= 18)::bigint           AS age18plus
    FROM (
      SELECT NULLIF(pd->>'patientAge','')::int AS age
      FROM {formsTable}
      CROSS JOIN LATERAL jsonb_array_elements(COALESCE(patient_data, '[]'::jsonb)) pd
      WHERE {whereClause}
      AND (pd->>'patientAttended')::boolean = true
    ) x;
    """
    params = {"cohort": cohort}
    if filters:
        params.update(filters)
    return readDf(engine, sql, params)

def getAgeList(engine, cohort, formsTable="dds4_boh3_forms", filters: dict = None, paramsAdd: dict = None):
    whereClause = getWhereStatement(cohort, filters) if filters else "cohort = :cohort"
    sql = f"""
    SELECT
      NULLIF(pd->>'patientAge','')::int AS age
    FROM {formsTable}
    CROSS JOIN LATERAL jsonb_array_elements(COALESCE(patient_data, '[]'::jsonb)) pd
    WHERE {whereClause}
      AND NULLIF(pd->>'patientAge','') ~ '^\d+$';  -- only numeric ages
    """
    params = {"cohort": cohort}
    if paramsAdd: # addtional params for age range filtering, etc; keys should match placeholders in whereClause
        params.update(paramsAdd)
    if filters:
        params.update(filters)
    return readDf(engine, sql, params)

def getAvgAge(engine, cohort, formsTable="dds4_boh3_forms", filters: dict = None):
    whereClause = getWhereStatement(cohort, filters) if filters else "cohort = :cohort"
    sql = f"""
    SELECT
      ROUND(AVG(
      CASE
        WHEN (pd->>'patientAge') ~ '^\s*\d+\s*$'
        AND (pd->>'patientAge')::int BETWEEN 0 AND 120
        THEN (pd->>'patientAge')::int
        ELSE NULL
      END
    )::numeric, 2) AS avgAge
    FROM {formsTable}
    CROSS JOIN LATERAL jsonb_array_elements(COALESCE(patient_data, '[]'::jsonb)) pd
    WHERE {whereClause};
    """
    params = {"cohort": cohort}
    if filters:
        params.update(filters)
    return readDf(engine, sql, params)

def getPatientsPerStudentStats(engine, cohort, formsTable="dds4_boh3_forms", attended = True, filters: dict = None):
    whereClause = getWhereStatement(cohort, filters) if filters else "cohort = :cohort"   
    sql = f"""
    WITH perStudent AS (
      SELECT
        student_name,
        COUNT(*)::int AS patients
      FROM {formsTable}
      CROSS JOIN LATERAL jsonb_array_elements(COALESCE(patient_data, '[]'::jsonb)) pd
      WHERE {whereClause}
        AND student_name IS NOT NULL
        AND (pd->>'patientAttended')::boolean = :attended
      GROUP BY student_name
    )
    SELECT
      ROUND(AVG(patients)::numeric, 2) AS avgPatientsPerStudent,
      MIN(patients) AS minPatientsPerStudent,
      MAX(patients) AS maxPatientsPerStudent
    FROM perStudent;
    """
    params = {"cohort": cohort, "attended": attended}
    if filters:
        params.update(filters)
    return readDf(engine, sql, params)

def getPatientPerStudent(engine, cohort, formsTable="dds4_boh3_forms", attended = True, filters: dict = None):
    whereClause = getWhereStatement(cohort, filters) if filters else "cohort = :cohort"
    sql = f"""
        SELECT
      f.student_name,
      COUNT(pd)::int AS patients
    FROM {formsTable} f
    LEFT JOIN LATERAL (
      SELECT *
      FROM jsonb_array_elements(
        CASE
          WHEN jsonb_typeof(f.patient_data) = 'array' THEN f.patient_data
          ELSE '[]'::jsonb
        END
      ) elem
      WHERE (elem->>'patientAttended')::boolean = :attended
    ) pd ON TRUE
    WHERE {whereClause}
      AND f.student_name IS NOT NULL
    GROUP BY f.student_name
    ORDER BY patients DESC, f.student_name;

    """
    params = {"cohort": cohort, "attended": attended}
    if filters:
        params.update(filters)
    return readDf(engine, sql, params)

def getClinicForStudent(engine, cohort, formsTable="dds4_boh3_forms", filters: dict = None):
    whereClause = getWhereStatement(cohort, filters) if filters else "cohort = :cohort"
    sql = f"""
    SELECT
      student_name,
      COALESCE(NULLIF(external_clinic, ''), '(unknown)') AS clinic
    FROM {formsTable}
    WHERE {whereClause}
      AND student_name IS NOT NULL
    ORDER BY student_name, datetimeutc DESC;
    """
    params = {"cohort": cohort}
    if filters:
        params.update(filters)
    return readDf(engine, sql, params)

def getClinicPatientCounts(engine, cohort, formsTable="dds4_boh3_forms", attended = True, filters: dict = None):
    # uses external_clinic as clinic label; counts patients via patient_data array
    whereClause = getWhereStatement(cohort, filters) if filters else "cohort = :cohort"
    sql = f"""
    SELECT
      COALESCE(NULLIF(external_clinic,''), '(unknown)') AS clinic,
      COUNT(*)::bigint AS patients
    FROM {formsTable}
    CROSS JOIN LATERAL jsonb_array_elements(COALESCE(patient_data, '[]'::jsonb)) pd
    WHERE {whereClause}
      AND (pd->>'patientAttended')::boolean = :attended
    GROUP BY 1
    ORDER BY patients DESC, clinic;
    """
    params = {"cohort": cohort, "attended": attended}
    if filters:
        params.update(filters)
    return readDf(engine, sql, params)

def getSubmittedCounts(engine, cohort, formsTable="dds4_boh3_forms", filters: dict = None):
    whereClause = getWhereStatement(cohort, filters) if filters else "cohort = :cohort"
    sql = f"""
    SELECT
      COUNT(*) FILTER (WHERE submitted_by_student IS TRUE)::bigint  AS submitted_By_Student,
      COUNT(*) FILTER (WHERE submitted_by_assessor IS TRUE)::bigint AS submitted_By_Assessor
    FROM {formsTable}
    WHERE {whereClause};
    """
    params = {"cohort": cohort}
    if filters:
        params.update(filters)
    return readDf(engine, sql, params)

def getTopItemCodes(engine, cohort, formsTable="dds4_boh3_forms", limit=5, filters: dict = None):
    whereClause = getWhereStatement(cohort, filters) if filters else "cohort = :cohort"

    # Build age filter from the lateral-joined pd
    ageClause = ""
    if filters:
        if "age_min" in filters:
            ageClause += " AND NULLIF(pd->>'patientAge','')::int >= :age_min"
        if "age_max" in filters:
            ageClause += " AND NULLIF(pd->>'patientAge','')::int <= :age_max"

    sql = f"""
    SELECT
      ic->>'code' AS itemCode,
      COUNT(*)::bigint AS freq
    FROM {formsTable} f
    CROSS JOIN LATERAL jsonb_array_elements(COALESCE(f.patient_data, '[]'::jsonb)) pd
    CROSS JOIN LATERAL jsonb_array_elements(COALESCE(pd->'itemCodes', '[]'::jsonb)) ic
    WHERE {whereClause}
      AND ic ? 'code'
      {ageClause}
    GROUP BY 1
    ORDER BY freq DESC, itemCode
    LIMIT :limit;
    """
    params = {"cohort": cohort, "limit": int(limit)}
    if filters:
        params.update(filters)
    return readDf(engine, sql, params)

def getCafFinalEvalScoreStudent(engine, cohort, formsTable="dds4_boh3_forms", filters: dict = None):
    # Student checklist options: Done well / Done / Mostly done / Sometimes done / Not done
    # Map to 1 / 0.8 / 0.6 / 0.4 / 0
    whereClause = getWhereStatement(cohort, filters) if filters else "cohort = :cohort"
    sql = f"""
    WITH itemScores AS (
      SELECT
        f.assessmentid,
        f.form_code,
        f.cohort,
        kv.key AS mc,
        CASE kv.value
          WHEN 'Done well'     THEN 1.0
          WHEN 'Done'          THEN 0.8
          WHEN 'Mostly done'   THEN 0.6
          WHEN 'Sometimes done' THEN 0.4
          WHEN 'Not done'      THEN 0.0
          ELSE NULL
        END::numeric AS score
      FROM {formsTable} f
      CROSS JOIN LATERAL jsonb_each_text(
        COALESCE(f.student_data->'checklists'->'checklist-caf-final-eval', '{{}}'::jsonb)
      ) kv(key, value)
      WHERE {whereClause}
    )
    SELECT
      cohort,
      ROUND(AVG(score), 4)::numeric(6,4) AS avgStudentChecklistScore
    FROM itemScores
    WHERE score IS NOT NULL
    GROUP BY cohort;
    """
    params = {"cohort": cohort}
    if filters:
        params.update(filters)
    return readDf(engine, sql, params)

def getAdditionalConcerns(engine, cohort, formsTable="dds4_boh3_forms", filters: dict = None):
    whereClause = getWhereStatement(cohort, filters) if filters else "cohort = :cohort"
    sql = f"""
    SELECT cohort, student_name, date_trunc('day', datetimeutc) AS datetimeutc, assessor_name, additional_concerns
    FROM {formsTable}
    WHERE {whereClause}
      AND additional_concerns IS NOT NULL
      AND TRIM(additional_concerns) <> '';
    """
    params = {"cohort": cohort}
    if filters:
        params.update(filters)
    df = readDf(engine, sql, params)
    df["datetimeutc"] = pd.to_datetime(df["datetimeutc"]).dt.date
    return df

def getClinicalIncidentSummary(engine, cohort, formsTable="dds4_boh3_forms", filters: dict = None):
    """
    Returns 1 row per (assessmentId, form_code) with:
    - hasClinicalIncident (bool)
    - clinicalIncidents (semicolon-joined descriptions)
    """
    whereClause = getWhereStatement(cohort, filters) if filters else "cohort = :cohort"
    sql = f"""
    SELECT
      f.cohort AS cohort,
      date_trunc('day', f.datetimeutc) AS datetimeutc,
      f.assessor_name AS assessor_name,
      f.student_name AS student_name,
      STRING_AGG(ci->>'value', '; ' ORDER BY ci->>'name') AS clinical_incidents
    FROM {formsTable} f
    LEFT JOIN LATERAL jsonb_array_elements(
      f.assessor_data->'multi-select'->'clinical-incident'
    ) AS ci ON TRUE
    WHERE {whereClause}
      AND ci IS NOT NULL
    GROUP BY f.cohort, date_trunc('day', f.datetimeutc), f.assessor_name, f.student_name
    ORDER BY date_trunc('day', f.datetimeutc), f.assessor_name;
    """
    params = {"cohort": cohort}
    if filters:
        params.update(filters)
    df = readDf(engine, sql, params)
    df["datetimeutc"] = pd.to_datetime(df["datetimeutc"]).dt.date
    return df

def getPatientCountPerRow(engine, cohort, formsTable="dds4_boh3_forms", attended = True, filters: dict = None):
    whereClause = getWhereStatement(cohort, filters) if filters else "cohort = :cohort"
    sql = f"""
        SELECT
      assessmentid,
      form_code,
      student_name,
      external_clinic,
      rotation,
      COUNT(pd) AS patients
    FROM {formsTable}
    LEFT JOIN LATERAL (
      SELECT *
      FROM jsonb_array_elements(
        CASE
          WHEN jsonb_typeof(patient_data) = 'array' THEN patient_data
          ELSE '[]'::jsonb
        END
      ) elem
      WHERE (elem->>'patientAttended')::boolean = :attended
    ) pd ON TRUE
    WHERE {whereClause}
    GROUP BY assessmentid, form_code, student_name;

    """
    params = {"cohort": cohort, "attended": attended}
    if filters:
        params.update(filters)
    return readDf(engine, sql, params)

def getEntrustmentSummary(engine, cohort, formsTable="dds4_boh3_forms", filters: dict = None):
    whereClause = getWhereStatement(cohort, filters) if filters else "cohort = :cohort"
    sql = f"""
    SELECT
      CASE NULLIF(assessor_data->'scales'->'scale-entrustment'->>'scale','')
        WHEN 'S1' THEN 1
        WHEN 'S2' THEN 2
        WHEN 'S3' THEN 3
        WHEN 'S4' THEN 4
      END::smallint AS entrustment,
      COUNT(*)::bigint AS cnt
    FROM {formsTable}
    WHERE {whereClause}
      AND assessor_data->'scales'->'scale-entrustment'->>'scale' IS NOT NULL
    GROUP BY entrustment
    ORDER BY entrustment;
    """
    params = {"cohort": cohort}
    if filters:
        params.update(filters)
    return readDf(engine, sql, params)

def getAvgEntrustment(engine, cohort, formsTable="dds4_boh3_forms", filters: dict = None):
    whereClause = getWhereStatement(cohort, filters) if filters else "cohort = :cohort"
    sql = f"""
    SELECT
      ROUND(AVG(
        CASE NULLIF(assessor_data->'scales'->'scale-entrustment'->>'scale','')
          WHEN 'S1' THEN 1
          WHEN 'S2' THEN 2
          WHEN 'S3' THEN 3
          WHEN 'S4' THEN 4
          ELSE NULL
        END
      )::numeric, 2) AS avgEntrustment
    FROM {formsTable}
    WHERE {whereClause};
    """
    params = {"cohort": cohort}
    if filters:
        params.update(filters)
    return readDf(engine, sql, params)

def getNotSubmitted(engine, cohort, formsTable="dds4_boh3_forms", filters: dict = None):
    whereClause = getWhereStatement(cohort, filters) if filters else "cohort = :cohort"
    sql = f"""
    SELECT assessmentid, form_code, student_name, assessor_name, datetimeutc::date AS datetimeutc, submitted_by_student, submitted_by_assessor, external_clinic, rotation
    FROM {formsTable}
    WHERE {whereClause}
      AND (NOT submitted_by_student 
      OR NOT submitted_by_assessor);
    """
    params = {"cohort": cohort}
    if filters:
        params.update(filters)
    return readDf(engine, sql, params)

def createPatientStatsPerClinic(engine, cohort, formsTable="dds4_boh3_forms", attended = True, filters: dict = None):
  countsperrow = getPatientCountPerRow(engine, cohort, formsTable, attended, filters)
  studentClinicDf = (countsperrow.groupby(["external_clinic", "student_name"], as_index=False).agg(totalPatients=("patients", "sum")))
  clinicStats = studentClinicDf.groupby("external_clinic")["totalPatients"].agg(avg="mean", min="min", max="max").reset_index()
  clinicStats["patientsSummary"] = clinicStats["avg"].astype(int).astype(str) + " (" + clinicStats["min"].astype(str) + "-" + clinicStats["max"].astype(str) + ")"
  resultDf = clinicStats[["external_clinic", "patientsSummary"]]
  return resultDf, studentClinicDf

countperrowdds4 = getPatientCountPerRow(engine, "DDS4")
countsperrowboh3 = getPatientCountPerRow(engine, "BOH3")
# display(countsperrowboh3)
# get avg patient count per row for each cohort
avgdds4 = countperrowdds4["patients"].mean()
avgboh3 = countsperrowboh3["patients"].mean()
print(f"DDS4 avg patients per row: {avgdds4:.2f}")
print(f"BOH3 avg patients per row: {avgboh3:.2f}")


patientperstudentBOH3 = getPatientPerStudent(engine, "BOH3")
patientFTAperstudentBOH3 = getPatientPerStudent(engine, "BOH3", attended=False)
clinicperstudentBOH3 = getClinicForStudent(engine, "BOH3")
# display(clinicperstudentBOH3.head())
patientperstudentBOH3 = patientperstudentBOH3.merge(patientFTAperstudentBOH3, how='left', left_on='student_name', right_on='student_name', suffixes=('_attended', '_fta'))
# patientperstudentBOH3 = patientperstudentBOH3.merge(clinicperstudentBOH3, how='left', left_on='student_name', right_on='student_name')
# display(patientperstudentBOH3.head(10))

patientperstudentDDS4 = getPatientPerStudent(engine, "DDS4")
patientFTAperstudentDDS4 = getPatientPerStudent(engine, "DDS4", attended=False)
# clinicperstudentDDS4 = getClinicForStudent(engine, "DDS4")
patientperstudentDDS4 = patientperstudentDDS4.merge(patientFTAperstudentDDS4, how='left', left_on='student_name', right_on='student_name', suffixes=('_attended', '_fta'))
# patientperstudentDDS4 = patientperstudentDDS4.merge(clinicperstudentDDS4, how='left', left_on='student_name', right_on='student_name')
# display(patientperstudentDDS4.head(10))


def getFrontPageSummaryTable(engine, cohort, formsTable="dds4_boh3_forms", filters: dict = None):
    totalFormsDf = getTotalForms(engine, cohort, formsTable, filters)
    ageDf = getAgeCounts(engine, cohort, formsTable, filters)
    avgAge = getAvgAge(engine, cohort, formsTable, filters)
    agelist = getAgeList(engine, cohort, formsTable, filters)
    # get average from agelist and round to 2 decimals, ignoring nulls
    avgAgeFromList = round(agelist["age"].mean(), 2) if not agelist.empty else None
    print(agelist)
    print(f"Average age from list: {avgAgeFromList}", "Average age from SQL: ", avgAge.iloc[0]['avgage'] if not avgAge.empty else "NA")
    ptsDf = getPatientsPerStudentStats(engine, cohort, formsTable, filters = filters)
    clinicDf = getClinicPatientCounts(engine, cohort, formsTable, filters = filters)
    submitDf = getSubmittedCounts(engine, cohort, formsTable, filters = filters)
    topItemsDf = getTopItemCodes(engine, cohort, formsTable, limit=5, filters=filters)
    cafScoreDf = getCafFinalEvalScoreStudent(engine, cohort, formsTable, filters = filters)
    entrustmentDf = getEntrustmentSummary(engine, cohort, formsTable, filters = filters)
    # display(entrustmentDf)
    # ---- extract scalars safely ----
    totalForms = int(totalFormsDf.iloc[0]["totalforms"])

    age0to6 = int(ageDf.iloc[0]["age0to6"])
    age7to17 = int(ageDf.iloc[0]["age7to17"])
    age18plus = int(ageDf.iloc[0]["age18plus"])
    avgAge = float(avgAge.iloc[0]["avgage"]) if not avgAge.empty else None

    avgPts = ptsDf.iloc[0]["avgpatientsperstudent"]
    minPts = ptsDf.iloc[0]["minpatientsperstudent"]
    maxPts = ptsDf.iloc[0]["maxpatientsperstudent"]

    submittedStudent = int(submitDf.iloc[0]["submitted_by_student"])
    submittedAssessor = int(submitDf.iloc[0]["submitted_by_assessor"])

    cafScore = cafScoreDf.iloc[0]["avgstudentchecklistscore"] if not cafScoreDf.empty else None

    # ---- format complex fields ----
    ageStr = f"0–6: {age0to6}<br/> 7–17: {age7to17}<br/> 18+: {age18plus}"
    patientStatsStr = f"{int(avgPts)} ({int(minPts)} - {int(maxPts)})"

    clinicStr = "<br/>".join(
        f"{row.clinic}: {row.patients}"
        for _, row in clinicDf.iterrows()
    )

    # topItemsStr = "<br/> ".join(
    #     f"{row.itemcode}: {row.freq}"
    #     for _, row in topItemsDf.iterrows()
    # )
    topItemsStr = ", ".join(f"{row.itemcode}" for _, row in topItemsDf.iterrows())

    cafScoreStr = f"{cafScore:.2f}" if cafScore is not None else "NA"

    entrustmentStr = "<br/>".join(
        f"S{int(row.entrustment)}: {row.cnt}" 
        for _, row in entrustmentDf.iterrows()
    )

    # ---- final one-row table ----
    summary = {
        # "Cohort": cohort,
        "Total forms": totalForms,
        "Submitted by students": submittedStudent,
        "Submitted by assessors": submittedAssessor,
        "Patient age distribution": ageStr,
        "Average patient age": f"{avgAge:.2f}" if avgAge is not None else "NA",
        "Patients per student (avg/min/max)": patientStatsStr,
        "Clinic patient counts": clinicStr,
        "Top item codes": topItemsStr,
        "Avg CAF final eval score (student)": cafScoreStr,
        "Entrustment levels": entrustmentStr,
        "Avg Entrustment": getAvgEntrustment(engine, cohort).iloc[0,0]
    }

    return pd.DataFrame([summary])


def plotItemCodeFrequencies(itemCodesDf, ax, uniColor= variableUtils.uniColor):
    ax.bar(itemCodesDf["itemcode"], itemCodesDf["freq"], color=uniColor)
    ax.set_title("Top Item Codes", fontsize=14, color=uniColor)
    ax.set_xlabel("")
    ax.set_ylabel("Frequency", fontsize=10, color=uniColor)
    ax.set_xticklabels(itemCodesDf["itemcode"], rotation=90, ha='right', fontsize=8, color=uniColor)
    ax.set_ylim(0, itemCodesDf["freq"].max() * 1.2)  # add some space on top for labels
    print("Max count:", itemCodesDf['freq'].max())
    for i, row in itemCodesDf.iterrows():
        ax.text(i, row["freq"], str(row["freq"]), ha='center', va='bottom', fontsize=6, color=uniColor)

def buildFrontPage(engine, cohort, filters, elements:list, subheadingColor):
    metrics = getFrontPageSummaryTable(engine, cohort, filters=filters).transpose()
    # display(metrics)

    metrics = metrics.reset_index()
    metrics.columns = ["Metric", "Value"]


    summaryTable = createTable(metrics, colRatio=[2, 1], customTextCols=[0, 1], bottomPadding=6,
        topPadding=6, title="", titleStyle=subheadingStyle, headerColor=subheadingColor, tableTextStyle=tableTextStyleSmall)

    
    elements.append(summaryTable)

    # a graph of all item codes and their frequencies (only top 20 item codes)
    allItemCodesDf = getTopItemCodes(engine, cohort, limit=25, filters=filters)
    fig, ax = plt.subplots(figsize=(figSize[0], figSize[1]/4))
    plotItemCodeFrequencies(allItemCodesDf, ax, uniColor=uniColor)
    plt.close(fig)
    img = addPlotImage(fig, 0.9)
    elements.append(Spacer(1, 24))
    elements.append(img)  


def buildCohortSummaryPdf(*, engine, cohort, outPath, bannerTitle,
    subheadingStyle,subheadingColor, concernsDf, incidentsDf, patientPerStudentDf):

    elements = []
    elements.append(Spacer(1, 72))
    doc = SimpleDocTemplate(outPath, pagesize=pageSize, rightMargin=rightMargin, leftMargin=leftMargin, topMargin=topMargin, bottomMargin=bottomMargin)

    buildFrontPage(engine, cohort, filters=None, elements=elements, subheadingColor=subheadingColor)
    elements.append(PageBreak())

    # plot item codes for each age group (0-6, 7-17, 18+) in 3 subplots on the same page, with a common title "Top Item Codes by Age Group"
    ageGroups = [("0-6", {"age_min":0, "age_max":6}), ("7-17", {"age_min":7, "age_max":17}), ("18+", {"age_min":18, "age_max":120})]
    fig, axs = plt.subplots(3, 1, figsize=(figSize[0], figSize[1]))
    for ax, (ageGroupName, ageFilter) in zip(axs, ageGroups):
        itemCodeDf = getTopItemCodes(engine, cohort, limit=20, filters=ageFilter)
        plotItemCodeFrequencies(itemCodeDf, ax, uniColor=uniColor)
        ax.set_title(f"Age {ageGroupName}", fontsize=12, color=uniColor)
    plt.suptitle("Top Item Codes by Age Group", fontsize=14, color=uniColor)
    #increase spacing between subplots
    plt.subplots_adjust(hspace=0.5)
    plt.close(fig)
    img = addPlotImage(fig, 0.9)
    elements.append(img)
    elements.append(PageBreak())

    # only for rotation 1
    elements.append(Paragraph("Rotation 1 Summary", subheadingStyle))
    buildFrontPage(engine, cohort, filters={"rotation": "Rotation 1"}, elements=elements, subheadingColor=subheadingColor)
    elements.append(PageBreak())

    # only for rotation 2
    elements.append(Paragraph("Rotation 2 Summary", subheadingStyle))
    buildFrontPage(engine, cohort, filters={"rotation": "Rotation 2"}, elements=elements, subheadingColor=subheadingColor)
    elements.append(PageBreak())

    # only for rotation 3
    elements.append(Paragraph("Rotation 3 Summary", subheadingStyle))
    buildFrontPage(engine, cohort, filters={"rotation": "Rotation 3"}, elements=elements, subheadingColor=subheadingColor)
    # elements.append(PageBreak())


    hasConcerns = concernsDf is not None and not concernsDf.empty
    hasIncidents = incidentsDf is not None and not incidentsDf.empty

    if hasConcerns or hasIncidents:
        elements.append(PageBreak())

        if hasConcerns:
            concernsTable = createTable( concernsDf, colRatio=[1, 1, 1, 1, 3], customTextCols=list(range(concernsDf.shape[1])),
                bottomPadding=6, topPadding=6, title="Additional Concerns", titleStyle=subheadingStyle, headerColor=subheadingColor,
                tableTextStyle=tableTextStyleSmall)
            elements.append(concernsTable)
            elements.append(Spacer(1, 24))

        if hasIncidents:
            incidentsTable = createTable( incidentsDf, colRatio=[1, 1, 1, 1, 3], customTextCols=list(range(incidentsDf.shape[1])),
                bottomPadding=6, topPadding=6, title="Clinical Incidents", titleStyle=subheadingStyle, headerColor=subheadingColor,
                tableTextStyle=tableTextStyleSmall)
            elements.append(incidentsTable)

    # Table of patient count per student; columns: Student Name, Patient Count (attended), Patient Count (FTA)
    # add a column of age counts for each student: Age 0-6, Age 7-17, Age 18+; this requires joining with the patient count per row data to get the ages of patients for each student
    dfageCountsPerPatient = pd.DataFrame(columns = ['student_name', 'agetext'])
    for _, row in patientPerStudentDf.iterrows():
        student = row['student_name']
        counts = getAgeCounts(engine, cohort, filters={"student_name": student})
        ageText = f"0-6: {counts.iloc[0]['age0to6']}<br/> 7-17: {counts.iloc[0]['age7to17']}<br/> 18+: {counts.iloc[0]['age18plus']}"
        dfageCountsPerPatient = pd.concat([dfageCountsPerPatient, pd.DataFrame({"student_name": [student], "Age Count": [ageText]})], ignore_index=True)
    patientPerStudentDf = patientPerStudentDf.merge(dfageCountsPerPatient, how='left', left_on='student_name', right_on='student_name')
    patientPerStudentTable = createTable( patientPerStudentDf, colRatio=[2, 1, 1, 2], customTextCols=list(range(patientPerStudentDf.shape[1])),
        bottomPadding=6, topPadding=6, title="Patient Count per Student", titleStyle=subheadingStyle, headerColor=subheadingColor, tableTextStyle=tableTextStyleSmall)
    elements.append(PageBreak())
    elements.append(patientPerStudentTable)

    doc.build(elements, onFirstPage=getBannerDrawer(bannerTitle, ""))


dds4concerns = getAdditionalConcerns(engine, "DDS4")
boh3concerns = getAdditionalConcerns(engine, "BOH3")
display(dds4concerns)
display(boh3concerns)

dds4incidents = getClinicalIncidentSummary(engine, "DDS4")
boh3incidents = getClinicalIncidentSummary(engine, "BOH3")
display(dds4incidents)
display(boh3incidents)

# --- usage ---
buildCohortSummaryPdf( engine=engine, cohort="DDS4", outPath="BOH3_DDS4/Summary_DDS4.pdf", bannerTitle="Summary till date - DDS4",
    subheadingStyle=subheadingStyle, subheadingColor= uniColor, concernsDf=dds4concerns, incidentsDf=dds4incidents,
      patientPerStudentDf=patientperstudentDDS4,)

buildCohortSummaryPdf( engine=engine, cohort="BOH3", outPath="BOH3_DDS4/Summary_BOH3.pdf", bannerTitle="Summary till date - BOH3",
   subheadingStyle=subheadingStyle, subheadingColor= uniColor, concernsDf=boh3concerns, incidentsDf=boh3incidents, 
   patientPerStudentDf=patientperstudentBOH3,)


# combine both cohorts and concerns with clinical incidents into a single report
combinedConcerns = pd.concat([dds4concerns, boh3concerns], ignore_index=True)
# display(combinedConcerns)
combinedIncidents = pd.concat([dds4incidents, boh3incidents], ignore_index=True)
# change datetimeutc to date only
combinedConcerns["datetimeutc"] = pd.to_datetime(combinedConcerns["datetimeutc"]).dt.date
combinedIncidents["datetimeutc"] = pd.to_datetime(combinedIncidents["datetimeutc"]).dt.date
# save to excel with two sheets: Concerns and Clinical Incidents
path = "BOH3_DDS4/Concerns_CI_Report.xlsx"
wargs = getmodeArgs(path)

with pd.ExcelWriter(path, **wargs) as writer:
    combinedConcerns.to_excel(writer, sheet_name="Additional Concerns", index=False)
    combinedIncidents.to_excel(writer, sheet_name="Clinical Incidents", index=False)

# Table where student submitted but assessor did not submit; columns: Student Name, Date, Assessor Name
filepath = "BOH3_DDS4/Not_Submitted.xlsx"
studentSubmittedNotAssessorBOH3 = getNotSubmitted(engine, "BOH3")
studentSubmittedNotAssessorDDS4 = getNotSubmitted(engine, "DDS4")
wargs = getmodeArgs(filepath)
with pd.ExcelWriter(filepath, **wargs) as writer:
    studentSubmittedNotAssessorBOH3.to_excel(writer, sheet_name="BOH3", index=False)
    studentSubmittedNotAssessorDDS4.to_excel(writer, sheet_name="DDS4", index=False)



In [ ]:
def createPatientStatsPerClinicPerRotation(engine, cohort, formsTable="dds4_boh3_forms", attended = True, filters: dict = None):
  filepath = f"BOH3_DDS4/patient_stats_by_clinic_{cohort}.xlsx"
  studentclinicfilepath = f"BOH3_DDS4/patient_stats_by_clinic_{cohort}_detailed.xlsx"
  for rotation in ["Rotation 1", "Rotation 2", "Rotation 3"]:
      kwargs = getmodeArgs(filepath)
      kwargs2 = getmodeArgs(studentclinicfilepath)
      # print(f"Processing {rotation} for {cohort} with mode {kwargs}")
      filters = {"rotation": rotation}
      resultDf, studentClinicDf = createPatientStatsPerClinic(engine, "DDS4", filters=filters, formsTable=formsTable, attended=attended)
      # print(f"Patient counts per clinic for {rotation}:")
      # display(resultDf)
      with pd.ExcelWriter(filepath, **kwargs) as writer:
          resultDf.to_excel(writer, sheet_name=rotation, index=False)
      with pd.ExcelWriter(studentclinicfilepath, **kwargs2) as writer:
          studentClinicDf.to_excel(writer, sheet_name=rotation, index=False)

createPatientStatsPerClinicPerRotation(engine, "DDS4", formsTable="dds4_boh3_forms", attended=True)
createPatientStatsPerClinicPerRotation(engine, "BOH3", formsTable="dds4_boh3_forms", attended=True)

#### Student reports

In [ ]:
from pathlib import Path
def getStudentsInCohort(engine, cohort, formsTable="dds4_boh3_forms"):
    sql = f"""
    SELECT DISTINCT
      student_number,
      student_name
    FROM {formsTable}
    WHERE cohort = :cohort
      AND student_name IS NOT NULL
      AND student_name <> ''
      AND student_name <> 'Test Student'
    ORDER BY student_name;
    """
    return readDf(engine, sql, {"cohort": cohort})


def toInt(x):
    return int(round(x)) if x is not None else 0
# ---------- Extract + aggregate (student report) ----------

def getStudentPatientSummary(engine, cohort, studentName, formsTable="dds4_boh3_forms"):
    sql = f"""
    WITH pats AS (
      SELECT
        f.student_number,
        f.student_name,
        pd AS patient,
        NULLIF(pd->>'patientAge','')::int AS age
      FROM {formsTable} f
      CROSS JOIN LATERAL jsonb_array_elements(COALESCE(f.patient_data,'[]'::jsonb)) pd
      WHERE f.cohort = :cohort
        AND f.student_name = :studentName
        AND (pd->>'patientAttended')::boolean = true
    )
    SELECT
      COUNT(*)::int AS totalPatients,
      COUNT(*) FILTER (WHERE age BETWEEN 0 AND 6)::int  AS age0to6,
      COUNT(*) FILTER (WHERE age BETWEEN 7 AND 17)::int AS age7to17,
      COUNT(*) FILTER (WHERE age >= 18)::int            AS age18plus
    FROM pats;
    """
    return readDf(engine, sql, {"cohort": cohort, "studentName": studentName})

def getStudentTopItemCodes(engine, cohort, studentName, limit=10, formsTable="dds4_boh3_forms"):
    sql = f"""
    SELECT
      ic->>'code' AS itemCode,
      MAX(ic->>'description') AS description,
      SUM(COALESCE(NULLIF((ic->>'quantity')::int, NULL), 1))::int AS totalQty
    FROM {formsTable} f
    CROSS JOIN LATERAL jsonb_array_elements(COALESCE(f.patient_data,'[]'::jsonb)) pd
    CROSS JOIN LATERAL jsonb_array_elements(COALESCE(pd->'itemCodes','[]'::jsonb)) ic
    WHERE f.cohort = :cohort
      AND f.student_name = :studentName
      AND ic ? 'code'
    GROUP BY 1
    ORDER BY totalQty DESC, itemCode
    LIMIT :limit;
    """
    return readDf(engine, sql, {"cohort": cohort, "studentName": studentName, "limit": int(limit)})

def getStudentSelfSummary(engine, cohort, studentName, formsTable="dds4_boh3_forms"):
    # Self: practice readiness + reflection + CAF avg per form + overall avg
    sql = f"""
    WITH base AS (
      SELECT
        assessmentid, form_code, datetimeutc, subject, clinic, assessor_name,
        student_data->'texts'->>'reflection' AS student_reflection,
        NULLIF(student_data->'scales'->'scale-practice-readiness'->>'scale','') AS practice_readiness,
        student_data->'checklists'->'checklist-caf-final-eval' AS caf
      FROM {formsTable}
      WHERE cohort = :cohort
        AND student_name = :studentName
        AND submitted_by_student
    ),
    cafRows AS (
      SELECT
        b.assessmentid, b.form_code, b.datetimeutc, b.subject, b.clinic, b.assessor_name,
        b.student_reflection, b.practice_readiness,
        kv.key AS mc,
        kv.value AS ratingText,
        CASE kv.value
          WHEN 'Done well' THEN 1.0
          WHEN 'Done' THEN 0.8
          WHEN 'Mostly done' THEN 0.6
          WHEN 'Sometimes done' THEN 0.4
          WHEN 'Not done' THEN 0.0
          ELSE NULL
        END::numeric AS ratingScore
      FROM base b
      LEFT JOIN LATERAL jsonb_each_text(COALESCE(b.caf, '{{}}'::jsonb)) kv(key, value)
        ON TRUE
    )
    SELECT
      assessmentid,
      form_code,
      datetimeutc,
      subject,
      clinic,
      assessor_name,
      practice_readiness,
      student_reflection,
      ROUND(AVG(ratingScore), 3)::numeric(6,3) AS caf_avg_score
    FROM cafRows
    GROUP BY
      assessmentid, form_code, datetimeutc, subject, clinic, assessor_name,
      practice_readiness, student_reflection
    ORDER BY datetimeutc, assessmentid;
    """
    return readDf(engine, sql, {"cohort": cohort, "studentName": studentName})

def getStudentAssessorSummary(engine, cohort, studentName, formsTable="dds4_boh3_forms"):
    # Assessor: entrustment numeric + concerns + incidents + strengths/weaknesses text flattened
    sql = f"""
    WITH base AS (
      SELECT
        assessmentid, form_code, datetimeutc, subject, clinic, assessor_name,
        NULLIF(assessor_data->'scales'->'scale-entrustment'->>'scale','') AS entrustment_scale,
        NULLIF(additional_concerns,'') AS additional_concerns,
        assessor_data->'multi-select' AS multi
      FROM {formsTable}
      WHERE cohort = :cohort
        AND student_name = :studentName
        AND submitted_by_assessor
    ),
    extracted AS (
      SELECT
        b.*,
        (
          SELECT string_agg(x->>'value', E'\n' ORDER BY x->>'value')
          FROM jsonb_array_elements(COALESCE(b.multi->'strengths','[]'::jsonb)) x
        ) AS strengths_text,
        (
          SELECT string_agg(x->>'value', E'\n' ORDER BY x->>'value')
          FROM jsonb_array_elements(COALESCE(b.multi->'weakness-other','[]'::jsonb)) x
        ) AS weakness_other_text,
        (
          SELECT string_agg(x->>'value', E'\n' ORDER BY x->>'value')
          FROM jsonb_array_elements(COALESCE(b.multi->'clinical-incident','[]'::jsonb)) x
        ) AS clinical_incident_text,
        COALESCE(jsonb_array_length(COALESCE(b.multi->'clinical-incident','[]'::jsonb)),0) AS clinical_incident_count
      FROM base b
    )
    SELECT
      assessmentid, form_code, datetimeutc, subject, clinic, assessor_name,
      CASE entrustment_scale
        WHEN 'S1' THEN 1
        WHEN 'S2' THEN 2
        WHEN 'S3' THEN 3
        WHEN 'S4' THEN 4
        ELSE NULL
      END::smallint AS entrustment_numeric,
      additional_concerns,
      clinical_incident_count,
      strengths_text,
      weakness_other_text,
      clinical_incident_text
    FROM extracted
    ORDER BY datetimeutc, assessmentid;
    """
    return readDf(engine, sql, {"cohort": cohort, "studentName": studentName})

def getStudentSummaryTable(engine, cohort, studentNumber, studentName, formsTable="dds4_boh3_forms"):
    patientDf = getStudentPatientSummary(engine, cohort, studentName, formsTable)
    selfDf = getStudentSelfSummary(engine, cohort, studentName, formsTable)
    assessorDf = getStudentAssessorSummary(engine, cohort, studentName, formsTable)
    topItemsDf = getStudentTopItemCodes(engine, cohort, studentName, 8, formsTable)
    # display(assessorDf)
    totalForms = int(readDf(engine, f"""
        SELECT COUNT(*)::int AS n
        FROM {formsTable}
        WHERE cohort=:cohort AND student_name=:studentName;
    """, {"cohort": cohort, "studentName": studentName}).iloc[0]["n"])

    avgEntrustment = assessorDf["entrustment_numeric"].dropna().astype(float).mean()
    entrustmentDistribution = assessorDf["entrustment_numeric"].value_counts().sort_index()
    print("Entrustment distribution:\n", entrustmentDistribution)
    avgCaf = selfDf["caf_avg_score"].dropna().astype(float).mean()

    totalConcerns = assessorDf["additional_concerns"].dropna().shape[0]
    totalIncidents = int(assessorDf["clinical_incident_count"].fillna(0).sum())
    p = patientDf.iloc[0] if not patientDf.empty else pd.Series(dtype="object")

    metrics = [
        ("Total Forms", str(toInt(totalForms))),
        ("Total Patients", str(toInt(p.get("totalpatients", 0) or 0))),
        ("Patients Age 0–6", str(toInt(p.get("age0to6", 0) or 0))),
        ("Patients Age 7–17", str(toInt(p.get("age7to17", 0) or 0))),
        ("Patients Age 18+", str(toInt(p.get("age18plus", 0) or 0))),
        ("Avg Self CAF Score", None if pd.isna(avgCaf) else round(avgCaf, 2)),
        ("Avg Assessor Entrustment (1–4)", None if pd.isna(avgEntrustment) else round(avgEntrustment, 2)),
        ("Entrustment Distribution (1–4)", "<br/>".join(f"Lvl {idx}: {cnt}" for idx, cnt in entrustmentDistribution.items())),
        ("Forms With Additional Concerns", str(toInt(totalConcerns))),
        ("Total Clinical Incidents Selected", str(toInt(totalIncidents))),
    ]
    metricsDf = pd.DataFrame(metrics, columns=["Metric", "Value"])

    return metricsDf, topItemsDf, selfDf, assessorDf

def getStudentRollup(engine, cohort, studentName, formsTable="dds4_boh3_forms"):
    # Self: avg CAF + readiness distribution + reflection count
    sql = f"""
    WITH base AS (
      SELECT
        student_data->'texts'->>'reflection' AS reflection,
        NULLIF(student_data->'scales'->'scale-practice-readiness'->>'scale','') AS readiness,
        student_data->'checklists'->'checklist-caf-final-eval' AS caf
      FROM {formsTable}
      WHERE cohort=:cohort AND student_name=:studentName
    ),
    caf_vals AS (
      SELECT
        CASE kv.value
          WHEN 'Done well' THEN 1.0
          WHEN 'Done' THEN 0.8
          WHEN 'Mostly done' THEN 0.6
          WHEN 'Sometimes done' THEN 0.4
          WHEN 'Not done' THEN 0.0
          ELSE NULL
        END::numeric AS score
      FROM base b
      LEFT JOIN LATERAL jsonb_each_text(COALESCE(b.caf,'{{}}'::jsonb)) kv(key, value) ON TRUE
    )
    SELECT
      (SELECT COUNT(*)::int FROM base) AS forms_count,
      (SELECT COUNT(*)::int FROM base WHERE NULLIF(reflection,'') IS NOT NULL) AS reflections_count,
      (SELECT ROUND(AVG(score),3)::numeric(6,3) FROM caf_vals WHERE score IS NOT NULL) AS caf_avg,
      (SELECT PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY score) FROM caf_vals WHERE score IS NOT NULL) AS caf_median
    ;
    """
    roll = readDf(engine, sql, {"cohort": cohort, "studentName": studentName})

    # readiness distribution
    sql2 = f"""
      SELECT
        student_config->'scales'->'scale-practice-readiness'->'fields'
          -> (student_data->'scales'->'scale-practice-readiness'->>'scale')
          AS readiness,
        COUNT(*) AS n
      FROM {formsTable}
      WHERE cohort = :cohort
        AND student_name = :studentName
      GROUP BY readiness
      ORDER BY readiness;
      """
    readinessDf = readDf(engine, sql2, {"cohort": cohort, "studentName": studentName})

    return roll, readinessDf

def getAssessorRollup(engine, cohort, studentName, formsTable="dds4_boh3_forms"):
    # Assessor: avg entrustment + strengths/weakness counts + incident count + concerns count
    sql = f"""
    WITH base AS (
      SELECT
        NULLIF(assessor_data->'scales'->'scale-entrustment'->>'scale','') AS entrustment_scale,
        NULLIF(additional_concerns,'') AS additional_concerns,
        assessor_data->'multi-select' AS multi
      FROM {formsTable}
      WHERE cohort=:cohort AND student_name=:studentName
    ),
    ent AS (
      SELECT
        CASE entrustment_scale
          WHEN 'S1' THEN 1
          WHEN 'S2' THEN 2
          WHEN 'S3' THEN 3
          WHEN 'S4' THEN 4
          ELSE NULL
        END::numeric AS entrustment_num
      FROM base
    ),
    counts AS (
      SELECT
        COALESCE(SUM(jsonb_array_length(COALESCE(multi->'strengths','[]'::jsonb))),0) AS strengths_n,
        COALESCE(SUM(jsonb_array_length(COALESCE(multi->'weakness-other','[]'::jsonb))),0) AS weakness_other_n,
        COALESCE(SUM(jsonb_array_length(COALESCE(multi->'weakness-timeliness','[]'::jsonb))),0) AS weakness_timeliness_n,
        COALESCE(SUM(jsonb_array_length(COALESCE(multi->'weakness-communication','[]'::jsonb))),0) AS weakness_communication_n,
        COALESCE(SUM(jsonb_array_length(COALESCE(multi->'weakness-technical-skills','[]'::jsonb))),0) AS weakness_technical_n,
        COALESCE(SUM(jsonb_array_length(COALESCE(multi->'weakness-person-centered-care','[]'::jsonb))),0) AS weakness_pcc_n,
        COALESCE(SUM(jsonb_array_length(COALESCE(multi->'weakness-professional-behaviour','[]'::jsonb))),0) AS weakness_professional_n,
        COALESCE(SUM(jsonb_array_length(COALESCE(multi->'weakness-risk-management','[]'::jsonb))),0) AS weakness_risk_n,
        COALESCE(SUM(jsonb_array_length(COALESCE(multi->'weakness-knowledge-clinical-reasoning','[]'::jsonb))),0) AS weakness_reasoning_n,
        COALESCE(SUM(jsonb_array_length(COALESCE(multi->'clinical-incident','[]'::jsonb))),0) AS incidents_n
      FROM base
    )
    SELECT
      (SELECT COUNT(*)::int FROM base) AS forms_count,
      (SELECT COUNT(*)::int FROM base WHERE additional_concerns IS NOT NULL) AS concerns_forms_n,
      (SELECT ROUND(AVG(entrustment_num),3)::numeric(6,3) FROM ent WHERE entrustment_num IS NOT NULL) AS entrustment_avg,
      (SELECT PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY entrustment_num) FROM ent WHERE entrustment_num IS NOT NULL) AS entrustment_median,
      strengths_n,
      weakness_other_n,
      weakness_timeliness_n,
      weakness_communication_n,
      weakness_technical_n,
      weakness_pcc_n,
      weakness_professional_n,
      weakness_risk_n,
      weakness_reasoning_n,
      incidents_n
    FROM counts;
    """
    roll = readDf(engine, sql, {"cohort": cohort, "studentName": studentName})
    return roll

def getTopMultiSelectValues(engine, cohort, studentName, key, limit=6, formsTable="dds4_boh3_forms"):
    # key examples: 'strengths', 'weakness-other', 'weakness-timeliness', 'clinical-incident'
    sql = f"""
    WITH vals AS (
      SELECT x->>'value' AS v
      FROM {formsTable} f
      CROSS JOIN LATERAL jsonb_array_elements(COALESCE(f.assessor_data->'multi-select'->:key,'[]'::jsonb)) x
      WHERE f.cohort=:cohort AND f.student_name=:studentName
        AND x ? 'value'
    )
    SELECT v AS value, COUNT(*)::int AS n
    FROM vals
    GROUP BY v
    ORDER BY n DESC, v
    LIMIT :limit;
    """
    return readDf(engine, sql, {"cohort": cohort, "studentName": studentName, "key": key, "limit": int(limit)})

def getSelfReflections(engine, cohort, studentName, formsTable="dds4_boh3_forms"):
  # instead of assessor_name get the item codes for the day
    sql = f"""
    SELECT
      datetimeutc::date AS datetimeutc,
      student_data->'texts'->>'reflection' AS self_reflection,
      (
        SELECT string_agg(ic->>'code', ', ' ORDER BY ic->>'code')
        FROM jsonb_array_elements(COALESCE(f.patient_data, '[]'::jsonb)) pd
        CROSS JOIN jsonb_array_elements(COALESCE(pd->'itemCodes', '[]'::jsonb)) ic
        WHERE ic ? 'code'
      ) AS item_codes

    FROM {formsTable} f
    WHERE cohort=:cohort AND student_name=:studentName
      AND submitted_by_student
      AND NULLIF(student_data->'texts'->>'reflection','') IS NOT NULL
      AND student_data->'texts'->>'reflection' <> ''
    ORDER BY datetimeutc;
    """
    return readDf(engine, sql, {"cohort": cohort, "studentName": studentName})

def buildStudentVsAssessorSection( engine, cohort,  studentName,  elements, styles, formsTable="dds4_boh3_forms"):
    
    selfRoll, readinessDf = getStudentRollup(engine, cohort, studentName, formsTable)
    assessorRoll = getAssessorRollup(engine, cohort, studentName, formsTable)

    # Top multi-select items (examples)
    topStrengths = getTopMultiSelectValues(engine, cohort, studentName, "strengths", 6, formsTable)
    topWeaknessOther = getTopMultiSelectValues(engine, cohort, studentName, "weakness-other", 6, formsTable)
    topIncidents = getTopMultiSelectValues(engine, cohort, studentName, "clinical-incident", 6, formsTable)

    elements.append(PageBreak())
    # elements.append(Paragraph("Student self judgement", subheadingStyle))
    # elements.append(Spacer(1, 12))

    # ---- Self summary table ----
    s = selfRoll.iloc[0].to_dict() if not selfRoll.empty else {}
    selfMetrics = pd.DataFrame([
        ("forms submitted", s.get("forms_count")),
        ("reflections submitted", s.get("reflections_count")),
        ("self caf avg score", s.get("caf_avg")),
        ("self caf median score", s.get("caf_median")),
    ], columns=["metric", "value"])

    # elements.append(createTable(
    #     selfMetrics,
    #     colRatio=[2, 1],
    #     customTextCols=[0, 1],
    #     bottomPadding=6,
    #     topPadding=6,
    #     title="Student self judgement"
    # ))
    # elements.append(Spacer(1, 12))

    # readiness distribution (small)
    if not readinessDf.empty:
        rd = readinessDf.copy()
        rd.columns = ["Practice Readiness", "count"]
        # remove none/null readiness
        rd = rd[rd["Practice Readiness"].notna() & (rd["Practice Readiness"] != "") & (rd["Practice Readiness"].str.lower() != "none")]
        elements.append(createTable(
            rd,
            colRatio=[3, 1],
            customTextCols=[0, 1],
            bottomPadding=6,
            topPadding=6,
            title="Student Judgement: Practice Readiness Distribution",
            titleStyle=subheadingStyle,
            headerColor=uniColor
        ))
        elements.append(Spacer(1, 18))
      

    # ---- Assessor summary table ----
    # elements.append(Paragraph("Assessor judgement", subheadingStyle))
    # elements.append(Spacer(1, 12))

    a = assessorRoll.iloc[0].to_dict() if not assessorRoll.empty else {}


    assessorMetrics = pd.DataFrame([
        ("Forms assessed",                     toInt(a.get("forms_count"))),
        ("Average entrustment (1–4)",          toInt(a.get("entrustment_avg"))),
        ("Median entrustment (1–4)",           toInt(a.get("entrustment_median"))),
        ("# Additional concerns",     toInt(a.get("concerns_forms_n"))),
        ("# Clinical incidents",        toInt(a.get("incidents_n"))),
        ("# Commendations given",                toInt(a.get("strengths_n"))),
    ], columns=["Metric", "Value"])

    elements.append(createTable(
        assessorMetrics,
        colRatio=[3, 1],
        customTextCols=[0, 1],
        bottomPadding=6,
        topPadding=6,
        title="Assessor judgement",
        titleStyle=subheadingStyle,
        headerColor=uniColor
    ))
    elements.append(Spacer(1, 18))

    # ---- Key feedback themes (top values) ----
    def renderTopDf(df, title, valuename):
        if df.empty:
            return
        tmp = df.copy()
        # replace values in first column \n to <br/>
        tmp.iloc[:,0] = tmp.iloc[:,0].str.replace('\n', '<br/>', regex=False)
        tmp.columns = [valuename, "Count"]
        elements.append(createTable(
            tmp,
            colRatio=[4, 1],
            customTextCols=[0, 1],
            bottomPadding=6,
            topPadding=6,
            title=title,
            titleStyle=subheadingStyle,
            headerColor=uniColor
        ))
        elements.append(Spacer(1, 12))

    # create different table for weaknesses/others?
    weaknessCounts = pd.DataFrame([
        ("Time management",           toInt(a.get("weakness_timeliness_n"))),
        ("Communication",             toInt(a.get("weakness_communication_n"))),
        ("Technical skills",          toInt(a.get("weakness_technical_n"))),
        ("Person-centred care",       toInt(a.get("weakness_pcc_n"))),
        ("Professional behaviour",    toInt(a.get("weakness_professional_n"))),
        ("Risk management",           toInt(a.get("weakness_risk_n"))),
        ("Knowledge & reasoning",     toInt(a.get("weakness_reasoning_n"))),
        ("Other weaknesses noted",              toInt(a.get("weakness_other_n"))),
    ], columns=["Weakness Type", "Count"])
    
    # remove with zero counts
    weaknessCounts = weaknessCounts[weaknessCounts["Count"] > 0]

    elements.append(createTable(
        weaknessCounts,
        colRatio=[4, 1],
        customTextCols=[0, 1],
        bottomPadding=6,
        topPadding=6,
        title="Weaknesses",
        titleStyle=subheadingStyle,
        headerColor=uniColor
    ))
    elements.append(Spacer(1, 18))

    renderTopDf(topStrengths, "Top commendations", "Commendation")
    renderTopDf(topWeaknessOther, "Other weaknesses and strengths", "Weakness/Strength")
    renderTopDf(topIncidents, "Top clinical incidents selected", "Incident")

        # reflection table
    reflectionsDf = getSelfReflections(engine, cohort, studentName, formsTable)
    if not reflectionsDf.empty:
        display(reflectionsDf)
        r = reflectionsDf.copy()
        r.columns = ["Date", "Self Reflection", "Item Codes"]
        r['Self Reflection'] = r['Self Reflection'].str.replace('\n', '<br/>')
        elements.append(createTable(
            r,
            colRatio=[1, 6, 1.2],
            customTextCols=[0, 1, 2],
            bottomPadding=6,
            topPadding=6,
            title="Student Judgement: Self Reflections",
            titleStyle=subheadingStyle,
            headerColor=uniColor,
            tableTextStyle=tableTextStyleSmall
        ))
        elements.append(Spacer(1, 18))


# ---------- PDF section builder ----------

# ---------- PDF builder (one student) ----------

def buildStudentPdf(  engine,  cohort,  studentNumber,  studentName,  outputPath, formsTable="dds4_boh4_forms", 
                    pageSize=None, leftMargin=36, rightMargin=36, topMargin=48, bottomMargin=36, styles=None):
    
    metricsDf, topItemsDf, selfDf, assessorDf = getStudentSummaryTable(
        engine, cohort, studentNumber, studentName, formsTable
    )
    display(assessorDf.head())
    display(selfDf.head())
    doc = SimpleDocTemplate( str(outputPath), pagesize=pageSize, rightMargin=rightMargin, leftMargin=leftMargin,  topMargin=topMargin, bottomMargin=bottomMargin)

    elements = []

    # --- Front page summary ---

    elements.append(Spacer(1, 72))
    elements.append(Paragraph("This is a summary report of your clinical activity so far in 2026.      \
      For detailed information please review your completed forms in the DASH program.<br/> We are working on an interactive live dashboard for future reports.", subsubheadingStyleL))
    elements.append(createTable(metricsDf, colRatio=[2, 1], customTextCols=[0, 1], bottomPadding=6, topPadding=6, title="Summary",
                                titleStyle=subheadingStyle, headerColor=uniColor)
                              )

    if not topItemsDf.empty:
        elements.append(Spacer(1, 18))
        topItemsDf2 = topItemsDf.copy()
        topItemsDf2.columns = ["Item Code", "Description", "Total Qty"]
        elements.append(createTable(topItemsDf2, colRatio=[1, 4, 1], customTextCols=[0, 1, 2], bottomPadding=6, topPadding=6,
                                     title="Top Procedures", titleStyle=subheadingStyle, headerColor=uniColor))

    # --- Self vs Assessor ---

    buildStudentVsAssessorSection( engine, cohort, studentName, elements, styles,formsTable)

    bannerTitle = f"Student Summary - {cohort}"
    bannerSubtitle = f"{studentName} ({studentNumber})" if studentNumber else studentName
    doc.build(elements, onFirstPage=getBannerDrawer(bannerTitle, bannerSubtitle))


# ---------- Batch: build all student PDFs for a cohort ----------

def buildCohortStudentReports( engine, cohort, outputDir, formsTable="dds4_boh4_forms",
                              pageSize=None, leftMargin=36, rightMargin=36, topMargin=48,bottomMargin=36,styles=None):
    outputDir = Path(outputDir)
    outputDir.mkdir(parents=True, exist_ok=True)

    studentsDf = getStudentsInCohort(engine, cohort, formsTable)
    for _, row in studentsDf.iterrows():
        studentNumber = row["student_number"]
        studentName = row["student_name"]

        safeName = "".join(c for c in str(studentNumber) if c.isalnum() or c in (" ", "_", "-")).strip()
        outPath = outputDir / f"{safeName}.pdf"

        buildStudentPdf(engine=engine,cohort=cohort,studentNumber=studentNumber,studentName=studentName,
            outputPath=outPath,formsTable=formsTable,pageSize=pageSize,leftMargin=leftMargin,
            rightMargin=rightMargin,topMargin=topMargin,bottomMargin=bottomMargin, styles=styles)
        # break  # for testing, remove this to build for all students        

buildCohortStudentReports(engine, cohort="DDS4", outputDir="BOH3_DDS4/DDS4_StudentReports", formsTable="dds4_boh3_forms", pageSize=pageSize, 
                          leftMargin=leftMargin, rightMargin=rightMargin, topMargin=topMargin, bottomMargin=bottomMargin, styles=styles)

buildCohortStudentReports(engine,cohort="BOH3",outputDir="BOH3_DDS4/BOH3_StudentReports",formsTable="dds4_boh3_forms",pageSize=pageSize,
                          leftMargin=leftMargin,rightMargin=rightMargin,topMargin=topMargin,bottomMargin=bottomMargin, styles=styles)

#### In excel sheet

In [ ]:
import re
import pandas as pd
from openpyxl.utils import get_column_letter
from openpyxl.styles import Font, Alignment
from openpyxl import Workbook

mainSql = """WITH base AS (
  SELECT
    student_name,
    assessor_name,
    datetimeutc::date AS date,
    student_data,
    assessor_data
  FROM dds4_boh3_forms
  WHERE cohort = :cohort
    AND student_name = :studentName
)
SELECT
  b.student_name,
  b.date,
  b.assessor_name,
  NULLIF(b.student_data->'texts'->>'reflection','') AS student_reflection,

  -- other notes = weakness-other + strengths where name='Other'
  os.other_notes

FROM base b
LEFT JOIN LATERAL (
  SELECT NULLIF(string_agg(DISTINCT NULLIF(trim(v), ''), ', '), '') AS other_notes
  FROM (
    SELECT e->>'value' AS v
    FROM jsonb_array_elements(COALESCE(b.assessor_data->'multi-select'->'weakness-other','[]'::jsonb)) e

    UNION ALL

    SELECT e->>'value' AS v
    FROM jsonb_array_elements(COALESCE(b.assessor_data->'multi-select'->'strengths','[]'::jsonb)) e
    WHERE COALESCE(e->>'name','') = 'Other'
  ) t
) os ON TRUE
ORDER BY b.date, b.assessor_name;
"""
weaknessSql = """WITH base AS (
  SELECT assessor_data
  FROM dds4_boh3_forms
  WHERE cohort = :cohort
    AND student_name = :studentName
)
SELECT
  trim(e->>'value') AS weakness,
  COUNT(*)::int AS n
FROM base b
JOIN LATERAL jsonb_each(COALESCE(b.assessor_data->'multi-select','{}'::jsonb)) kv(key, arr) ON TRUE
JOIN LATERAL jsonb_array_elements(COALESCE(kv.arr,'[]'::jsonb)) e ON TRUE
WHERE kv.key LIKE 'weakness-%'
  AND kv.key <> 'weakness-other'
  AND NULLIF(trim(e->>'value'), '') IS NOT NULL
GROUP BY weakness
ORDER BY n DESC, weakness;
"""
strengthSql = """WITH base AS (
  SELECT assessor_data
  FROM dds4_boh3_forms
  WHERE cohort = :cohort
    AND student_name = :studentName
)
SELECT
  trim(e->>'value') AS strength,
  COUNT(*)::int AS n
FROM base b
JOIN LATERAL jsonb_array_elements(COALESCE(b.assessor_data->'multi-select'->'strengths','[]'::jsonb)) e ON TRUE
WHERE COALESCE(e->>'name','') <> 'Other'
  AND NULLIF(trim(e->>'value'), '') IS NOT NULL
GROUP BY strength
ORDER BY n DESC, strength;
"""
concernsSql = """WITH base AS (
  SELECT
    student_name,
    assessor_name,
    datetimeutc::date AS date,
    additional_concerns,
    assessor_data
  FROM dds4_boh3_forms
  WHERE cohort = :cohort
    AND student_name = :studentName
)
SELECT
  b.student_name,
  b.date,
  b.assessor_name,
  NULLIF(b.additional_concerns,'') AS additional_concerns,
  ci.critical_incidents
FROM base b
LEFT JOIN LATERAL (
  SELECT NULLIF(string_agg(DISTINCT NULLIF(trim(e->>'value'), ''), ', '), '') AS critical_incidents
  FROM jsonb_array_elements(COALESCE(b.assessor_data->'multi-select'->'clinical-incident','[]'::jsonb)) e
) ci ON TRUE
WHERE NULLIF(b.additional_concerns,'') IS NOT NULL
   OR ci.critical_incidents IS NOT NULL
ORDER BY b.date, b.assessor_name;
"""

def sanitizeSheetName(name):
    safe = re.sub(r'[\[\]\:\*\?\/\\]', '', str(name or '')).strip()
    return (safe[:31] or "Sheet")

def autoFitColumns(ws, minWidth=10, maxWidth=60):
    for colCells in ws.columns:
        maxLen = 0
        colLetter = get_column_letter(colCells[0].column)
        for cell in colCells:
            if cell.value is None:
                continue
            maxLen = max(maxLen, len(str(cell.value)))
        ws.column_dimensions[colLetter].width = max(minWidth, min(maxWidth, maxLen + 2))

def writeTitle(ws, row, title):
    cell = ws.cell(row=row, column=1, value=title)
    cell.font = Font(bold=True, size=14)
    return row + 2

def writeTable(ws, df, startRow, startCol=1, title=None):
    r = startRow
    if title:
        ws.cell(row=r, column=startCol, value=title).font = Font(bold=True, size=12)
        r += 1

    # header
    for j, col in enumerate(df.columns, start=startCol):
        c = ws.cell(row=r, column=j, value=str(col))
        c.font = Font(bold=True)
        c.alignment = Alignment(wrap_text=True, vertical="top")
    r += 1

    # rows
    for _, row in df.iterrows():
        for j, col in enumerate(df.columns, start=startCol):
            c = ws.cell(row=r, column=j, value=row[col])
            c.alignment = Alignment(wrap_text=True, vertical="top")
        r += 1

    return r + 2  # leave a blank line

def buildStudentSheet(engine, wb, cohort, studentName, readDf):
    sheetName = sanitizeSheetName(studentName)
    ws = wb.create_sheet(title=sheetName)

    # mainSql = mainSql
    # weaknessSql = weaknessSql
    # strengthSql = strengthSql
    # concernsSql = concernsSql

    mainDf = readDf(engine, mainSql, {"cohort": cohort, "studentName": studentName})
    weaknessDf = readDf(engine, weaknessSql, {"cohort": cohort, "studentName": studentName})
    strengthDf = readDf(engine, strengthSql, {"cohort": cohort, "studentName": studentName})
    concernsDf = readDf(engine, concernsSql, {"cohort": cohort, "studentName": studentName})

    # normalize column names
    for df in [mainDf, weaknessDf, strengthDf, concernsDf]:
        if not df.empty:
            df.columns = [c.lower() for c in df.columns]

    # main table: date | assessor_name | student_reflection | other_notes
    if not mainDf.empty:
        keepCols = ["date", "assessor_name", "student_reflection", "other_notes"]
        mainOut = mainDf[keepCols].copy()
    else:
        mainOut = pd.DataFrame(columns=["date", "assessor_name", "student_reflection", "other_notes"])

    row = 1
    row = writeTitle(ws, row, f"Student Summary: {studentName} ({cohort})")
    row = writeTable(ws, mainOut, row, title="Entries (Reflection + Other Notes)")

    # weakness + strengths counts
    if weaknessDf.empty:
        weaknessDf = pd.DataFrame(columns=["weakness", "n"])
    row = writeTable(ws, weaknessDf, row, title="Weaknesses (Counts)")

    if strengthDf.empty:
        strengthDf = pd.DataFrame(columns=["strength", "n"])
    row = writeTable(ws, strengthDf, row, title="Strengths (Counts)")

    # concerns + incidents only if exists
    if not concernsDf.empty:
        concernsOut = concernsDf[["date", "assessor_name", "additional_concerns", "critical_incidents"]].copy()
        row = writeTable(ws, concernsOut, row, title="Additional Concerns & Critical Incidents")

    autoFitColumns(ws)

def exportStudentTextWorkbook(engine, cohort, outPath, readDf):
    studentsSql = """
    SELECT DISTINCT student_name
    FROM dds4_boh3_forms
    WHERE cohort = :cohort
      AND student_name IS NOT NULL
      AND student_name <> 'Test Student'
    ORDER BY student_name;
    """
    studentsDf = readDf(engine, studentsSql, {"cohort": cohort})
    studentNames = studentsDf["student_name"].dropna().tolist()

    wb = Workbook()
    # remove default empty sheet
    wb.remove(wb.active)

    for studentName in studentNames:
        buildStudentSheet(engine, wb, cohort, studentName, readDf)

    wb.save(outPath)

# usage:
exportStudentTextWorkbook(engine, cohort="DDS4", outPath="BOH3_DDS4/DDS4_Textual_Report.xlsx", readDf=readDf)
exportStudentTextWorkbook(engine, cohort="BOH3", outPath="BOH3_DDS4/BOH3_Textual_Report.xlsx", readDf=readDf)


#### Individual report

In [ ]:
sql = """
WITH b AS (
  SELECT *
  FROM dds4_boh3_forms
  WHERE assessmentid = :assessmentId
),
student AS (
  SELECT
    b.assessmentid,
    b.form_code,
    b.cohort,
    b.subject,
    b.type,
    b.createdat::date AS created_date,
    b.updatedat::date AS updated_date,
    b.clinic,
    b.rotation,

    b.student_number,
    b.student_name,
    b.student_email,

    b.assessorid,
    b.assessor_name,

    NULLIF(b.student_data->'texts'->>'reflection','') AS student_reflection,

    b.student_data->'scales'->'scale-practice-readiness'->>'scale' AS practice_readiness_code,

    -- full statement for practice readiness
    b.student_config->'scales'->'scale-practice-readiness'->'fields'
      -> (b.student_data->'scales'->'scale-practice-readiness'->>'scale') AS practice_readiness_text,

    b.assessor_data->'scales'->'scale-entrustment'->>'scale' AS entrustment_code,

    -- CAF checklist raw labels (Done well / Done / Mostly done / Sometimes done / Not done)
    b.student_data->'checklists'->'checklist-caf-final-eval'->>'MC1' AS mc1,
    b.student_data->'checklists'->'checklist-caf-final-eval'->>'MC2' AS mc2,
    b.student_data->'checklists'->'checklist-caf-final-eval'->>'MC3' AS mc3,
    b.student_data->'checklists'->'checklist-caf-final-eval'->>'MC4' AS mc4,
    b.student_data->'checklists'->'checklist-caf-final-eval'->>'MC5' AS mc5,
    b.student_data->'checklists'->'checklist-caf-final-eval'->>'MC6' AS mc6,
    b.student_data->'checklists'->'checklist-caf-final-eval'->>'MC7' AS mc7,



    -- numeric entrustment
    CASE b.assessor_data->'scales'->'scale-entrustment'->>'scale'
      WHEN 'S1' THEN 1
      WHEN 'S2' THEN 2
      WHEN 'S3' THEN 3
      WHEN 'S4' THEN 4
      ELSE NULL
    END AS entrustment_num,

    -- full statement for entrustment (from assessor_config)
    b.assessor_config->'scales'->'scale-entrustment'->'fields'
      -> (b.assessor_data->'scales'->'scale-entrustment'->>'scale') AS entrustment_text,

    NULLIF(b.assessor_data->'texts'->>'additional_comments','') AS assessor_comments,

    NULLIF(b.additional_concerns,'') AS additional_concerns,

    b.patient_data,
    b.assessor_data

  FROM b
),
agg AS (
  SELECT
    s.*,

    -- strengths (exclude "Other")
    (
      SELECT NULLIF(string_agg(DISTINCT trim(e->>'value'), ', '), '')
      FROM jsonb_array_elements(COALESCE(s.assessor_data->'multi-select'->'strengths','[]'::jsonb)) e
      WHERE COALESCE(e->>'name','') <> 'Other'
        AND NULLIF(trim(e->>'value'), '') IS NOT NULL
    ) AS strengths,

    -- other strengths text
    (
      SELECT NULLIF(string_agg(DISTINCT trim(e->>'value'), ', '), '')
      FROM jsonb_array_elements(COALESCE(s.assessor_data->'multi-select'->'strengths','[]'::jsonb)) e
      WHERE COALESCE(e->>'name','') = 'Other'
        AND NULLIF(trim(e->>'value'), '') IS NOT NULL
    ) AS strengths_other,

    -- weaknesses across all weakness-* (excluding weakness-other)
    (
      SELECT NULLIF(string_agg(DISTINCT trim(e->>'value'), ', '), '')
      FROM jsonb_each(COALESCE(s.assessor_data->'multi-select','{}'::jsonb)) kv(key, arr)
      JOIN LATERAL jsonb_array_elements(COALESCE(kv.arr,'[]'::jsonb)) e ON TRUE
      WHERE kv.key LIKE 'weakness-%'
        AND kv.key <> 'weakness-other'
        AND NULLIF(trim(e->>'value'), '') IS NOT NULL
    ) AS weaknesses,

    -- weakness-other text
    (
      SELECT NULLIF(string_agg(DISTINCT trim(e->>'value'), ', '), '')
      FROM jsonb_array_elements(COALESCE(s.assessor_data->'multi-select'->'weakness-other','[]'::jsonb)) e
      WHERE NULLIF(trim(e->>'value'), '') IS NOT NULL
    ) AS weaknesses_other,

    -- clinical incidents selected
    (
      SELECT NULLIF(string_agg(DISTINCT trim(e->>'value'), ', '), '')
      FROM jsonb_array_elements(COALESCE(s.assessor_data->'multi-select'->'clinical-incident','[]'::jsonb)) e
      WHERE NULLIF(trim(e->>'value'), '') IS NOT NULL
    ) AS clinical_incidents

  FROM student s
)
SELECT
  assessmentid, form_code, cohort, subject, type, created_date, updated_date, clinic, rotation,
  student_number, student_name, student_email,
  assessorid, assessor_name, practice_readiness_text,
  entrustment_num, entrustment_text,
  student_reflection, assessor_comments,
  strengths, strengths_other, weaknesses, weaknesses_other,
  clinical_incidents, additional_concerns,
  patient_data
FROM agg;
"""

mcSql = """
SELECT
    b.assessmentid,
    b.student_name,
    kv.key AS "MC Code",

    -- Full description from config
    b.student_config
        -> 'checklists'
        -> 'checklist-caf-final-eval'
        -> 'fields'
        ->> kv.key AS "Full MC Text",

    -- Student selected value (Done well / Done etc.)
    kv.value AS "MC Rating",

    -- Numeric score mapping
    CASE kv.value
        WHEN 'Done well' THEN 1.0
        WHEN 'Done' THEN 0.8
        WHEN 'Mostly done' THEN 0.6
        WHEN 'Sometimes done' THEN 0.4
        WHEN 'Not done' THEN 0.0
        ELSE NULL
    END AS "MC Score"

FROM dds4_boh3_forms b,

LATERAL jsonb_each_text(
    b.student_data
        -> 'checklists'
        -> 'checklist-caf-final-eval'
) kv

WHERE b.assessmentid = :assessmentId;
"""


row = readDf(engine, sql, {"assessmentId": 14632}).iloc[0]
display(row)

mcDf = readDf(engine, mcSql, {"assessmentId": 14632})
display(mcDf)
doc = SimpleDocTemplate("BOH3_DDS4/Student_Detailed_Entry_Test.pdf", pagesize=pageSize, rightMargin=rightMargin, 
                        leftMargin=leftMargin,  topMargin=topMargin, bottomMargin=bottomMargin)

def createPage(elements, row, mcDf):
    # first the dates and clinic/rotation info
    infoData = [
        ("Creation Date", row["created_date"]),
        ("Updated Date", row["updated_date"]),
        ("Assessor", row["assessor_name"]),
        ("Clinic", row["clinic"]),
        ("Rotation", row["rotation"])
    ]
    infoTable = createTable(pd.DataFrame(infoData, columns=["Field", "Value"]), colRatio=[1, 2], customTextCols=[0, 1], bottomPadding=4, topPadding=4,
                            headerColor=uniColor, tableTextStyle=tableTextStyleSmall, title="Entry Information", titleStyle=subheadingStyle)
    elements.append(infoTable)

    # then the MC checklist items with descriptions and ratings
    if not mcDf.empty:
        mcTable = createTable(mcDf[["MC Code", "Full MC Text", "MC Rating"]], colRatio=[1, 4, 1], customTextCols=[0, 1, 2], bottomPadding=4, topPadding=4,
                            headerColor=uniColor, tableTextStyle=tableTextStyleSmall, title="CAF Checklist", titleStyle=subheadingStyle)
        elements.append(Spacer(1, 12))
        elements.append(mcTable)

    # finally the reflections and comments
    reflectionsData = [
        ("Student Reflection", row["student_reflection"]),
        ("Assessor Comments", row["assessor_comments"]),
        ("Strengths Noted", row["strengths"]),
        ("Other Strengths", row["strengths_other"]),
        ("Weaknesses Noted", row["weaknesses"]),
        ("Other Weaknesses", row["weaknesses_other"]),
        ("Clinical Incidents", row["clinical_incidents"]),
        ("Additional Concerns", row["additional_concerns"])
    ]
    reflectionsTable = createTable(pd.DataFrame(reflectionsData, columns=["Field", "Content"]), colRatio=[1, 3], customTextCols=[0, 1], bottomPadding=4, topPadding=4,
                                headerColor=uniColor, tableTextStyle=tableTextStyleSmall, title="Reflections & Comments", titleStyle=subheadingStyle)
    elements.append(Spacer(1, 12))
    elements.append(reflectionsTable)

elements = []
elements.append(Spacer(1, 72))
createPage(elements, row, mcDf)
doc.build(elements, onFirstPage=getBannerDrawer("Student Detailed Entry", f"{row['student_name']} ({row['student_number']})"))
    